# خط أنابيب تجهيز بيانات العملات الرقمية — دفتر موحّد (تاريخي + حيّ)

دفتر Jupyter واحد يجمع كل وحدات حزمتَي `crypto_model.config` و`crypto_model.data`
(الإعدادات، المحاذاة، التطبيع، الميزات، التقسيم، المصادر، خط الأنابيب...) في
مكان واحد **بلا أي اعتماد على ملفات `.py` خارجية**. كل الدوال معرَّفة هنا
مباشرة بنفس المنطق الأصلي.

**تغييران جوهريان عن الأصل:**

1. **تحميل البيانات التاريخية حصراً عبر Google Drive المُركَّب** (`drive.mount`)
   — لا رابط عام (`drive.google.com/uc?id=`) ولا مكتبة `gdown`.
2. **دعم تحضير بيانات حيّة (live)** من Binance Futures مباشرة — عبر **نفس**
   `prepare_single_asset`/`build_dataset_from_loader` المستخدمتين للتاريخي
   بلا أي تعديل عليهما ولا أي نسخة موازية من منطق التجهيز. الفارق الوحيد
   فعلياً هو مصدر الجلب (`fetch_data` من Binance بدل `load_asset` من Drive).
   لجعل آخر شمعة حقيقية تُحسَب بدل أن تُسقَط (لغياب "مستقبل" حقيقي لها بعد)،
   تُلحَق بذيل البيانات المجلوبة دفعة صورية قبل التمرير لخط الأنابيب، فتحصل
   على هدف صوري بدل أن تُستبعَد — بالضبط كما طلبت. النتيجة: كل عيّنات كل
   عملة عدا الأخيرة أهدافها **حقيقية 100%** (اختبار فوري على أحدث بيانات
   لم يرها التدريب)، والعيّنة الأخيرة **حقيقية الميزات بالكامل** بهدف صوري
   يُتجاهَل — تُستخلَص عبر `extract_last_batch()`.

## كيف تستخدم هذا الدفتر
1. شغّل الخلايا بالترتيب من الأعلى للأسفل.
2. عدّل خلية **⚙️ الإعدادات الافتراضية (`DEFAULT_CONFIG`)** بما يطابق
   مشروعك، أو عدّل `CONFIG` مباشرة بعد بنائه عبر `update_config(...)`.
3. للبيانات **التاريخية**: عدّل `drive_raw_dir` / `asset_registry_path`
   ثم اتبع قسم **"مثال استخدام شامل (تاريخي)"**.
4. للبيانات **الحيّة**: عيّن `API_KEY`/`API_SECRET` (أو اتركهما فارغتين
   لبيانات السوق العامة فقط) ثم اتبع قسم **"مثال استخدام شامل (حيّ)"**.

> ⚠️ القيم الافتراضية أمثلة توضيحية — راجعها قبل الاستخدام الفعلي.

> ℹ️ ملفات `crypto_model/config/*.py` (defaults, heads, runtime) وصلت الآن
> كاملة، فقسم الإعدادات هنا **مطابق للأصل حرفياً** (خلافاً للنسخة السابقة من
> هذا الدفتر التي بنت `CONFIG` تخمينياً لغياب هذه الملفات وقتها).

---

## سجل التعديلات في هذه النسخة (مبنية على تدقيق البيانات)

| # | المشكلة (الدليل) | الإصلاح | لماذا هكذا، وما البديل المرفوض |
|---|---|---|---|
| 1 | `TIME_hour_sin` / `TIME_hour_cos` ثابتتان على `base_tf='1D'` — `audit_normalization` يصنّفهما «بلا معلومة»، ودفتر التدقيق يُنذر في §٦ ويُلوِّن **كل** العملات في §٩ | لا تُولَّدان إلا إن كان أصغر فريم في `tf_order` داخل اليوم (`_hour_features_meaningful`). عدد الميزات 39 → 37 | الساعة لا تتغيّر على شمعة يومية أصلاً. **مرفوض:** إبقاؤهما وترك `process_windows` يُصفّرهما — تبقيان عمودين ميّتين في الإدخال ويُخفيان إنذارات التدقيق الحقيقية |
| 2 | `split_data` يُرجع **val و test فارغتين** على بيانات فعلية: على أطوال العملات الـ٤٢٢ في مخرجاتك (18,924 عيّنة) يقع `train_end` قبل النهاية بـ٦٤ يوماً بينما فجوة العزل ٣٣ يوماً تُفقَد مرتين | حدّا التقسيم يُشتقّان بحيث تقترب نِسَب **العيّنات المُبقاة** من 90/5/5 (`_resolve_by_kept_share`)، وخطأ صريح إن قلّ أي قسم عن `min_split_samples` | **مرفوض:** تصغير فجوة العزل — النوافذ متراكبة (٣٢ يوماً) فيعود التسرّب. **مرفوض:** الاكتفاء بـ`split_dates` اليدوي — فشل صامت يعود عند أي تغيير في البيانات |
| 3 | طلب: استبعاد عملات بعينها | `exclude_coins(configs, [...])` و`CONFIG['excluded_coins']` تُطبَّق تلقائياً في `build_dataset*`؛ الأسماء غير الموجودة يُحذَّر منها؛ والمتخطّاة في `dataset['skipped_assets']` | مطابقة الاسم الكامل بلا حساسية لحالة الأحرف. **مرفوض:** المطابقة بالجذر (`'BTC'`) — تلتقط `BTCDOMUSDT` وأخواتها بصمت |
| 4 | الجلب الحيّ بحدّ 1500 شمعة يُنتج «32 شمعة فقط»: `limit=5000` كان يُقتطع صامتاً إلى 1500 ساعة ≈ 62 يوماً، وبعد إحماء المؤشرات (~50) لا يبقى ما يكفي لنافذة 32 | `fetch_data` تقسّم الطلب على عدة صفحات (`endTime`) وتدمجها مرتّبة بلا تكرار؛ فشل أي صفحة ⇒ `None` | **مرفوض:** إرجاع صفحات ناجحة فقط — سجلّ مبتور من الأقدم يمرّ بصمت |
| 5 | طلب: حدّ أقصى للطلبات في الجلب الحيّ | `RateLimiter` بنافذة منزلقة آمنة للخيوط، `CONFIG['live_max_requests_per_minute']=2000`، يشمل كل صفحة وإعادة محاولة | **مرفوض:** عدّاد يُصفَّر كل دقيقة — يسمح بضعف الحدّ عند حافة الدقيقتين |
| 6 | طلب: استبعاد ميزات | `exclude_features(['RSI_14','BB*'])` (أسماء/أنماط) تُحدّث `exclude_from_features` و`feature_order` | الأسماء المجهولة تُحذَّر ولا تُضاف بصمت؛ استبعاد كل الميزات يُرفض |
| 7 | نُقلت تعديلاتك من الدفتر المرسل | بيانات `history_1d` وعمود `datetime_utc`، تقسيم 80/10/10، إيقاف OBV، 25 خيطاً، خلايا الحفظ/التقسيم/`%run` | خلية `DEFAULT_CONFIG` في دفترك لا تُترجَم (`},` متبقية بعد التعليق) — أُوقف OBV هنا بـ`static_indicators=[]` |
| 8 | الهدف كان (سعر_مستقبلي − آخر_إغلاق)/IQR_النافذة — نفس نسبة الحركة تُنتج قيمة هدف مختلفة حسب تقلّب النافذة، وanchor مختلف عن رأس التصنيف على high/low | `reg_target_mode='return'` (افتراضي جديد): عائد مباشر `(مستقبلي/آخر_سعر_من_نفس_النوع) - 1`، بلا IQR، ونفس مرجع رأس التصنيف. `invert_reg_predictions` يعكسه لسعر حقيقي | **مرفوض:** إبقاء IQR النافذة مرجعاً — هو أصل مشكلة الانكماش نحو المتوسط الموصوفة سابقاً؛ `'window_scale'` باقٍ خياراً للمقارنة فقط |
| 9 | طلب: تطبيع مقطعي عبر الأصول (Qlib CSZScoreNorm/CSRankNorm) | `cross_sectional_normalize(dataset)` اختيارية بعد `build_dataset` — تقيس كل عيّنة نسبة لعوائد أقرانها في نفس اللحظة، مع `invert_cross_sectional` للعكس (zscore فقط، تاريخياً) | تُحوِّل دلالة الرأس إلى أداء نسبي لا مطلق — موثَّق صراحةً في الدالة؛ ترفض `'window_scale'` بخطأ صريح بدل خلط مرجعين |
| 10 | طلب: حدّ فاصل مشترك يُختار بنسبة "آخر N من كل العيّنات" مباشرة، لا بمطابقة العيّنات المُبقاة بعد العزل | `split_cutoff_method='raw_quantile'` (اختياري) في `resolve_split_dates`/`split_data`، أو `build_leak_free_split()` كغلاف جاهز — عبر `compute_global_cutoff` | الافتراضي يبقى `'kept_share'` القائم: `raw_quantile` مبنيّ ليفشل بخطأ صريح على بيانات مكدَّسة (كبياناتك الحقيقية) بدل قسم شبه فارغ صامت — **مرفوض** جعله الافتراضي لهذا السبب تحديداً |
| 11 | فريم التحميل الحيّ كان يُثبَّت "1h" داخل fetch_data بصرف النظر عن الطلب، ومُقارَناً ضمنياً بفريم النموذج | `download_interval` إعداد مستقلّ عن `tf_order`؛ `fetch_data` تُحاسِب الفريم المطلوب فعلياً، و`_live_pad_length`/`_auto_live_limit` تحوّلان الوحدات بينهما | **مرفوض:** إبقاء الفريمين متطابقين افتراضاً — يمنع تدريب نموذج يومي من بيانات ساعية مُجمَّعة (resample)، وهو الاستخدام الشائع |
| 12 | طلب: سياق سوقي عابر للأصول عبر BTC | `market_context` (بروح FreqAI `include_corr_pairlist`): عائد BTC يُضاف كميزة `MKT_ret_N` لكل عملة أخرى، محاذى زمنياً؛ صفر للعملة المرجعية نفسها | **مرفوض:** قيمة غير صفرية للمرجع نفسه — دائرية (الهدف يتنبأ بنفسه جزئياً عبر ميزة مطابقة تقريباً) |
| 13 | طلب: إعادة تدريب دورية | `rolling_split_schedule`/`rolling_splits`: عدّة نوافذ (train,val,test) متحرّكة، كل نافذة مُطهَّرة (عزل) بنفس منطق `_split_global_time` | **مرفوض:** تجاهل الفجوة بين نوافذ متتالية توفيراً للبيانات — يُعيد تسرّب النوافذ المتراكبة |
| 14 | طلب: تحميل بيانات مشتقات (تمويل/فائدة مفتوحة) وتخزينها لاحقاً | `fetch_funding_rate` (تاريخ كامل، كالشموع) و`fetch_open_interest_hist` (محدودة بـ٣٠ يوماً من Binance نفسها)، و`save/load_funding_open_interest` لأرشفة تراكمية على Drive | لم تُدمَج بعد في `add_features` — تحتاج محاذاة زمنية مستقلّة، تُترَك لطلب لاحق صريح كما طُلب |
| 15 | استهلاك رام أعلى بكثير من حجم البيانات النهائي (قِيس: ~12 جيجا لناتج <1.5 جيجا)، وخطر فقدان معالجة كاملة عند إعادة بناء الجلسة | `align_multi_timeframes_time_based` (فريم واحد): `sliding_window_view` بدل نسخ كل نافذة في حلقة (ذروة 3.16×→1.13× الحجم النهائي، مقاس فعلياً). `process_windows`: float32 طوال الطريق بلا مصفوفة `out` موازية (7.77×→2.64×). `default_workers`: سقف رام احترازي إضافي. `checkpoint_dir` في `build_dataset_from_loader`: حفظ كل أصل فور معالجته، واستئناف بلا إعادة تنزيل/حساب | **مرفوض:** تقليل عدد الخيوط فقط بلا إصلاح الذروة نفسها — يُبطئ البناء دون علاج السبب؛ الإصلاحان معاً |
| 16 | طلب: دمج معدّل التمويل والفائدة المفتوحة (مبنيان مسبقاً — بند #14 — غير مُدمَجين) كميزات فعلية، بدء المرحلة ١ من خطة اكتشاف الإشارة | `add_funding_oi_features` تُلحق `FUND_rate`/`FUND_rate_z` (أرشيف كامل التاريخ) و`OI_chg_N` لكل أصل من أرشيفه الخاص عبر `reindex(..., method='ffill')` — بروح `add_market_context` لكن بلا مرجع مشترك. `CONFIG['funding_rate']['as_feature']`/`['open_interest']['as_feature']` (جديدان، افتراضهما True) يفصلان الجلب/الأرشفة عن الإدماج في المدخلات؛ ميزات إضافية 3-5 حسب التفعيل | **مرفوض:** ترك فترات بلا أرشيف (شائعة تاريخياً، خصوصاً OI محدودة ٣٠ يوماً) كـNaN ثم `dropna()` في `add_features` — يُسقط كل التاريخ الأقدم من أول أرشفة بصمت. **مرفوض:** ملء صفر بلا تمييز — يخلط "لا تغيّر" بـ"لا بيانات" على النموذج؛ لذا `FUND_available`/`OI_available` صريحان |
| 17 | `save_data_to_drive`/`load_preprocessed_data_from_drive` غير متوافقتين فعلياً (الأولى تحفظ على Drive المُركَّب بمسار مُوسَّم بالوقت بلا نسخة ثابتة؛ الثانية تُحمِّل عبر `gdown`+رابط عام مختلف تماماً — لا تقرأ ما تكتبه الأولى)، ضمن توحيد دفاتر المشروع (`main`/`model_v2`) لتستهلك مخرَج هذا الدفتر مباشرة | `save_data_to_drive` تكتب الآن نسخة مؤرَّخة **و`_latest.pkl.gz`** بمسار ثابت عبر `mount_drive()` نفسها المستخدَمة لأرشيف funding/OI؛ `load_data_from_drive` الجديدة تقرأ من نفس المسار مباشرة (بلا `gdown`/`file_id`). `load_preprocessed_data_from_drive` أُبقيت للتوافق الخلفي فقط، وعليها تنبيه صريح بعدم استخدامها لقراءة مخرَج `save_data_to_drive` | **مرفوض:** إبقاء الاعتماد على رابط عام يدوي لكل تشغيل — يكسر أي أتمتة بين الدفاتر ويحتاج نسخ `file_id` يدوياً في كل مرة |
| 18 | خلية «مثال استخدام شامل (حيّ)» (قسم ٢١) كانت الوحيدة من بين كل خلايا الأمثلة تحتوي استدعاءً فعلياً غير مُعلَّق (`build_dataset_live(symbols, ...)`) — أي أن تشغيل هذا الدفتر بالكامل تسلسلياً (أو عبر `%run` من دفتر `main` لجلب التعريفات فقط) كان يستهلك حصة Binance API الفعلية بصمت، خلافاً لتحذير القسم نفسه ("تشغيل هذه الخلية يستهلك حصص Binance API الفعلية") ولاتساق كل خلايا الأمثلة الأخرى (معرَّفة/معلَّقة بلا تنفيذ فعلي) | عُلِّقت الاستدعاءات الفعلية في الخلية بنفس نمط بقية خلايا الأمثلة — تبقى مرجعاً جاهزاً للنسخ، بلا تنفيذ تلقائي | **مرفوض:** حراسة الخلية بشرط بيئة (مثل `if RUN_LIVE_EXAMPLE:`) بدل التعليق — يُبقي احتمال تفعيلها بالخطأ عبر متغيّر عامّ منسي من جلسة سابقة |
| 19 | خلية في قسم «مثال استخدام شامل (تاريخي)» عرَّفت `load_asse()` — دالة مكرِّرة لـ`load_asset_registry()` لكن عبر رابط Drive عام (`pd.read_csv('https://drive.google.com/uc?id=...')`) — **واستدعتها فوراً** (`registry = load_asse()`) بلا تعليق، فتُحاول اتصالاً شبكياً حقيقياً في كل مرّة يُشغَّل فيها الدفتر بالكامل، مخالِفةً تحذير القسم نفسه وسياسة "Drive مُركَّب فقط" المُعلَنة في الرأس | حُذفت الدالة والاستدعاء، واستُبدلا بتعليق يوجّه لاستخدام `load_asset_registry()` الحقيقية (Drive مُركَّب، بلا رابط عام) | **مرفوض:** إبقاء الدالة معرَّفة بلا استدعاء فقط — تبقى فخّاً لأي نسخ-لصق مستقبلي لكودها القديم بدل الدالة الصحيحة الموجودة فعلاً |

**تنبيه إضافي:** تغيّرت **دلالة** `{t}_reg` (عائد مباشر لا سعر مُعاد تسويته)، فأي نموذج أو
معايرة (`inverse_scale`) بُنيت على السلوك القديم لم تعد صحيحة إلا بالتحويل لـ
`reg_target_mode='window_scale'` صراحةً أو تحديث كود العكس لاستخدام `invert_reg_predictions`.

**تنبيه:** تغيير عدد الميزات يعني أن أي بيانات محفوظة سابقاً (`preprocessing_output_*.pkl.gz`) وأي نموذج دُرِّب عليها **لا يتوافقان** مع هذه النسخة — أعد بناء البيانات ثم التدريب.

الاختبارات: القسم «19-ب» يشغّل `run_pipeline_selftests()` بلا Drive.


**⚠️ تنبيه هام (٢٠) — أثر مرجع "نفس النوع" على `high_reg`/`low_reg` (اكتُشف أثناء بناء بنية بحث الإشارات):**
هدفا `high_reg`/`low_reg` (`reg_target_mode='return'`) يُقارَنان بـ**نفس نوعهما**
من الشمعة الأخيرة (`last_high`/`last_low`) لا بـ`last_close` — قرار مقصود
لمواءمة مرجع رأسي `_class`/`_reg` لنفس الهدف (راجع تعليق `prepare_single_asset`).
لكن هذا يجعل `last_high`/`last_low` نفسهما **مقياساً لحجم ظل الشمعة الأخيرة**
(`last_high` مرتفع حين يكون الظل العلوي كبيراً، بصرف النظر عن أي حركة سعرية
لاحقة) — فأي ميزة تصف شكل الشمعة (`BODY_ratio`، `WICK_upper`، `WICK_lower`،
وبدرجة أقل `RET_1` المرتبط بها) تُظهر ارتباطاً ظاهرياً **قوياً جداً** (سبيرمان
≈+0.51 قِيس فعلياً على بيانات حقيقية) مع `y_high_reg`/`y_low_reg` — **لا علاقة
له بالتنبؤ الحقيقي**: حين يُعاد حساب نفس الهدف بمرجع موحّد (`last_close` لكل
الأهداف)، يتلاشى الارتباط تماماً (من +0.51 إلى ≈+0.01–0.05). السبب رياضي بحت:
شكل الشمعة الأخيرة يحدّد ضمنياً **مقام** `y_high_reg`/`y_low_reg` نفسه (عبر
`last_high`/`last_low`)، لا اتجاه السعر المستقبلي.

**الأثر العملي:** أي بحث عن إشارات (خاصة ميزات مرتبطة بالتقلب/شكل الشمعة/المدى:
`BODY_ratio`, `WICK_*`, `NATR_14`, `RANGE_rel`, `POS_14/50`, `RET_1`) ضد
`high_reg`/`low_reg` مباشرة **مُعرَّض لنتائج متضخّمة زائفة** بهذا السبب تحديداً
— يجب التحقق دائماً بمرجع `last_close` الموحّد (أو الاقتصار على `close_reg`،
غير المتأثّر لأنه أصلاً بمرجع `last_close`) قبل قبول أي إشارة ضد هذين الهدفين.
لم يُعدَّل تعريف الهدف نفسه هنا (قرار "نفس النوع" له مبرره الموثَّق في تحسين
`head_agreement`، وتغييره يتطلّب إعادة تدريب) — هذا توثيق للأثر الجانبي فقط،
مُطبَّق كحارس في `signal_discovery_lab.ipynb` (دالة `clean_reg_target` بمرجع
موحّد) لأي بحث إشارات مستقبلي.

## 1) تثبيت الاعتماديات


In [30]:
# pandas_ta_classic: المؤشرات الفنية (متوافقة مع pandas الحديث).
# python-binance:    عميل Binance الرسمي — جلب شموع OHLCV حيّة (futures_klines).
# ccxt:              يُستخدَم فقط لسرد رموز العقود الآجلة النشطة (load_markets).
!pip install -q pandas_ta_classic python-binance ccxt


## 2) الاستيرادات العامة


In [31]:
from __future__ import annotations

import gc
import json
import math
import os
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from copy import deepcopy
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, Iterator, List, Optional, Tuple, TypeVar

import numpy as np
import pandas as pd

T = TypeVar('T')
R = TypeVar('R')


## 3) Google Drive — التركيب (الطريقة الوحيدة للتحميل)

كل تحميل بيانات في هذا الدفتر يمرّ عبر `mount_drive()`: تركيب Drive عبر
`google.colab.drive`، ثم القراءة كملفات محلية من `MyDrive/...`. لا استخدام
لأي رابط عام أو `file_id` قابل للمشاركة، ولا لمكتبة `gdown`.


In [32]:
def is_colab() -> bool:
    """True إن كانت البيئة الحالية Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


#: جذر Drive المُركَّب — يُملأ مرة واحدة لكل جلسة (كاش بسيط).
_DRIVE_ROOT: Dict[str, Optional[Path]] = {"root": None}


def mount_drive(mount_point: Optional[str] = None,
                config: Optional[dict] = None) -> Optional[Path]:
    """يُركِّب Google Drive (مرة واحدة فقط لكل جلسة) ويُرجع مسار ``MyDrive``.

    يُرجع ``None`` خارج Colab بدل رفع استثناء مباشرة، فتبقى الدالة المستدعِية
    هي من تقرّر: بعضها يرفع خطأ واضحاً وبعضها يجرّب بديلاً.
    """
    config = CONFIG if config is None else config
    if not is_colab():
        return None
    if _DRIVE_ROOT["root"] is not None:
        return _DRIVE_ROOT["root"]

    from google.colab import drive

    mount_point = mount_point or config.get("drive_mount_point", "/content/drive")
    drive.mount(mount_point)
    root = Path(mount_point) / "MyDrive"
    _DRIVE_ROOT["root"] = root
    return root


## 4) ⚙️ الإعدادات الافتراضية (`defaults.py` سابقاً)

**المصدر الوحيد للحقيقة** لكل إعدادات النظام — منقول هنا **حرفياً** كما ورفعته
(لا تخمين هذه المرة). لا تُعدّل `DEFAULT_CONFIG` مباشرة أثناء العمل — استخدم
`update_config()` بعد بناء `CONFIG` في القسم التالي.

أضفتُ **مفتاحَين فقط** لم يكونا في الأصل (معلَّمين أدناه بوضوح) لأن الدفتر
يعتمد حصراً على Google Drive **المُركَّب** لا على `gdown`:
`drive_mount_point` و`asset_registry_path`. المفتاحان الأصليان
`asset_registry_file_id`/`preprocessed_data_file_id` (لمسار gdown القديم)
بقيا في القاموس للتوثيق فقط — **غير مُستخدَمين** فعلياً في هذا الدفتر.


In [33]:
"""
الإعدادات الافتراضية الموحّدة لكل النظام.

هذا الملف هو **المصدر الوحيد للحقيقة**: كل مرحلة (تجهيز البيانات، بناء النموذج،
التدريب، الفحص والتحقق) تقرأ إعداداتها من هنا عبر ``CONFIG`` (القسم التالي).
لا تُعدّل هذا القاموس مباشرة أثناء العمل — استخدم ``update_config()`` أو حمّل
ملف JSON عبر ``load_config()``.
"""

DEFAULT_CONFIG: dict = {
    # ══════════════════════════════════════════════════════════════════════
    # 1) الفريمات الزمنية والنوافذ
    # ══════════════════════════════════════════════════════════════════════
    "tf_order": ["1D"],
    "base_tf": "1D",
    "window_sizes": {"1D": 32},
    "stride": 1,
    "higher_tf_offset": 2,

    # ══════════════════════════════════════════════════════════════════════
    # 2) الأهداف
    # ══════════════════════════════════════════════════════════════════════
    "targets": ["high", "low", "close"],
    "forecast_horizon": 1,
    # "direction" → أهداف ±1 ، "regression" → سعر مُطبَّع.
    # ملاحظة: رؤوس '_class' تُدرَّب دائماً كاتجاه ±1، ورؤوس '_reg' كسعر مُطبَّع،
    # لذا هذا المفتاح يؤثر فقط على الأدوات القديمة أحادية الرأس.
    "target_mode": "direction",

    # ══════════════════════════════════════════════════════════════════════
    # 3) التحكم في الرؤوس (المخرجات) — مفتاح إطفاء/تشغيل لكل رأس على حدة
    # ══════════════════════════════════════════════════════════════════════
    # لكل هدف سعري رأسان: '_class' (تصنيف اتجاه ±1) و '_reg' (سعر مُطبَّع).
    # ضع False لأي رأس لا تحتاجه، فيُحذف بالكامل من: تحضير البيانات (لا يُحسب
    # أصلاً)، بناء النموذج (لا تُبنى طبقة Dense له)، الخسارة، التنبؤ، والتقييم.
    "enabled_heads": {
        "high_class": True,
        "high_reg": True,
        "low_class": True,
        "low_reg": True,
        "close_class": True,
        "close_reg": True,
    },

    # أوزان الخسارة لكل هدف سعري (يُطبَّق نفس الوزن على رأسي الهدف).
    # 0.0 يُبقي الرأس موجوداً كمخرج لكنه لا يؤثر في التدريب إطلاقاً.
    "target_loss_weights": {"high": 1.0, "low": 1.0, "close": 1.0},

    # ══════════════════════════════════════════════════════════════════════
    # 4) طبقة الامتناع (Selective Prediction / Learning to Reject)
    # ══════════════════════════════════════════════════════════════════════
    # قابلة للإطفاء الكامل بمفتاح واحد ("enabled": False) دون تغيير أي كود.
    "abstention": {
        "enabled": True,

        # (أ) رأس عدم اليقين داخل النموذج (المسار البديل build_classification_heads):
        # True  → رؤوس '_class' تُبنى بـ MC-dropout وتُخرج epistemic حقيقية.
        # False → رؤوس Dense(tanh) بسيطة، والشكوك تُعامَل كأصفار.
        # ⚠️ تغيير هذه القيمة يُغيّر بنية النموذج ⇒ يتطلب إعادة تدريب.
        # ✅ مُعطَّلة: predict_batch لم يعد يُشغّل MC-Dropout وقت الاستدلال إطلاقاً
        # (عدم اليقين يأتي تحليلياً من نموذج NIG)، فتفعيلها يبني طاقة Dropout
        # بلا أي استفادة فعلية منها.
        "use_uncertainty_head": False,
        "head_hidden_dim": 128,
        "head_dropout": 0.15,

        # (ب) إشارات الثقة المُدمَجة (Meta-Confidence) — كل إشارة قابلة للإطفاء:
        "signals": {
            "margin": True,          # هامش |tanh| بعيداً عن الصفر
            "uncertainty": True,     # epistemic + aleatoric
            "head_agreement": True,  # تطابق اتجاه رأسي class و reg لنفس الهدف
        },

        # (ج) عتبات القرار (تُعاير على val عبر calibrate_abstention_thresholds):
        "min_margin": 0.20,
        "max_uncertainty_ratio": 1.0,
        "require_head_agreement": True,
        "min_coverage": 0.05,

        # (د) عتبة ديناميكية حسب التقلب:
        #     العتبة الفعلية = min_margin * (1 + k*(vol_ratio-1))
        "dynamic_by_volatility": False,
        "volatility_sensitivity": 0.5,

        # (هـ) تكاليف اقتصادية حقيقية (نسبة مئوية من السعر):
        "cost_per_trade_pct": 0.06,
        "slippage_pct": 0.02,
        "opportunity_cost_pct": 0.0,

        # (و) الرأس الذي تُبنى عليه قرارات الصفقات:
        "decision_head": "close_class",
    },

    # ══════════════════════════════════════════════════════════════════════
    # 5) التطبيع والتقسيم
    # ══════════════════════════════════════════════════════════════════════
    "scaler_type": "robust",          # "robust" أو "minmax"
    "scaling_window": 32,
    "eps": 1e-8,

    # وضع التقسيم:
    #   "global_time" (مُوصى به) — حدود زمنية مطلقة مشتركة بين كل العملات.
    #   "per_asset" — السلوك القديم (نسب لكل عملة على حدة). للتوافق فقط.
    "split_mode": "global_time",
    "split_dates": None,
    "train_pct": 0.80,
    "val_pct": 0.10,
    "keep_asset_test_separate": True,
    "embargo_candles": None,          # None → window_size(base_tf) + forecast_horizon

    # أقل عدد عيّنات مقبول في كلٍّ من train و val و test. تحت هذا الحدّ يرفع
    # split_data خطأً صريحاً بدل إرجاع قسم فارغ بصمت (كان val و test يخرجان
    # فارغتين حين يقصر الزمن المتبقي بعد train عن فجوتَي العزل).
    "min_split_samples": 64,
    # 🔧 فريم التحميل الحيّ (Binance) — مستقلّ الآن عن فريم عمل النموذج
    # (tf_order/base_tf): يُجلَب دائماً بهذا الفريم ثم يُجمَّع (resample) صعوداً
    # إلى كل فريم في tf_order. كان قبلاً مُثبَّتاً "1h" بصمت داخل fetch_data
    # بصرف النظر عمّا يُطلَب فعلاً — الآن صريح ومُعدَّل هنا فقط.
    "download_interval": "1h",

    # 🔧 سياق سوقي عابر للأصول: عائد عملة مرجعية (BTC افتراضياً) يُضاف كميزة
    # إدخال (بادئة MKT_) لكل عملة أخرى — تُشتقّ آلياً في feature_order، لا حاجة
    # لتضمينها يدوياً. للعملة المرجعية نفسها القيمة صفر دائماً (لا معنى لعائدها
    # نسبة لنفسها). التوسعة لاحقاً (قيمة سوقية إجمالية، هيمنة BTC...) تحتاج
    # مصدر بيانات آخر غير Binance — أضف مفتاحاً موازياً هنا حين يتوفّر.
    "market_context": {
        "enabled": True,
        "reference_symbol": "BTCUSDT",
        "return_periods": [1],        # أضف فترات أطول لاحقاً: [1, 3, 7]...
    },

    # 🔧 إعادة تدريب دورية (walk-forward) — قيم افتراضية لـ rolling_splits().
    # None يعني "يجب تحديدها صراحة عند الاستدعاء"؛ لا قيمة واحدة صحيحة عامة
    # لكل مشروع. مثال: {"test_span": "30D", "val_span": "15D",
    # "initial_train_span": "180D"} (نافذة تدريب متمدّدة) أو أضف "train_span"
    # لنافذة منزلقة بحجم ثابت. راجع توثيق rolling_split_schedule.
    "rolling_retrain": {
        "test_span": None,
        "val_span": None,
        "train_span": None,
        "initial_train_span": None,
        "step": None,
    },

    # 🔧 معدّل التمويل والفائدة المفتوحة (عقود Binance الآجلة) — تُجلَب وتُخزَّن
    # على Drive بدالتَي fetch_funding_rate/fetch_open_interest_hist و
    # save_funding_open_interest، ثم تُدمَج كميزات فعلية عبر
    # add_funding_oi_features (بروح add_market_context، لكن بأرشيف خاص بكل
    # عملة لا مرجعاً مشتركاً) — "as_feature" يتحكّم بالإدماج بمعزل عن
    # "enabled" (الجلب/الأرشفة)، فيمكن أرشفة بلا إدماج (كان السلوك الوحيد
    # سابقاً) أو العكس نظرياً (إدماج بلا enabled يُعيد أعمدة محايدة دائماً).
    # ⚠️ الفائدة المفتوحة محدودة من Binance نفسها بآخر ٣٠ يوماً فقط مهما
    # طُلب — راجع تحذير fetch_open_interest_hist. معدّل التمويل بلا هذا القيد،
    # لكن الأرشيف المحفوظ فعلياً يبدأ من أول تشغيل لـsave_funding_open_interest؛
    # فترات أقدم منه (شائعة في بيانات تاريخية طويلة) تُملأ بقيمة محايدة مع علم
    # توفّر صريح (FUND_available/OI_available) — راجع add_funding_oi_features.
    "funding_rate": {"enabled": True, "drive_dir": "funding_rate",
                     "as_feature": True, "zscore_window": 90},
    "open_interest": {"enabled": True, "drive_dir": "open_interest", "period": "1h",
                      "as_feature": True, "change_period": 1},


    # 🔧 كيف يُشتقّ حدّ (حدّا) التقسيم الزمني في split_mode='global_time':
    #   'kept_share'   (افتراضي، القائم) — يبحث عن الحدّين بحيث تقترب نِسَب
    #                  العيّنات **المُبقاة بعد فجوتَي العزل** من train_pct/val_pct.
    #                  الأدقّ، ويتعامل بنجاح مع بيانات مكدَّسة زمنياً (عملات كثيرة
    #                  حديثة الإدراج) — انظر توثيق resolve_split_dates.
    #   'raw_quantile' — حدّ مباشر: "آخر N من كل العيّنات (أي عملة)" يقارب
    #                  train_pct/val_pct **الخام قبل** خصم فجوة العزل — أبسط،
    #                  لكن الفرق عن 'kept_share' يتناسب مع (عيّنات العزل
    #                  المفقودة ÷ حجم القسم المستهدف)، فقد ينحرف بعشرات
    #                  بالمئة حتى على بيانات منتظمة إن كان val_pct/test_pct
    #                  صغيراً؛ وعلى بيانات مكدَّسة قد يُنتج أقساماً أصغر من
    #                  min_split_samples (يُرفَض بخطأ صريح حينها، لا قسم فارغ
    #                  صامت). راجع compute_global_cutoff و build_leak_free_split.
    "split_cutoff_method": "kept_share",

    # يُستبعد من **مدخلات النموذج** فقط — تبقى الأعمدة في البيانات لحساب
    # الأهداف والمؤشرات. قيست هذه الاستبعادات لا خُمِّنت:
    #   open        : مكرَّر مع إغلاق الشمعة السابقة
    #   high / low   : ترابط 0.99 مع close بعد التطبيع؛ معلومتها الحقيقية
    #                  (المدى وبنية الشمعة) تلتقطها RANGE_rel و BODY_ratio و WICK_*
    #   volume      : ترابط 0.96 مع VOLZ_20، وهذا الأخير مرجعه متدحرج أدقّ
    "exclude_from_features": ["open", "high", "low", "volume"],

    # تُملأ تلقائياً عبر refresh_features() — لا تكتبها يدوياً.
    "feature_order": None,

    # ══════════════════════════════════════════════════════════════════════
    # 6) المؤشرات الفنية
    # ══════════════════════════════════════════════════════════════════════
    "indicator_settings": {
        # ── اتجاه ────────────────────────────────────────────────────────
        'ema': [9, 26],              # متوسطان سريع/بطيء → الميل والتقاطع
        'adx': [14],                 # قوة الاتجاه (تُخرج ADX + DMP + DMN)
        # ── زخم ──────────────────────────────────────────────────────────
        'rsi': [14],                 # مذبذب الزخم المرجعي
        'stoch': [(14, 3)],          # موضع ضمن النطاق + تقاطع k/d
        'macd': [(12, 26, 9)],       # زخم اتجاهي بإشارة صفر ذات معنى
        # ── تقلب ─────────────────────────────────────────────────────────
        'natr': [14],                # مدى حقيقي **كنسبة** — خالٍ من المقياس
        'bbands': [20],              # نطاقات + عرض (BBB) + الموضع (BBP)
        # ── حجم وتدفق ────────────────────────────────────────────────────
        'mfi': [14],                 # زخم مرجَّح بالحجم
        'cmf': [20],                 # تدفق أموال تشايكين
        # ── مؤشرات إضافية متاحة (أزل التعليق — FEATURES تُشتق تلقائياً) ──
        # 'sma': [20], 'ppo': [(12,26,9)], 'mom': [10], 'roc': [10],
        # 'cci': [20],   ← أُسقط: ترابط 0.963 مع BBP
        # 'aroon': [14], 'willr': [14], 'er': [10], 'tsi': [13],
        # 'trix': [14], 'fisher': [9], 'slope': [14], 'chop': [14],
        # 'atr': [14], 'stdev': [14], 'zscore': [20], 'ui': [14],
        # 'efi': [13], 'adosc': [3], 'qstick': [14], 'kurtosis': [20],
    },
    # مؤشرات بلا معامل طول. OBV تراكمي يكمّل CMF/MFI بمنظور مختلف.
    "static_indicators": [],   # كان ['obv'] — أُوقف بقرارك؛ أعده بإضافة 'obv' هنا
    "static_indicator_params": {
        "log_return": {"length": 14},
        "percent_return": {"length": 14},
    },

    # ══════════════════════════════════════════════════════════════════════
    # 6-ب) الميزات المخصّصة (ساكنة وخالية من المقياس — انظر add_custom_features)
    # ══════════════════════════════════════════════════════════════════════
    "use_custom_features": True,
    "custom_settings": {
        "returns": [1, 3, 6, 12, 24],
        "range_windows": [14, 50],
        "vol_windows": [(6, 24), (12, 48)],
        "volume_window": 20,
        "candle_structure": True,
        "time_features": True,
    },

    # أقصى قيمة مطلقة بعد التطبيع (قصّ يمنع انفجار التدرّجات من شذوذ واحد).
    "clip_abs": 5.0,

    # 🔧 كيف يُحسب هدف الانحدار {t}_reg — مستوحى من عائد Qlib المباشر
    # (Ref($close,-2)/Ref($close,-1)-1) بدل سعر مُعاد تسويته بإحصاء خارجي:
    #   'return'       (افتراضي جديد) — عائد مباشر: (السعر_المستقبلي /
    #                  آخر_سعر_من_نفس_نوع_الهدف) - 1. بلا مرجع خارجي (لا وسيط
    #                  نافذة ولا IQR)، فنفس نسبة الحركة تُنتج نفس القيمة بصرف
    #                  النظر عن أي نافذة سقطت فيها العيّنة أو أي عملة. راجع
    #                  سجلّ التعديلات في رأس الدفتر لتفصيل الأثر على مشكلة
    #                  الانكماش نحو المتوسط.
    #   'window_scale' (القديم) — (السعر_المستقبلي - آخر_إغلاق) / IQR نافذة
    #                  الإغلاق، بصرف النظر عن نوع الهدف. للمقارنة أو التراجع.
    "reg_target_mode": "return",
    # قصّ هدف الانحدار بعد حسابه. None يختار افتراضاً حسب reg_target_mode
    # (1.0 أي ±100% لـ'return'، أو 10.0 كسابقاً لـ'window_scale') — الرقمان
    # بوحدتين مختلفتين تماماً فلا يصحّ ترك رقم واحد ثابتاً بين الوضعين. حدّده
    # صراحةً فقط إن أردتَ تجاوز اختيار الوضع التلقائي.
    "reg_target_clip": None,

    # 🔧 تطبيع مقطعي اختياري لأهداف الانحدار عبر كل الأصول عند نفس اللحظة —
    # بروح Qlib CSZScoreNorm/CSRankNorm على حقل label (وليس محاكاة حرفية لكودها).
    # كل عيّنة تُقاس نسبة إلى توزيع عوائد أقرانها في نفس اللحظة، لا نسبة إلى
    # ماضيها هي — فتصير قابلة للمقارنة عبر عملات مختلفة التقلّب في نفس اللحظة،
    # خلاف العائد الخام وحده (reg_target_mode='return') الذي يبقى ثابت المعنى
    # عبر الزمن لكن ليس عبر عملات مختلفة القوة في نفس اللحظة. يُطبَّق صراحةً
    # بعد build_dataset عبر cross_sectional_normalize() — ليس تلقائياً، لأنه
    # يُحوِّل دلالة الرأس من عائد مطلق إلى أداء نسبي مقابل الأقران (راجع
    # توثيق الدالة قبل استخدامه).
    "cs_norm": {
        "method": "zscore",     # 'zscore' (افتراضي، قابل للعكس) أو 'rank' ([-1,1] ترتيبي)
        "min_assets": 5,        # أقل عدد أصول باللحظة الواحدة ليُحسب التطبيع؛ وإلا يبقى العائد الخام
        "clip": 5.0,            # قصّ بعد 'zscore' فقط ('rank' محصورة أصلاً)
    },

    # ══════════════════════════════════════════════════════════════════════
    # 7) مصادر البيانات
    # ══════════════════════════════════════════════════════════════════════
    # ⚠️ القيمتان التاليتان من الأصل (مسار gdown القديم) — غير مُستخدَمتين في
    # هذا الدفتر (Drive المُركَّب فقط)، أُبقيتا للتوثيق.
    "asset_registry_file_id": "1hITH2zvR4plZeeKq5W1KSZO2RM_cHdU_",
    "preprocessed_data_file_id": "1EWTFjth2swnnTbnJ4gV_doFDfsJ4iqzP",

    # مسار ملفات العملات الخام على Drive **المُركَّب** (لا تنزيل عام).
    "drive_raw_dir": "history_1d",
    "drive_raw_pattern": "{name}.csv",
    # 🔧 إضافة جديدة (غير موجودة في الأصل): مسار سجل الأصول على Drive
    # المُركَّب — بديل asset_registry_file_id بعد إزالة مسار gdown بالكامل.
    "asset_registry_path": "crypto_data/asset_registry.csv",
    # 🔧 إضافة جديدة: نقطة تركيب Drive (يستخدمها mount_drive أعلاه).
    "drive_mount_point": "/content/drive",

    # 🔧 إضافة جديدة: أسماء عملات تُتخطّى كلياً في build_dataset* — قبل محاولة
    # التحميل أصلاً. المطابقة بلا حساسية لحالة الأحرف ولا للمسافات الطرفية.
    # تسري أيضاً على build_dataset_live. انظر exclude_coins.
    "excluded_coins": [],

    # 🔧 إضافة جديدة: الحدّ الأقصى لعدد طلبات Binance في الدقيقة في الجلب الحيّ.
    # يُطبَّق على **كل** طلب futures_klines (كل صفحة وكل إعادة محاولة) ومشترك بين كل
    # الخيوط بنافذة منزلقة. 0 أو None يعطّل الحدّ. ⚠️ هذا حدّ على *عدد* الطلبات؛
    # Binance تقيّد بـ«الوزن» لكل IP، وتكلفة الطلب تكبر مع limit (راجع rateLimits
    # في exchangeInfo أو الترويسة x-mbx-used-weight-1m).
    "live_max_requests_per_minute": 2000,

    # ══════════════════════════════════════════════════════════════════════
    # 8) بنية النموذج (RoPE + GQA + MoE Transformer)
    # ══════════════════════════════════════════════════════════════════════
    "model_tf": "4h",          # الفريم المُغذّى فعلياً للنموذج
    "d_model": 64,
    "num_layers": 3,
    "num_heads": 4,
    "num_kv_heads": 2,
    "kv_latent": None,         # None → d_model // 4
    "expansion": 8 / 3,
    "model_dropout": 0.15,
    "model_name": "ModernTST",

    "use_decomposition": True,
    "decomp_kernels": None,

    "moe_every": 0,
    "n_shared": 2,
    "n_routed": 4,
    "top_k": 2,

    # ══════════════════════════════════════════════════════════════════════
    # 9) التدريب
    # ══════════════════════════════════════════════════════════════════════
    "learning_rate": 5e-4,
    "gradient_clipnorm": 1.0,
    "epochs": 100,
    "batch_size": 64,
    "use_class_weights": True,
    "class_weight_smoothing": 0.5,
    "early_stopping_patience": 30,
    "reduce_lr_patience": 10,
    "reduce_lr_factor": 0.5,

    # ══════════════════════════════════════════════════════════════════════
    # 10) التقييم
    # ══════════════════════════════════════════════════════════════════════
    "eval_batch_size": 256,
    "eval_range_frac": 0.2,
    "eval_n_display": 5,

    # ══════════════════════════════════════════════════════════════════════
    # 11) التخزين (تُحلّ تلقائياً حسب البيئة: Colab / محلي)
    # ══════════════════════════════════════════════════════════════════════
    "project_name": "crypto_model",
    "workspace_dir": None,     # None → يُكتشف تلقائياً
    "use_drive": True,         # يُتجاهَل تلقائياً خارج Colab
}

#: قائمة المؤشرات الثابتة الكاملة (غير مُفعَّلة افتراضياً — انظر التعليق أعلاه).
STATIC_INDICATORS_FULL = [
    'obv', 'ad', 'kvo', 'true_range', 'log_return',
    'percent_return', 'bop', 'uo', 'hlc3', 'wcp',
]

#: كون العملات الافتراضي — يُستخدم في صفحة تجهيز البيانات.
COINS_BY_CATEGORY = {
    "Large_Caps": [
        "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
        "ADAUSDT", "AVAXUSDT", "DOTUSDT0", "DOGEUSDT", "TRXUSDT",
        "LINKUSDT", "LTCUSDT", "ATOMUSDT", "UNIUSDT", "TONUSDT",
        "ICPUSDT", "ETCUSDT", "FILUSDT", "APTUSDT", "XLMUSDT",
        "INJUSDT", "NEARUSDT", "XMRUSDT", "OPUSDT",
    ],
    "DeFi": [
        "AAVEUSDT", "COMPUSDT", "SNXUSDT", "SUSHIUSDT",
        "LDOUSDT", "RUNEUSDT", "FXSUSDT", "PENDLEUSDT", "DYDXUSDT",
    ],
    "AI_BigData": [
        "TAOUSDT", "FETUSDT", "RNDRUSDT", "AKTUSDT", "NMRUSDT",
        "GRTUSDT", "PHBUSDT", "AIUSDT",
    ],
    "Gaming_Metaverse": [
        "SANDUSDT", "MANAUSDT", "IMXUSDT", "GALAUSDT", "AXSUSDT",
        "ENJUSDT", "YGGUSDT", "MAGICUSDT", "PORTALUSDT", "PIXELUSDT",
    ],
    "Meme": [
        "1000SHIBUSDT", "1000PEPEUSDT",
    ],
    "Layer1_Layer2": [
        "SUIUSDT", "SEIUSDT", "ALGOUSDT", "EGLDUSDT", "ARBUSDT",
        "STRKUSDT", "ZETAUSDT", "KASUSDT", "TIAUSDT", "HBARUSDT",
        "EOSUSDT", "XTZUSDT",
    ],
    "Various_Services": [
        "VETUSDT", "THETAUSDT", "QNTUSDT", "CHZUSDT", "HOTUSDT",
        "BATUSDT", "ZILUSDT", "ONTUSDT", "IOTXUSDT", "ONEUSDT",
        "CELOUSDT", "CFXUSDT", "ANKRUSDT", "STORJUSDT", "DENTUSDT",
        "GASUSDT", "NEOUSDT", "DASHUSDT",
    ],
}

#: الفئات المُفعَّلة افتراضياً.
DEFAULT_ENABLED_CATEGORIES = ["Large_Caps"]


## 5) الإعدادات الحيّة `CONFIG` (`runtime.py` سابقاً)

``CONFIG`` هو **نفس القاموس** أينما استُخدم في هذا الدفتر — تعديله عبر
`update_config()` يسري فوراً على كل الدوال (لأنها كلها تقرأ نفس المرجع).
لذلك يُعدَّل دائماً **في المكان** ولا يُعاد إسناده أبداً بـ `CONFIG = {...}`.


In [34]:
"""
كائن الإعدادات الحيّ ``CONFIG`` وأدوات تعديله وحفظه وتحميله.

``CONFIG`` هو **نفس القاموس** في كل مكان من الدفتر: أي دالة تستخدمه تحصل على
المرجع ذاته، فتعديله مرة واحدة يسري فوراً على كل المراحل.
لذلك نُعدّله دائماً **في المكان** (in-place) ولا نُعيد إسناده أبداً.
"""

#: قاموس الإعدادات الحيّ — المرجع المشترك بين كل دوال هذا الدفتر.
CONFIG: Dict[str, Any] = deepcopy(DEFAULT_CONFIG)


def _deep_update(base: dict, new: dict) -> dict:
    """دمج متداخل: القواميس تُدمج، وأي قيمة أخرى تُستبدل."""
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            _deep_update(base[k], v)
        else:
            base[k] = v
    return base


def get_config() -> Dict[str, Any]:
    """يُرجع مرجع الإعدادات الحيّ (وليس نسخة)."""
    return CONFIG


def update_config(overrides: Optional[dict] = None, **kwargs) -> Dict[str, Any]:
    """تحديث ``CONFIG`` في المكان بدمج متداخل.

    >>> update_config({"abstention": {"min_margin": 0.3}}, epochs=50)
    """
    if overrides:
        _deep_update(CONFIG, overrides)
    if kwargs:
        _deep_update(CONFIG, kwargs)
    return CONFIG


def reset_config() -> Dict[str, Any]:
    """إعادة ``CONFIG`` إلى القيم الافتراضية (في المكان)."""
    CONFIG.clear()
    CONFIG.update(deepcopy(DEFAULT_CONFIG))
    return CONFIG


def _jsonable(obj: Any) -> Any:
    """تحويل القيم غير القابلة للتسلسل (tuples داخل indicator_settings) إلى قوائم."""
    if isinstance(obj, dict):
        return {k: _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    return obj


def _retuple_indicators(cfg: dict) -> dict:
    """يُعيد القوائم المتداخلة داخل ``indicator_settings`` إلى tuples.

    ``add_features`` تفكّ ``(fast, slow, signal)`` بالتفريغ، وJSON لا يحفظ الـ tuple.
    """
    settings = cfg.get("indicator_settings")
    if isinstance(settings, dict):
        for ind, values in settings.items():
            if isinstance(values, list):
                settings[ind] = [tuple(v) if isinstance(v, list) else v for v in values]
    return cfg


def save_config(path, config: Optional[dict] = None) -> Path:
    """حفظ الإعدادات كملف JSON (يشمل ``feature_order`` المُشتقة)."""
    config = CONFIG if config is None else config
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(_jsonable(config), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"💾 حُفظت الإعدادات: {path}")
    return path


def load_config(path, merge: bool = True) -> Dict[str, Any]:
    """تحميل إعدادات من ملف JSON إلى ``CONFIG`` الحيّ.

    Args:
        merge: True يدمج فوق الحالي، False يعيد التعيين للافتراضي أولاً.
    """
    path = Path(path)
    loaded = _retuple_indicators(json.loads(path.read_text(encoding="utf-8")))
    if not merge:
        reset_config()
    _deep_update(CONFIG, loaded)
    print(f"📂 حُمّلت الإعدادات: {path}")
    return CONFIG


def refresh_features(config: Optional[dict] = None, force: bool = True) -> List[str]:
    """إعادة اشتقاق ``feature_order`` من ``indicator_settings`` الحالية.

    استدعِها بعد أي تعديل على المؤشرات أو ``exclude_from_features``.
    """
    config = CONFIG if config is None else config
    if not force and config.get("feature_order"):
        return config["feature_order"]
    features = infer_feature_columns(config)
    config["feature_order"] = features
    print(f"🧬 عدد الميزات المُشتقة: {len(features)}")
    return features


def feature_order(config: Optional[dict] = None) -> List[str]:
    """قائمة الميزات الحالية، تُشتق تلقائياً عند أول طلب إن لم تكن موجودة."""
    config = CONFIG if config is None else config
    if not config.get("feature_order"):
        refresh_features(config)
    return config["feature_order"]


def n_features(config: Optional[dict] = None) -> int:
    return len(feature_order(config))


def seq_len(config: Optional[dict] = None) -> int:
    """طول التسلسل المُغذّى للنموذج (نافذة الفريم ``model_tf``)."""
    config = CONFIG if config is None else config
    return config["window_sizes"][config["model_tf"]]


def describe(config: Optional[dict] = None) -> None:
    """ملخّص مقروء للإعدادات الفعّالة — يُستخدم في رأس كل صفحة."""
    config = CONFIG if config is None else config
    heads = get_target_heads(config=config)
    print("═" * 66)
    print("⚙️  الإعدادات الفعّالة")
    print("═" * 66)
    print(f"  الفريمات       : {config['tf_order']}  (أساسي={config['base_tf']}, "
          f"نموذج={config['model_tf']})")
    print(f"  النوافذ        : {config['window_sizes']}  | stride={config['stride']}")
    print(f"  الأهداف        : {config['targets']}  | أفق={config['forecast_horizon']}")
    print(f"  الرؤوس المُفعَّلة : {heads}")
    print(f"  أوزان الأهداف  : {config.get('target_loss_weights')}")
    print(f"  الامتناع       : {'مُفعَّل' if abstention_enabled(config) else 'مُعطَّل'}"
          f" | رأس عدم اليقين: "
          f"{'مُفعَّل' if uncertainty_head_enabled(config) else 'مُعطَّل'}")
    feats = config.get("feature_order")
    print(f"  الميزات        : {len(feats) if feats else 'لم تُشتق بعد'}")
    print(f"  النموذج        : d_model={config['d_model']}, layers={config['num_layers']}, "
          f"heads={config['num_heads']}/{config['num_kv_heads']}")
    print(f"  التدريب        : lr={config['learning_rate']}, epochs={config['epochs']}, "
          f"batch={config['batch_size']}")
    print("═" * 66)


## 6) أدوات رؤوس المخرجات (`heads.py` سابقاً)

كل هدف سعري (high / low / close) له رأسان: `{target}_class` (تصنيف اتجاه ±1)
و`{target}_reg` (قيمة انحدار). كل مراحل النظام تستدعي `get_target_heads` بدل
تثبيت الأسماء يدوياً، فإطفاء أي رأس في `CONFIG['enabled_heads']` يسري تلقائياً
على النظام كله — **هذا يستبدل** `get_target_heads` المبنية تخمينياً في النسخة
السابقة من هذا الدفتر (كانت تفترض `enabled_heads` قائمة أسماء، بينما هي فعلياً
قاموس `{اسم_الرأس: True/False}`).


In [35]:
"""
أدوات التعامل مع رؤوس المخرجات (heads).

كل هدف سعري (high / low / close) له رأسان:
  * ``{target}_class`` — تصنيف اتجاه ±1
  * ``{target}_reg``   — قيمة انحدار (سعر مُطبَّع)

كل مراحل النظام (تحضير البيانات، بناء النموذج، الخسارة، التنبؤ، التقييم) تستدعي
:func:`get_target_heads` بدل تثبيت الأسماء يدوياً، لذا إطفاء أي رأس في
``CONFIG['enabled_heads']`` يسري تلقائياً على النظام كله.
"""

HEAD_KINDS: Tuple[str, str] = ('class', 'reg')

#: بادئة اسم طبقة/مفتاح المخرج المقابل لكل رأس.
#: اسم الرأس ``'close_class'`` ⇄ اسم المخرج ``'y_close_class'``.
OUTPUT_PREFIX = 'y_'


def output_name(head: str) -> str:
    """اسم مخرج النموذج المقابل لرأس: ``'close_class'`` → ``'y_close_class'``."""
    return head if head.startswith(OUTPUT_PREFIX) else f'{OUTPUT_PREFIX}{head}'


def head_name(output: str) -> str:
    """عكس :func:`output_name`: ``'y_close_class'`` → ``'close_class'``."""
    return output[len(OUTPUT_PREFIX):] if output.startswith(OUTPUT_PREFIX) else output


def get_output_names(targets: Optional[List[str]] = None,
                     config: Optional[dict] = None) -> List[str]:
    """أسماء مخرجات النموذج للرؤوس المُفعَّلة، بالترتيب نفسه."""
    return [output_name(h) for h in get_target_heads(targets, config)]


def get_target_heads(targets: Optional[List[str]] = None,
                     config: Optional[dict] = None,
                     include_disabled: bool = False) -> List[str]:
    """يبني قائمة أسماء رؤوس المخرجات المُفعَّلة من قائمة الأهداف السعرية.

    مثال: ``['high','low','close']`` →
    ``['high_class','high_reg','low_class','low_reg','close_class','close_reg']``

    Args:
        targets: الأهداف السعرية (افتراضياً ``config['targets']``).
        config: قاموس الإعدادات (افتراضياً ``CONFIG``).
        include_disabled: True يُرجع كل الرؤوس متجاهلاً مفاتيح التعطيل (للتشخيص).
    """
    config = CONFIG if config is None else config
    targets = list(targets) if targets is not None else list(config['targets'])
    enabled = config.get('enabled_heads', {}) or {}
    heads: List[str] = []
    for t in targets:
        for kind in HEAD_KINDS:
            head = f'{t}_{kind}'
            if include_disabled or enabled.get(head, True):
                heads.append(head)
    return heads


def split_head_name(head: str) -> Tuple[str, str]:
    """``'low_reg'`` → ``('low', 'reg')``."""
    for kind in HEAD_KINDS:
        suffix = f'_{kind}'
        if head.endswith(suffix):
            return head[: -len(suffix)], kind
    raise ValueError(f"اسم رأس غير معروف (لا ينتهي بـ _class أو _reg): {head}")


def is_regression_head(head: str) -> bool:
    return split_head_name(head)[1] == 'reg'


def is_class_head(head: str) -> bool:
    return split_head_name(head)[1] == 'class'


def get_class_heads(config: Optional[dict] = None) -> List[str]:
    """رؤوس التصنيف المُفعَّلة فقط."""
    return [h for h in get_target_heads(config=config) if is_class_head(h)]


def get_reg_heads(config: Optional[dict] = None) -> List[str]:
    """رؤوس الانحدار المُفعَّلة فقط."""
    return [h for h in get_target_heads(config=config) if is_regression_head(h)]


def paired_targets(config: Optional[dict] = None) -> List[str]:
    """الأهداف التي رأساها (class و reg) مُفعَّلان معاً — يلزمها head_agreement."""
    config = CONFIG if config is None else config
    heads = get_target_heads(config=config)
    return [t for t in config['targets']
            if f'{t}_class' in heads and f'{t}_reg' in heads]


# ══════════════════════════════════════════════════════════════════════════
# طبقة الامتناع
# ══════════════════════════════════════════════════════════════════════════
def abstention_cfg(config: Optional[dict] = None) -> dict:
    """إعدادات الامتناع مع قيم افتراضية آمنة."""
    config = CONFIG if config is None else config
    return config.get('abstention', {}) or {'enabled': False}


def abstention_enabled(config: Optional[dict] = None) -> bool:
    return bool(abstention_cfg(config).get('enabled', False))


def uncertainty_head_enabled(config: Optional[dict] = None) -> bool:
    """هل تُبنى رؤوس التصنيف بطبقة عدم اليقين (MC-dropout)؟

    يتطلب أن يكون الامتناع مُفعَّلاً — تعطيله يُعيد الرؤوس البسيطة.
    """
    cfg = abstention_cfg(config)
    return bool(cfg.get('enabled', False) and cfg.get('use_uncertainty_head', False))


def signal_enabled(name: str, config: Optional[dict] = None) -> bool:
    """هل إشارة ثقة معيّنة (margin / uncertainty / head_agreement) مُفعَّلة؟"""
    cfg = abstention_cfg(config)
    if not cfg.get('enabled', False):
        return False
    return bool((cfg.get('signals', {}) or {}).get(name, False))


def decision_head(config: Optional[dict] = None) -> Optional[str]:
    """رأس القرار المستخدم في التقييم الانتقائي، مع تراجع آمن لأول رأس تصنيف متاح."""
    cfg = abstention_cfg(config)
    heads = get_target_heads(config=config)
    wanted = cfg.get('decision_head')
    if wanted and wanted in heads:
        return wanted
    class_heads = [h for h in heads if is_class_head(h)]
    return class_heads[0] if class_heads else None


# ══════════════════════════════════════════════════════════════════════════
# فحص التناسق
# ══════════════════════════════════════════════════════════════════════════
def validate_head_config(config: Optional[dict] = None,
                         verbose: bool = True) -> List[str]:
    """فحص تناسق إعدادات الرؤوس قبل التدريب — يكشف الأخطاء الصامتة مبكراً.

    يُرجع قائمة التحذيرات (فارغة = كل شيء متناسق).
    """
    config = CONFIG if config is None else config
    warnings_list: List[str] = []
    heads = get_target_heads(config=config)

    if not heads:
        warnings_list.append(
            "❌ كل الرؤوس مُعطَّلة في enabled_heads — لا يمكن بناء نموذج بلا مخرجات.")

    unknown = set(config.get('enabled_heads', {})) - set(
        get_target_heads(config=config, include_disabled=True))
    if unknown:
        warnings_list.append(
            f"⚠️ مفاتيح في enabled_heads لا تطابق أي هدف في targets: {sorted(unknown)}")

    model_tf = config.get('model_tf')
    if model_tf and model_tf not in config.get('tf_order', []):
        warnings_list.append(
            f"❌ model_tf='{model_tf}' غير موجود في tf_order={config.get('tf_order')} — "
            "لن يجد النموذج مصفوفة الإدخال الخاصة به.")
    if model_tf and model_tf not in config.get('window_sizes', {}):
        warnings_list.append(f"❌ لا يوجد window_size للفريم '{model_tf}'.")

    cfg = abstention_cfg(config)
    if cfg.get('enabled', False):
        wanted = cfg.get('decision_head')
        if wanted and wanted not in heads:
            warnings_list.append(
                f"⚠️ رأس القرار '{wanted}' مُعطَّل أو غير موجود — "
                "سيُختار أول رأس تصنيف متاح بدلاً منه.")
        if signal_enabled('uncertainty', config) and not uncertainty_head_enabled(config):
            warnings_list.append(
                "⚠️ إشارة 'uncertainty' مُفعَّلة لكن use_uncertainty_head=False — "
                "الشكوك ستكون أصفاراً فلن تُقصي أي عينة (الإشارة بلا أثر فعلي).")
        if signal_enabled('head_agreement', config) and not paired_targets(config):
            warnings_list.append(
                "⚠️ إشارة 'head_agreement' مُفعَّلة لكن لا يوجد هدف لديه رأسا "
                "class و reg مُفعَّلان معاً — الإشارة ستُتجاهل تلقائياً.")

    if verbose:
        if warnings_list:
            print("🔍 فحص إعدادات الرؤوس:")
            for w in warnings_list:
                print(f"   {w}")
        else:
            print(f"✅ إعدادات الرؤوس متناسقة | الرؤوس المُفعَّلة ({len(heads)}): {heads}")
            print(f"   الامتناع: {'مُفعَّل' if abstention_enabled(config) else 'مُعطَّل'}"
                  f" | رأس عدم اليقين: "
                  f"{'مُفعَّل' if uncertainty_head_enabled(config) else 'مُعطَّل'}")
    return warnings_list


def head_loss_weights(config: Optional[dict] = None,
                      as_outputs: bool = True) -> Dict[str, float]:
    """وزن الخسارة لكل رأس مُفعَّل، مشتقاً من ``target_loss_weights``.

    Args:
        as_outputs: True يُرجع المفاتيح بأسماء المخرجات (``y_...``) كما يتوقّعها
            ``model.compile``؛ False يُرجعها بأسماء الرؤوس المجرّدة.
    """
    config = CONFIG if config is None else config
    weights = config.get('target_loss_weights', {}) or {}
    key = output_name if as_outputs else (lambda h: h)
    return {key(h): float(weights.get(split_head_name(h)[0], 1.0))
            for h in get_target_heads(config=config)}


## 7) الميزات المخصّصة الساكنة (`custom.py` سابقاً)

ميزات خالية من المقياس بالبناء (عوائد لوغاريتمية، مدى نسبي، بنية الشمعة...)
تُعمِّم عبر كل الأصول بغضّ النظر عن مستوى سعرها.


In [36]:
"""
ميزات مخصّصة للعملات الرقمية — ساكنة (stationary) وخالية من المقياس.

**لماذا هذه ولم تكفِ مؤشرات ``pandas_ta``؟**

النموذج يتدرّب على عشرات العملات بمستويات أسعار تختلف بمقدار ستة أوامر عشرية
(بيتكوين 60,000 مقابل عملة بـ 0.00003). المؤشرات الكلاسيكية إمّا بوحدة السعر
(فتحتاج تطبيعاً لكل نافذة يمحو جزءاً من معلومتها) أو مذبذبات محصورة تكرّر بعضها.
الميزات هنا **خالية من المقياس بالبناء**: نِسَب ولوغاريتمات عوائد، فتعني الشيء
نفسه على أي عملة وفي أي حقبة سعرية — وهذا شرط أساسي للتعميم عبر الأصول.

قيس فعلياً على الميزات القديمة: ``high``/``low``/``close`` مترابطة بـ 0.98–0.99
بعد التطبيع، لأن معلومتها الحقيقية هي **الفرق بينها** لا مستواها. لذا نستبدلها
هنا بترميز غير مكرَّر: مدى نسبي + موضع الإغلاق داخل الشمعة.

العائلات:

============ ==========================================================
البادئة       المعلومة
============ ==========================================================
``RET_``     عوائد لوغاريتمية على آفاق متعددة — أقوى عائلة ساكنة
``RANGE_``   المدى النسبي (تقلب داخل الشمعة) بالنسبة المئوية
``BODY_``    نسبة جسم الشمعة إلى مداها — بنية الشمعة
``WICK_``    الظلال العليا/السفلى — ضغط الرفض
``POS_``     موضع الإغلاق داخل نطاق N شمعة — ارتداد/اختراق
``VOLR_``    لوغاريتم نسبة التقلب القصير للطويل — كاشف النظام السعري
``VOLZ_``    درجة معيارية للحجم — الحجم غير الاعتيادي
``TIME_``    موسمية يومية/أسبوعية (السوق يعمل 24/7 وله أنماط حقيقية)
``FRACTAL_`` نقاط تطرّف محلي مؤكَّدة (فركتال ويليامز)، سببية بالكامل عبر
             إزاحة نصف النافذة — مُعطَّلة افتراضياً (``fractal_windows=[]``)
============ ==========================================================
"""

EPS = 1e-12

#: الإعدادات الافتراضية للميزات المخصّصة.
DEFAULT_CUSTOM_SETTINGS: Dict[str, object] = {
    "returns": [1, 3, 6, 12, 24],      # آفاق العوائد اللوغاريتمية (بالشموع)
    "range_windows": [14, 50],         # نوافذ الموضع داخل النطاق
    "vol_windows": [(6, 24), (12, 48)],  # (قصير، طويل) لنِسَب التقلب
    "volume_window": 20,               # نافذة درجة الحجم المعيارية
    "candle_structure": True,          # جسم/ظلال الشمعة
    "time_features": True,             # موسمية الساعة/اليوم
    # نوافذ فركتال ويليامز (فردية فقط) — [] يعني مُعطَّلة تماماً (لا أعمدة
    # FRACTAL_* تُنتَج). راجع docstring _add_fractal_features لسبب التعطيل
    # الافتراضي وشرح الإزاحة السببية (٢٤ سبتمبر ٢٠٢٦، مرشّح H003 المفتوح).
    "fractal_windows": [],
}


def _hour_features_meaningful(config: Optional[dict] = None) -> bool:
    """هل لميزات الساعة (``TIME_hour_*``) معنى في هذه الإعدادات؟

    الساعة داخل اليوم لا تتغيّر على فريم يومي فما فوق (كل شمعة تبدأ عند 00:00
    UTC)، فتصير ``TIME_hour_sin`` و``TIME_hour_cos`` ثابتتين بلا أي معلومة —
    وهذا ما كشفه ``audit_normalization`` ودفتر التدقيق (§٦ و§٩) على
    ``base_tf='1D'``. المعيار **أصغر** فريم في ``tf_order``: إن وُجد فريم
    داخل اليوم بين الفريمات تبقى الميزتان (تحملان معلومة عليه، وعلى الفريمات
    الأكبر تُصفَّران تلقائياً في ``process_windows`` لأنها ثابتة داخل نافذتها).
    """
    config = CONFIG if config is None else config
    tfs = [tf for tf in (config.get("tf_order") or [config.get("base_tf")]) if tf]
    try:
        smallest = min(pd.Timedelta(tf) for tf in tfs)
    except (ValueError, TypeError):
        return True        # فريم لم نفهمه: لا نحذف ميزة بصمت
    return smallest < pd.Timedelta(days=1)


def add_custom_features(data: pd.DataFrame,
                        settings: Optional[dict] = None,
                        config: Optional[dict] = None) -> pd.DataFrame:
    """يُضيف الميزات المخصّصة إلى ``DataFrame`` بأعمدة OHLCV.

    كل الميزات **سببية**: تعتمد على الشمعة الحالية والسابقة فقط، ولا تستخدم أي
    قيمة مستقبلية (``shift`` موجب فقط، و``rolling`` بلا ``center``).
    """
    config = CONFIG if config is None else config
    s = {**DEFAULT_CUSTOM_SETTINGS, **(settings or config.get("custom_settings") or {})}

    df = pd.DataFrame(data).copy()
    close = df['close'].astype('float64')
    high = df['high'].astype('float64')
    low = df['low'].astype('float64')
    open_ = df['open'].astype('float64') if 'open' in df else close.shift(1).bfill()

    # ── العوائد اللوغاريتمية (بالنسبة المئوية) ───────────────────────────
    log_close = np.log(close.clip(lower=EPS))
    ret_1 = log_close.diff()
    for h in s["returns"]:
        df[f'RET_{h}'] = (log_close - log_close.shift(h)) * 100.0

    # ── المدى النسبي ─────────────────────────────────────────────────────
    hl_range = (high - low)
    df['RANGE_rel'] = (hl_range / close.clip(lower=EPS)) * 100.0

    # ── بنية الشمعة ──────────────────────────────────────────────────────
    if s["candle_structure"]:
        denom = hl_range.replace(0, np.nan)
        df['BODY_ratio'] = ((close - open_) / denom).fillna(0.0)
        df['WICK_upper'] = ((high - np.maximum(close, open_)) / denom).fillna(0.0)
        df['WICK_lower'] = ((np.minimum(close, open_) - low) / denom).fillna(0.0)

    # ── الموضع داخل النطاق ───────────────────────────────────────────────
    for w in s["range_windows"]:
        lo = low.rolling(w, min_periods=w).min()
        hi = high.rolling(w, min_periods=w).max()
        df[f'POS_{w}'] = ((close - lo) / (hi - lo).replace(0, np.nan)).fillna(0.5)

    # ── نظام التقلب ──────────────────────────────────────────────────────
    for short, long in s["vol_windows"]:
        v_s = ret_1.rolling(short, min_periods=short).std()
        v_l = ret_1.rolling(long, min_periods=long).std()
        df[f'VOLR_{short}_{long}'] = np.log(
            (v_s / v_l.replace(0, np.nan)).clip(lower=EPS)).fillna(0.0)

    # ── الحجم غير الاعتيادي ──────────────────────────────────────────────
    if 'volume' in df:
        w = int(s["volume_window"])
        lv = np.log1p(df['volume'].clip(lower=0).astype('float64'))
        mu = lv.rolling(w, min_periods=w).mean()
        sd = lv.rolling(w, min_periods=w).std()
        df[f'VOLZ_{w}'] = ((lv - mu) / sd.replace(0, np.nan)).fillna(0.0)

    # ── الموسمية ─────────────────────────────────────────────────────────
    if s["time_features"] and isinstance(df.index, pd.DatetimeIndex):
        hour = df.index.hour.values + df.index.minute.values / 60.0
        dow = df.index.dayofweek.values
        if _hour_features_meaningful(config):
            df['TIME_hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
            df['TIME_hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
        df['TIME_dow_sin'] = np.sin(2 * np.pi * dow / 7.0)
        df['TIME_dow_cos'] = np.cos(2 * np.pi * dow / 7.0)

    # ── فركتالات ويليامز (مُعطَّلة افتراضياً، fractal_windows=[]) ──────────
    for w in s["fractal_windows"]:
        df[f'FRACTAL_high_{w}'], df[f'FRACTAL_low_{w}'] = _fractal_columns(high, low, w)

    return df


def _fractal_columns(high: pd.Series, low: pd.Series, window: int):
    """فركتال ويليامز — سببي بالكامل رغم استخدام ``center=True`` داخلياً.

    ``high.rolling(window, center=True).max()`` عند الموضع ``t`` يحتاج فعلياً
    بيانات حتى ``t + window//2`` — قيمة مستقبلية غير معروفة عند اتخاذ القرار
    في ``t`` (بالضبط فخّ DPO الموثَّق في خطة المشروع، قسم "تقطير الرؤية
    المتأخّرة"؛ راجع أيضاً حاشية 24 سبتمبر 2026 هناك). الإصلاح هنا **إزاحة**
    الناتج الخام بمقدار ``window//2`` إلى الأمام (``shift``، لا حذف/تصفير)،
    فتصبح القيمة عند ``t`` هي "هل كان ``t - window//2`` قمّة/قاعاً مؤكَّدة؟" —
    وهذا يحتاج فقط بيانات حتى ``t`` نفسها (``(t-window//2)+window//2 = t``)،
    أي **سببي تماماً** عند كل موضع بما في ذلك آخر شمعة في أي نافذة، لا حاجة
    لتصفير آخر خطوتين يدوياً: الإزاحة تُغني عن ذلك لأنها تُعيد توسيم كل نقطة
    بزمنها الحقيقي (زمن التأكيد) بدل تركها بزمن الرصد الخام غير المُتاح بعد.
    فقط بداية كامل السجل (قبل توفّر أول ``window`` شمعة) تبقى NaN→0 (إحماء
    عادي، لا تسرّب)."""
    half = window // 2
    raw_high = (high == high.rolling(window, center=True).max()).astype('float64')
    raw_low = (low == low.rolling(window, center=True).min()).astype('float64')
    return raw_high.shift(half).fillna(0.0), raw_low.shift(half).fillna(0.0)


def custom_feature_names(settings: Optional[dict] = None,
                         config: Optional[dict] = None) -> List[str]:
    """أسماء الميزات المخصّصة التي ستُنتَج بالإعدادات الحالية."""
    config = CONFIG if config is None else config
    s = {**DEFAULT_CUSTOM_SETTINGS, **(settings or config.get("custom_settings") or {})}
    names = [f'RET_{h}' for h in s["returns"]] + ['RANGE_rel']
    if s["candle_structure"]:
        names += ['BODY_ratio', 'WICK_upper', 'WICK_lower']
    names += [f'POS_{w}' for w in s["range_windows"]]
    names += [f'VOLR_{a}_{b}' for a, b in s["vol_windows"]]
    names += [f'VOLZ_{int(s["volume_window"])}']
    if s["time_features"]:
        if _hour_features_meaningful(config):
            names += ['TIME_hour_sin', 'TIME_hour_cos']
        names += ['TIME_dow_sin', 'TIME_dow_cos']
    for w in s["fractal_windows"]:
        names += [f'FRACTAL_high_{w}', f'FRACTAL_low_{w}']
    return names



def _test_fractal_features_causal():
    """اختبار ذاتي لـ_fractal_columns/add_custom_features: (أ) مُعطَّلة
    افتراضياً (fractal_windows=[]) فلا تُغيّر أي سلوك قديم، (ب) عند التفعيل
    كلّ قيمة سببية فعلاً (تقطيع السلسلة عند أي نقطة يُعطي نفس آخر قيمة التي
    تُنتجها السلسلة الكاملة عند تلك النقطة — أي لا تعتمد على أي بيانات بعدها)،
    (ج) قمّة/قاعاً صناعيين واضحين يُكتشَفان عند الموضع المتوقَّع بالضبط."""
    import numpy as np, pandas as pd
    n = 60
    idx = pd.date_range('2024-01-01', periods=n, freq='D')
    rng = np.random.default_rng(0)
    close = 100 + np.cumsum(rng.normal(0, 0.3, n))
    high = pd.Series(close + rng.random(n) * 0.5, index=idx)
    low = pd.Series(close - rng.random(n) * 0.5, index=idx)
    df = pd.DataFrame({'open': close, 'high': high.values, 'low': low.values,
                       'close': close, 'volume': rng.random(n) * 100}, index=idx)

    # (أ) مُعطَّلة افتراضياً
    out_default = add_custom_features(df, config={'custom_settings': {}})
    assert not any(c.startswith('FRACTAL_') for c in out_default.columns),         'fractal_windows=[] يجب ألا يُنتج أي عمود FRACTAL_ (توافق خلفي)'

    # (ب) السببية: تقطيع عند أي نقطة يُطابق القيمة المقابلة من السلسلة الكاملة
    fh_full, fl_full = _fractal_columns(high, low, window=5)
    for k in (20, 40, 59, 60):
        fh_k, fl_k = _fractal_columns(high.iloc[:k], low.iloc[:k], window=5)
        assert fh_k.iloc[-1] == fh_full.iloc[k - 1], f'تسرّب معلومة مستقبلية عند k={k} (high)'
        assert fl_k.iloc[-1] == fl_full.iloc[k - 1], f'تسرّب معلومة مستقبلية عند k={k} (low)'

    # (ج) قمّة/قاعاً صناعيان عند مواضع معروفة (window=5 → half=2)
    h2 = high.copy()
    h2.iloc[30] = h2.iloc[25:36].max() + 10.0  # قمّة واضحة عند 30
    l2 = low.copy()
    l2.iloc[30] = l2.iloc[25:36].min() - 10.0  # قاع واضح عند نفس الموضع (يُلغي بعضه)
    fh2, _ = _fractal_columns(h2, low, window=5)
    assert fh2.iloc[32] == 1.0, 'القمّة الصناعية عند 30 يجب أن تظهر مُؤكَّدة عند 30+half=32'
    assert fh2.iloc[31] == 0.0 and fh2.iloc[33] == 0.0, 'لا تأكيد قبل/بعد نقطة التأكيد مباشرة'

    print('✅ _test_fractal_features_causal: مُعطَّلة افتراضياً + سببية مُتحقَّقة + كشف صحيح لقمّة/قاع صناعيين')


_test_fractal_features_causal()

## 8) توليد الميزات الفنية (`features.py` سابقاً)

يستخدم `pandas_ta_classic` (أو `pandas_ta`) لحساب المؤشرات المُعرَّفة في
`CONFIG['indicator_settings']`، ثم يستدعي ميزات custom.py، ثم ينظّف القيم
المفقودة/غير المنتهية.


In [37]:
"""
توليد الميزات الفنية واشتقاق قائمة الأعمدة تلقائياً.

المبدأ الأساسي: قائمة الميزات **لا تُكتب يدوياً أبداً** — تُشتق بتشغيل
:func:`add_features` على بيانات تركيبية بنفس ``CONFIG`` الحالي، فتبقى مطابقة
لمخرجات ``pandas_ta`` الفعلية مهما عُدّلت ``indicator_settings``.
"""

# تسجيل ملحق df.ta — ``pandas_ta_classic`` هو الحزمة المتوافقة مع pandas الحديثة،
# مع تراجع تلقائي إلى ``pandas_ta`` الأصلية.
try:                                              # pragma: no cover - يعتمد على البيئة
    import pandas_ta_classic as ta                # noqa: F401
except ImportError:                               # pragma: no cover
    try:
        import pandas_ta as ta                    # noqa: F401
    except ImportError as exc:                    # pragma: no cover
        raise ImportError(
            "يلزم تثبيت pandas_ta_classic (أو pandas_ta):\n"
            "    pip install pandas_ta_classic"
        ) from exc

OHLCV = ("open", "high", "low", "close", "volume")


def add_features(data, config: Optional[dict] = None) -> pd.DataFrame:
    """إضافة كل المؤشرات المُعرَّفة في ``config['indicator_settings']``.

    المؤشرات الثابتة (``static_indicators``) تُحسب **مرة واحدة بعد** حلقة
    المؤشرات المُعامَلية، ولكل منها معاملاته الخاصة من ``static_indicator_params``.
    """
    config = CONFIG if config is None else config
    df = pd.DataFrame(data).copy()
    indicator_settings = config["indicator_settings"]
    static_indicators = config.get("static_indicators", [])
    static_params = config.get("static_indicator_params", {})

    for ind, values in indicator_settings.items():
        if ind == 'stoch':
            for fast_k, slow_k in values:
                df.ta.stoch(fast_k=fast_k, slow_k=slow_k, append=True)
        elif ind == 'stochrsi':
            for rsi_len, stoch_len in values:
                df.ta.stochrsi(rsi_length=rsi_len, stoch_length=stoch_len, append=True)
        elif ind == 'macd':
            for f, s, sig in values:
                df.ta.macd(fast=f, slow=s, signal=sig, append=True)
        elif ind == 'ppo':
            for f, s, sig in values:
                df.ta.ppo(fast=f, slow=s, signal=sig, append=True)
        elif ind == 'donchian':
            for length in values:
                df.ta.donchian(lower_length=length, upper_length=length, append=True)
        else:
            for v in values:
                if hasattr(df.ta, ind):
                    getattr(df.ta, ind)(length=v, append=True)
                else:
                    print(f"⚠️ المؤشر غير موجود: {ind}")

    # ✅ الميزات المخصّصة الساكنة تُضاف قبل الحذف النهائي لصفوف NaN
    if config.get("use_custom_features", True):
        df = add_custom_features(df, config=config)

    for ind in static_indicators:
        if ind == 'cross_value':
            df.ta.cross_value(value=50, append=True)
            continue
        if not hasattr(df.ta, ind):
            print(f"⚠️ المؤشر غير موجود: {ind}")
            continue
        getattr(df.ta, ind)(append=True, **static_params.get(ind, {}))

    df.dropna(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df


def infer_feature_columns(config: Optional[dict] = None,
                          min_rows: int = 600) -> List[str]:
    """اشتقاق قائمة الميزات الحقيقية بتشغيل :func:`add_features` على بيانات تركيبية.

    يمنع أي تعارض بين الأسماء المتوقَّعة والأسماء التي تُنتجها ``pandas_ta`` فعلياً.
    """
    config = CONFIG if config is None else config
    rng = np.random.default_rng(0)
    close = 100 + np.cumsum(rng.normal(0, 0.5, min_rows))
    high = close + rng.random(min_rows) * 1.5
    low = close - rng.random(min_rows) * 1.5
    open_ = close + rng.normal(0, 0.3, min_rows)
    volume = rng.random(min_rows) * 1000 + 100
    idx = pd.date_range("2024-01-01", periods=min_rows, freq="h")
    sample = pd.DataFrame(
        {"open": open_, "high": high, "low": low, "close": close, "volume": volume},
        index=idx,
    )
    processed = add_features(sample, config=config)
    exclude = set(config.get("exclude_from_features", []))
    cols = [c for c in processed.columns if c not in exclude]
    # ✅ أعمدة السياق السوقي (MKT_) لا تُنتجها add_features (مصدرها عملة
    # أخرى، لا هذا الإطار نفسه) — تُدرَج هنا فقط لتظهر في feature_order/
    # X_tf تلقائياً؛ قيمها الفعلية تُملأ لاحقاً بـ add_market_context.
    cols += [c for c in market_context_columns(config) if c not in exclude]
    # ✅ معدّل التمويل/الفائدة المفتوحة (add_funding_oi_features) — مصدرها
    # أرشيف Drive الخاص بكل رمز، لا هذا الإطار نفسه، كـMKT_ أعلاه بالضبط.
    cols += [c for c in funding_oi_feature_columns(config) if c not in exclude]
    return cols


def available_features(config: Optional[dict] = None, min_rows: int = 600) -> List[str]:
    """كل أعمدة ``add_features`` الممكنة **قبل** ``exclude_from_features``.

    مرجع الأسماء لـ :func:`exclude_features` (وما يمكنك كتابته في أنماطها).
    """
    config = CONFIG if config is None else config
    return infer_feature_columns({**config, "exclude_from_features": []},
                                 min_rows=min_rows)


def exclude_features(names: Any,
                     config: Optional[dict] = None,
                     refresh: bool = True,
                     verbose: bool = True) -> List[str]:
    """يستبعد ميزات من **مدخلات النموذج** بالاسم أو بنمط، ويُحدّث ``feature_order``.

    >>> exclude_features(['RSI_14', 'BB*', 'TIME_*'])

    * ``names``: أسماء أعمدة أو أنماط ``*``/``?`` (بلا حساسية لحالة الأحرف).
      أسماؤك تُطابَق على :func:`available_features`، فما لا يوجد فيها (خطأ إملائي
      غالباً) يُحذَّر منه ولا يُضاف بصمت.
    * تُضاف الأسماء إلى ``CONFIG['exclude_from_features']`` — المصدر الوحيد للحقيقة —
      فتبقى الأعمدة محسوبة داخلياً (للأهداف والمقياس) وتغيب عن المدخلات فقط.
      لإلغاء الاستبعاد: ``update_config(exclude_from_features=[...])`` بالقائمة المطلوبة.
    * استدعِها **قبل** ``build_dataset``: بيانات بُنيت سابقاً تحمل الأبعاد القديمة.
    * ترفع ``ValueError`` (دون تعديل شيء) إن كان الاستبعاد سيُفرغ كل الميزات.

    Returns:
        الأسماء التي أُضيفت فعلاً في هذا الاستدعاء (فارغة إن لم يتغيّر شيء).

    لإيقاف **عائلة مؤشر كاملة** من الحساب أصلاً (لا مجرد إخفائها) عدّل
    ``indicator_settings`` أو ``static_indicators`` — الاستبعاد هنا أسرع تجريباً
    وأأمن لأنه لا يمسّ الميزات التي تعتمد عليها الأهداف.
    """
    import fnmatch

    config = CONFIG if config is None else config
    if isinstance(names, str):
        names = [names]
    patterns = [str(n).strip() for n in names if str(n).strip()]

    universe = available_features(config)
    current = list(config.get("exclude_from_features") or [])
    taken = set(current)
    added: List[str] = []
    already: List[str] = []
    unknown: List[str] = []

    for pat in patterns:
        hits = [c for c in universe if fnmatch.fnmatchcase(c.lower(), pat.lower())]
        if not hits:
            unknown.append(pat)
            continue
        for c in hits:
            if c in taken:
                if c not in already:
                    already.append(c)
            else:
                taken.add(c)
                added.append(c)

    if not [c for c in universe if c not in taken]:
        raise ValueError("❌ هذا الاستبعاد سيُزيل كل الميزات — لم يُغيَّر شيء.")

    if added:
        config["exclude_from_features"] = current + added
        if refresh:
            refresh_features(config)
    if verbose:
        if added:
            print(f"🚫 استُبعدت {len(added)} ميزة: {added[:12]}{'...' if len(added) > 12 else ''}")
        if already:
            print(f"   ℹ️ مُستبعَدة مسبقاً: {already[:8]}{'...' if len(already) > 8 else ''}")
        if unknown:
            print(f"   ⚠️ لا ميزة بهذه الأسماء/الأنماط (تحقّق من الإملاء أو استعرض "
                  f"available_features()): {unknown}")
    return added


def extract_features(df: pd.DataFrame, features: List[str]) -> pd.DataFrame:
    """انتقاء أعمدة الميزات المتاحة فقط، مع تحذير عن المفقود بدل ``KeyError``."""
    available = [c for c in features if c in df.columns]
    missing = [c for c in features if c not in df.columns]
    if missing:
        print(f"⚠️ أعمدة مفقودة ({len(missing)}) — تُتجاهَل: {missing[:8]}"
              f"{'...' if len(missing) > 8 else ''}")
    return df[available]


def price_indices(features: Optional[List[str]] = None,
                  config: Optional[dict] = None) -> dict:
    """فهارس أعمدة السعر داخل مصفوفة الميزات.

    تُستخدم بدل تثبيت أرقام الأعمدة (0/1/2)، فتبقى صحيحة مهما تغيّرت
    ``exclude_from_features`` أو ترتيب المؤشرات.
    """
    config = CONFIG if config is None else config
    if features is None:
        features = feature_order(config)
    idx = {}
    for col in ("high", "low", "close", "open", "volume"):
        if col in features:
            idx[col] = features.index(col)
    for required in ("high", "low", "close"):
        if required not in idx:
            raise ValueError(
                f"العمود '{required}' غير موجود في feature_order — "
                "لا يمكن حساب الأهداف. راجع exclude_from_features.")
    return idx


## 9) التطبيع (`normalize.py` سابقاً)

تطبيع كل ميزة حسب تصنيفها الدلالي (مستوى سعري، مذبذب محصور، تراكمي...) بدل
تطبيع موحّد يخلط بين ما يجب تمركزه وما لا يجوز.


In [38]:
# @title
"""
التطبيع الموحّد للنوافذ — مبني على **تصنيف دلالي** لكل ميزة.

المبدأ: لكل ميزة *وحدة قياس* و*مرجع صفر*، والتطبيع الصحيح يعتمد عليهما:

============= ================================================ ==========================
النوع          الدلالة                                          التحويل
============= ================================================ ==========================
price_level   مستوى سعري (close, EMA, نطاقات بولنجر)            ``(x - c) / s``
price_scale   كمية بوحدة السعر لكن ليست مستوى (ATR, MACD, STDEV) ``x / s``  ← بلا تمركز
percent       نسبة مئوية (ROC, PPO, NATR)                       ``x / 10``
osc_0_100     مذبذب محصور [0, 100] (RSI, STOCH, ADX)            ``(x - 50) / 50``
osc_sym_100   مذبذب متماثل [-100, 100] (CMO, AROONOSC)          ``x / 100``
osc_willr     مذبذب [-100, 0] (WILLR)                           ``(x + 50) / 50``
unit_sym      محصور [-1, 1] (BOP, CMF)                          ``x``
unit_0_1      محصور [0, 1] (ER, BBP)                            ``(x - 0.5) * 2``
zscore        درجة معيارية أصلاً (ZS)                            ``x / 3``
sign_robust   مذبذب غير محصور حول الصفر (TSI, TRIX, CCI, FISHER) ``x / (1.4826·med|x|)``
cumulative    تراكمي بمرجع اعتباطي (OBV, AD, KVO, EFI)          تقييس داخل النافذة
log_volume    الحجم                                             ``log1p`` ثم تقييس
============= ================================================ ==========================

**لماذا هذا التمييز ضروري** — ثلاثة أخطاء كانت قائمة وقيست فعلياً:

1. ``ATRr_14`` كان مُصنَّفاً ``price_level``، فيُطرح منه *سعر الإغلاق*. على
   بيتكوين بسعر 61,000 و ATR=350 كانت النتيجة **-60.7** بينما ``close`` نفسه
   ينتج ``±1`` — أي أن عموداً واحداً أكبر بـ 61× ويبتلع الطبقة الأولى.
2. ``volume`` كان ``log1p`` فقط بلا تمركز، فمتوسطه **13.0** لا 0.
3. أعمدة مداها ``[-1, 1]`` (``CMF``, ``BOP``, ``BBP``, ``UI``, ``ZS``) كانت
   تُقسَم على 100، فصار انحرافها المعياري ~0.002 — **معلومة معدومة عملياً**.

كذلك المذبذبات حول الصفر (``MACD``, ``TSI``, ``TRIX``) كانت تُقيَّس داخل النافذة
بطرح وسيطها، فتضيع **إشارتها** — و«MACD فوق الصفر» معلومة حقيقية لا يجوز محوها.
"""

# ══════════════════════════════════════════════════════════════════════════
# أنواع الميزات
# ══════════════════════════════════════════════════════════════════════════
PRICE_LEVEL = 'price_level'
PRICE_SCALE = 'price_scale'
PERCENT = 'percent'
OSC_0_100 = 'osc_0_100'
OSC_SYM_100 = 'osc_sym_100'
OSC_WILLR = 'osc_willr'
UNIT_SYM = 'unit_sym'
UNIT_0_1 = 'unit_0_1'
ZSCORE = 'zscore'
SIGN_ROBUST = 'sign_robust'
CUMULATIVE = 'cumulative'
LOG_VOLUME = 'log_volume'

#: النوع الافتراضي لأي عمود غير مُصنَّف — تقييس داخل النافذة (آمن دائماً).
DEFAULT_KIND = CUMULATIVE

#: أقصى قيمة مطلقة بعد التطبيع. القصّ يمنع انفجار التدرّجات من عيّنة شاذة
#: واحدة، ويُبقي 99%+ من القيم بلا مساس (انظر ``audit_normalization``).
CLIP_ABS = 5.0

#: تصنيف الميزات ببادئة الاسم. الترتيب مهم: تُفحص البادئات الأطول أولاً
#: تلقائياً في :func:`classify_feature`، فلا يلتقط ``BB`` ما هو لـ ``BBP``.
FEATURE_KINDS: Dict[str, str] = {
    # ── مستويات سعرية ────────────────────────────────────────────────────
    **{p: PRICE_LEVEL for p in (
        'open', 'high', 'low', 'close', 'hl2', 'HLC3', 'OHLC4', 'WCP',
        'SMA', 'EMA', 'WMA', 'DEMA', 'TEMA', 'TRIMA', 'HMA', 'RMA', 'VWMA',
        'VWAP', 'ALMA', 'T3_', 'ZL_EMA', 'ZLMA', 'KAMA', 'FWMA', 'SINWMA',
        'SSF', 'SWMA', 'PWMA', 'LINREG', 'MIDPOINT', 'MIDPRICE',
        'BBL', 'BBM', 'BBU', 'DCL', 'DCM', 'DCU', 'KCL', 'KCB', 'KCU',
        'SUPERT', 'PSAR', 'ICHIMOKU', 'ISA_', 'ISB_', 'ITS_', 'IKS_',
    )},
    # ── كميات بوحدة السعر لكنها ليست مستوى (لا تُمركَز أبداً) ─────────────
    **{p: PRICE_SCALE for p in (
        'ATR', 'ATRr', 'TRUERANGE', 'TR_', 'STDEV', 'VAR_', 'MAD_',
        'MACD', 'MOM', 'SLOPE', 'MIDP', 'DPO', 'ABER',
    )},
    # ── نِسَب مئوية ───────────────────────────────────────────────────────
    **{p: PERCENT for p in (
        'ROC', 'PPO', 'NATR', 'BBB', 'CHOP', 'MASSI', 'PVR',
    )},
    # ── مذبذبات [0, 100] ─────────────────────────────────────────────────
    **{p: OSC_0_100 for p in (
        'RSI', 'STOCHk', 'STOCHd', 'STOCHh', 'STOCHRSIk', 'STOCHRSId',
        'ADX', 'ADXR', 'DMP', 'DMN', 'AROOND', 'AROONU', 'MFI', 'RVI',
        'UO_', 'CRSI', 'PSL', 'RVGI', 'KST',
    )},
    # ── مذبذبات [-100, 100] ──────────────────────────────────────────────
    **{p: OSC_SYM_100 for p in ('CMO', 'AROONOSC', 'CFO', 'CTI')},
    # ── مذبذب [-100, 0] ──────────────────────────────────────────────────
    **{p: OSC_WILLR for p in ('WILLR',)},
    # ── محصورة [-1, 1] ───────────────────────────────────────────────────
    **{p: UNIT_SYM for p in ('BOP', 'CMF', 'QS_', 'INC_', 'DEC_')},
    # ── محصورة [0, 1] ────────────────────────────────────────────────────
    **{p: UNIT_0_1 for p in ('ER_', 'BBP', 'SQZ_ON', 'SQZ_OFF', 'SQZ_NO',
                             'UI_')},
    # ── درجات معيارية ────────────────────────────────────────────────────
    **{p: ZSCORE for p in ('ZS_', 'SKEW', 'KURT')},
    # ── مذبذبات غير محصورة حول الصفر (الإشارة معلومة) ────────────────────
    **{p: SIGN_ROBUST for p in (
        'CCI', 'TSI', 'TRIX', 'FISHERT', 'LOGRET', 'PCTRET', 'PVO', 'PVI',
        'NVI', 'BIAS', 'COPC', 'EOM', 'SMI', 'STC', 'QQE', 'TD_SEQ',
    )},
    # ── تراكمية بمرجع اعتباطي ────────────────────────────────────────────
    # ENTP مداها ~log(length) (≈3.3 لطول 10) لا [0,1] — تقييس داخل النافذة
    **{p: CUMULATIVE for p in ('OBV', 'AD', 'ADOSC', 'KVO', 'EFI', 'PVT',
                               'CMF_SUM', 'VP_', 'ENTP')},
    # ── الحجم ────────────────────────────────────────────────────────────
    **{p: LOG_VOLUME for p in ('volume', 'VOL_')},

    # ══ الميزات المخصّصة (custom.py) ══════════════════════════════════════
    # كلها ساكنة وخالية من المقياس بالبناء، فتصنيفها مباشر لا اجتهاد فيه.
    **{p: SIGN_ROBUST for p in ('RET_', 'VOLR_')},   # عوائد ونِسَب لوغاريتمية
    # ── سياق سوقي عابر للأصول (add_market_context) ─────────────────────────
    **{p: SIGN_ROBUST for p in ('MKT_ret_',)},       # عوائد العملة المرجعية
    **{p: PERCENT for p in ('RANGE_',)},             # مدى نسبي بالنسبة المئوية
    **{p: UNIT_SYM for p in ('BODY_', 'TIME_')},     # [-1, 1] أصلاً
    **{p: UNIT_0_1 for p in ('WICK_', 'POS_')},      # [0, 1] أصلاً
    **{p: ZSCORE for p in ('VOLZ_',)},               # درجة معيارية أصلاً
}

#: البادئات مرتّبة تنازلياً بالطول — تضمن أن ``BBP`` تسبق ``BB``، و``ATRr`` تسبق ``ATR``.
_SORTED_PREFIXES: List[str] = sorted(FEATURE_KINDS, key=len, reverse=True)


def classify_feature(col_name: str,
                     kinds: Optional[Dict[str, str]] = None,
                     default: str = DEFAULT_KIND) -> str:
    """نوع الميزة الدلالي حسب بادئة اسمها (أطول بادئة مطابِقة تفوز)."""
    table = FEATURE_KINDS if kinds is None else kinds
    prefixes = _SORTED_PREFIXES if kinds is None else sorted(table, key=len, reverse=True)
    for p in prefixes:
        if col_name.startswith(p):
            return table[p]
    return default


#: اسم متوافق مع الكود القديم.
classify_column = classify_feature


#: حدّ أدنى مطلق للمقياس — يمنع القسمة على صفر حرفي فقط.
SCALE_ABS_FLOOR = 1e-8
#: حدّ أدنى **نسبي** لمستوى السعر (0.1%). أصل بسعر مرتفع في نافذة شبه مسطّحة
#: يكون IQR لها صغيراً بالمطلق دون أن يلمس ``SCALE_ABS_FLOOR`` فعلياً، فينفجر
#: أي هدف/ميزة يُطبَّع بها إلى مئات الأضعاف عند أي حركة سعرية لاحقة — قيس فعلياً:
#: ``y_high_reg`` بمدى ``[-41, +728]`` بدل ``[-3, +3]`` المتوقَّع لمقياس سليم.
SCALE_REL_FLOOR = 1e-3


# ══════════════════════════════════════════════════════════════════════════
# أدوات القياس
# ══════════════════════════════════════════════════════════════════════════
def calc_scale_params(data: np.ndarray, method: str = "robust") -> Tuple[float, float]:
    """``(المركز، المقياس)`` — وسيط/IQR لـ ``robust``، أو min/range لـ ``minmax``.

    المقياس محدود بحدّين معاً: مطلق (``SCALE_ABS_FLOOR``) لمنع القسمة على
    صفر حرفي، ونسبي لمستوى السعر (``SCALE_REL_FLOOR``) لمنع انفجار الهدف/الميزة
    حين يكون IQR صغيراً بالمطلق نسبةً لسعر مرتفع — انظر التعليق أعلاه.
    """
    if method == 'robust':
        q75, q50, q25 = np.percentile(data, [75, 50, 25])
        rel_floor = abs(q50) * SCALE_REL_FLOOR
        return q50, max(q75 - q25, rel_floor, SCALE_ABS_FLOOR)
    lo, hi = data.min(), data.max()
    rel_floor = abs(lo) * SCALE_REL_FLOOR
    return lo, max(hi - lo, rel_floor, SCALE_ABS_FLOOR)


def scale_data(data: np.ndarray, center: float, scale: float) -> np.ndarray:
    return (data - center) / scale


def inverse_scale(preds: np.ndarray, bases: np.ndarray) -> np.ndarray:
    """عكس التطبيع السعري: ``preds * iqr + median`` لكل عيّنة."""
    if preds.ndim == 2:
        return preds * bases[:, 1:2] + bases[:, 0:1]
    return preds * bases[:, 1] + bases[:, 0]


def _robust_magnitude(x: np.ndarray, eps: float) -> float:
    """مقياس حجم حول الصفر يحفظ الإشارة (لا يُمركِز)."""
    mag = 1.4826 * np.median(np.abs(x))
    if mag < eps:                      # نافذة شبه ثابتة عند الصفر
        mag = max(np.std(x), eps)
    return float(mag)


def _standardize(x: np.ndarray, method: str, eps: float) -> np.ndarray:
    """تقييس داخل النافذة — للأعمدة التي مرجعها اعتباطي."""
    if method == "minmax":
        lo, hi = np.nanmin(x), np.nanmax(x)
        return (x - lo) / (hi - lo + eps)
    c, s = calc_scale_params(x, method)
    return (x - c) / s


# ══════════════════════════════════════════════════════════════════════════
# التطبيع
# ══════════════════════════════════════════════════════════════════════════
def normalize_column(x: np.ndarray, kind: str, center: float, scale: float,
                     method: str = "robust", eps: float = 1e-8) -> np.ndarray:
    """يُطبّع عموداً واحداً حسب نوعه الدلالي.

    Args:
        center / scale: مركز ومقياس نافذة **الإغلاق** — يُستخدمان للأعمدة
            السعرية فقط، فتبقى كلها على مرجع مشترك واحد.
    """
    if kind == PRICE_LEVEL:
        return (x - center) / scale
    if kind == PRICE_SCALE:
        # ✅ قسمة بلا تمركز: ATR=350 على سعر 61,000 يعطي 0.35 لا -60.7
        return x / scale
    if kind == PERCENT:
        return x / 10.0
    if kind == OSC_0_100:
        return (x - 50.0) / 50.0
    if kind == OSC_SYM_100:
        return x / 100.0
    if kind == OSC_WILLR:
        return (x + 50.0) / 50.0
    if kind == UNIT_SYM:
        return x
    if kind == UNIT_0_1:
        return (x - 0.5) * 2.0
    if kind == ZSCORE:
        return x / 3.0
    if kind == SIGN_ROBUST:
        # ✅ يحفظ الإشارة: 'MACD فوق الصفر' معلومة حقيقية لا تُمحى
        return x / _robust_magnitude(x, eps)
    if kind == LOG_VOLUME:
        return _standardize(np.log1p(np.clip(x, 0, None)), method, eps)
    # CUMULATIVE والافتراضي
    return _standardize(x, method, eps)


def process_window(w: np.ndarray, columns: list, med_p: float, iqr_p: float,
                   method: str = "robust", config: Optional[dict] = None,
                   clip_abs: Optional[float] = None) -> np.ndarray:
    """تطبيع نافذة ``(T, F)`` عموداً عموداً حسب النوع الدلالي لكل عمود."""
    config = CONFIG if config is None else config
    if not isinstance(columns, (list, tuple)):
        columns = list(columns)
    eps = config.get("eps", 1e-8)
    clip_abs = config.get("clip_abs", CLIP_ABS) if clip_abs is None else clip_abs

    out = np.asarray(w, dtype='float64').copy()
    for j in range(out.shape[1]):
        x = out[:, j]
        finite = np.isfinite(x)
        if not finite.all():
            x = np.where(finite, x, np.nanmedian(x[finite]) if finite.any() else 0.0)

        lo, hi = np.nanmin(x), np.nanmax(x)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
            out[:, j] = 0.0
            continue

        out[:, j] = normalize_column(
            x, classify_feature(columns[j]), med_p, iqr_p, method, eps)

    out = np.nan_to_num(out, nan=0.0, posinf=clip_abs, neginf=-clip_abs)
    if clip_abs:
        np.clip(out, -clip_abs, clip_abs, out=out)
    return out.astype('float32')


def process_windows(windows: np.ndarray, columns: List[str],
                    centers: np.ndarray, scales: np.ndarray,
                    method: str = "robust", config: Optional[dict] = None,
                    clip_abs: Optional[float] = None) -> np.ndarray:
    """تطبيع **كل النوافذ دفعةً واحدة** — النسخة المتّجهة من :func:`process_window`.

    :func:`process_window` تعالج نافذة واحدة بحلقة بايثون على الأعمدة، فتصير
    التكلفة ``N × F`` دورة بايثون (≈9,600 دورة لأصل واحد بـ 246 نافذة و39 ميزة).
    هنا نُجمّع الأعمدة حسب نوعها الدلالي **مرة واحدة**، ثم نُطبّق تحويل كل نوع
    على كل النوافذ معاً بعمليات numpy متّجهة. النتيجة مطابقة عددياً.

    ✅ **float32 من البداية للنهاية** (كانت ترفع إلى float64 ثم تُخصّص ``out``
    منفصلة، فتحمل ذروتها ~7.77× حجم المصفوفة النهائية — قِيس فعلياً؛ الحساب
    هنا يكتب في ``X`` نفسها بلا مصفوفة ``out`` موازية، فتنزل الذروة إلى ~2.64×
    (رقم حقيقي مُقاس، لا نظري). الفرق العددي عن النسخة القديمة ≤5e-7 — أصغر
    من دقّة float32 نفسها، لا يُغيّر شيئاً عملياً.

    Args:
        windows: ``(N, T, F)`` نوافذ خام.
        centers / scales: ``(N,)`` مركز ومقياس نافذة الإغلاق لكل عيّنة.
    """
    config = CONFIG if config is None else config
    eps = config.get("eps", 1e-8)
    clip_abs = config.get("clip_abs", CLIP_ABS) if clip_abs is None else clip_abs

    X = np.asarray(windows, dtype='float32')
    n, _, n_cols = X.shape
    c = np.asarray(centers, dtype='float32').reshape(n, 1, 1)
    s = np.asarray(scales, dtype='float32').reshape(n, 1, 1)

    # استبدال غير المنتهي بوسيط العمود داخل نافذته (مطابق للنسخة المفردة)
    finite = np.isfinite(X)
    if not finite.all():
        med = np.nanmedian(np.where(finite, X, np.nan), axis=1, keepdims=True)
        X = np.where(finite, X, np.nan_to_num(med, nan=0.0)).astype('float32', copy=False)

    # الأعمدة الثابتة داخل نافذتها لا تحمل معلومة → تُصفَّر لاحقاً
    lo = X.min(axis=1)
    hi = X.max(axis=1)
    degenerate = (hi == lo) | ~np.isfinite(lo) | ~np.isfinite(hi)   # (N, F)

    # تجميع الأعمدة حسب النوع مرة واحدة بدل تصنيفها لكل نافذة
    by_kind: Dict[str, List[int]] = {}
    for j, name in enumerate(columns[:n_cols]):
        by_kind.setdefault(classify_feature(name), []).append(j)

    # ✅ نكتب في X نفسها بدل تخصيص out منفصلة: مجموعات by_kind أعمدتها
    # متفرّقة لا تتداخل أبداً (كل عمود ينتمي لنوع واحد فقط)، فالكتابة في
    # X[:, :, idx] لا تُفسد قراءة مجموعة أخرى لاحقة — هذا ما يوفّر مصفوفة
    # float32 كاملة إضافية بالمقارنة مع النسخة القديمة (out = np.empty_like).
    for kind, idx in by_kind.items():
        sub = X[:, :, idx]

        if kind == PRICE_LEVEL:
            res = (sub - c) / s
        elif kind == PRICE_SCALE:
            res = sub / s
        elif kind == PERCENT:
            res = sub / 10.0
        elif kind == OSC_0_100:
            res = (sub - 50.0) / 50.0
        elif kind == OSC_SYM_100:
            res = sub / 100.0
        elif kind == OSC_WILLR:
            res = (sub + 50.0) / 50.0
        elif kind == UNIT_SYM:
            res = sub
        elif kind == UNIT_0_1:
            res = (sub - 0.5) * 2.0
        elif kind == ZSCORE:
            res = sub / 3.0
        elif kind == SIGN_ROBUST:
            mag = 1.4826 * np.median(np.abs(sub), axis=1, keepdims=True)
            fallback = np.std(sub, axis=1, keepdims=True)
            mag = np.where(mag < eps, np.maximum(fallback, eps), mag)
            res = sub / mag
        elif kind == LOG_VOLUME:
            res = _standardize_batch(np.log1p(np.clip(sub, 0, None)), method, eps)
        else:                                   # CUMULATIVE والافتراضي
            res = _standardize_batch(sub, method, eps)

        X[:, :, idx] = res

    X[np.broadcast_to(degenerate[:, None, :], X.shape)] = 0.0
    X = np.nan_to_num(X, nan=0.0, posinf=clip_abs, neginf=-clip_abs, copy=False)
    if clip_abs:
        np.clip(X, -clip_abs, clip_abs, out=X)
    return X


def _standardize_batch(sub: np.ndarray, method: str, eps: float) -> np.ndarray:
    """تقييس متّجه داخل كل نافذة على حدة (المحور الزمني)."""
    if method == "minmax":
        lo = sub.min(axis=1, keepdims=True)
        hi = sub.max(axis=1, keepdims=True)
        return (sub - lo) / (hi - lo + eps)
    q25, q50, q75 = np.percentile(sub, [25, 50, 75], axis=1, keepdims=True)
    return (sub - q50) / np.maximum(q75 - q25, 1e-8)


# ══════════════════════════════════════════════════════════════════════════
# التدقيق
# ══════════════════════════════════════════════════════════════════════════
def describe_features(features: List[str]) -> Dict[str, List[str]]:
    """يُجمّع الميزات حسب نوعها الدلالي — لمراجعة التصنيف قبل التدريب."""
    groups: Dict[str, List[str]] = {}
    for f in features:
        groups.setdefault(classify_feature(f), []).append(f)
    return dict(sorted(groups.items()))


def audit_normalization(windows: np.ndarray, features: List[str],
                        verbose: bool = True) -> pd.DataFrame:
    """يقيس التوزيع الفعلي لكل عمود بعد التطبيع ويكشف الأعمدة المُعطَّلة.

    Args:
        windows: مصفوفة ``(N, T, F)`` **بعد** :func:`process_window`.

    يُرجع جدولاً بأعمدة: النوع، المتوسط، الانحراف، أقصى قيمة مطلقة، وحكم.
    """
    rows = []
    for j, name in enumerate(features):
        v = np.asarray(windows[:, :, j]).ravel()
        v = v[np.isfinite(v)]
        if not v.size:
            continue
        std = float(v.std())
        absmax = float(np.abs(v).max())
        if std < 0.01:
            verdict = "🚨 بلا معلومة (شبه ثابت)"
        elif absmax > CLIP_ABS * 1.5:
            verdict = "🚨 متطرف"
        elif std > 3.0:
            verdict = "⚠️ تشتت عالٍ"
        else:
            verdict = "✅"
        rows.append({
            'feature': name, 'kind': classify_feature(name),
            'mean': float(v.mean()), 'std': std,
            'p01': float(np.percentile(v, 1)), 'p99': float(np.percentile(v, 99)),
            'absmax': absmax, 'verdict': verdict,
        })

    table = pd.DataFrame(rows)
    if verbose and len(table):
        bad = table[~table.verdict.str.startswith("✅")]
        print(f"🔬 تدقيق التطبيع: {len(table)} ميزة | "
              f"سليمة: {len(table) - len(bad)} | تحتاج مراجعة: {len(bad)}")
        if len(bad):
            print(bad[['feature', 'kind', 'mean', 'std', 'absmax', 'verdict']]
                  .to_string(index=False))
    return table


#: خريطة الأنواع القديمة (رقمية) — للرجوع فقط، لم تعد مستخدَمة.
LEGACY_NORM_TYPES = {0: PRICE_LEVEL, 1: OSC_0_100, 2: OSC_SYM_100,
                     3: CUMULATIVE, 4: LOG_VOLUME}


## 10) محاذاة الفريمات الزمنية المتعدّدة (`align.py` سابقاً)


In [39]:
# @title
"""
محاذاة الفريمات الزمنية المتعدّدة على أساس الوقت (وليس الفهرس).

لكل نافذة على الفريم الأصغر، تُقتطع نافذة مقابلة من كل فريم أعلى تنتهي عند
شمعة **مغلقة بالفعل** قبل زمن النهاية (``higher_tf_offset``)، منعاً لتسرّب
معلومات من المستقبل.
"""

def align_multi_timeframes_time_based(
    dfs: Dict[str, pd.DataFrame],
    tf_order: Optional[List[str]] = None,
    window_sizes: Optional[Dict[str, int]] = None,
    stride: Optional[int] = None,
    config: Optional[dict] = None,
) -> Tuple[Dict[str, np.ndarray], List[pd.Timestamp]]:
    """يُرجع ``({tf: مصفوفة نوافذ (N, T, F)}, قائمة أزمنة النهاية)``.

    ✅ **مسار سريع لفريم واحد** (``len(tf_order) == 1`` — حالتك الفعلية):
    عبر ``numpy.lib.stride_tricks.sliding_window_view`` بدل نسخ كل نافذة على
    حدة في حلقة بايثون ثم تكديسها. قِيس فعلياً: ذروة الرام تنزل من ~3.16×
    حجم المصفوفة النهائية إلى ~1.13× (شبه الحدّ الأدنى النظري) على أصل بـ900
    يوم — النسختان تُنتجان **نفس المصفوفة تماماً** (تحقّقتُ رقمياً، لا فرق
    عددي إطلاقاً، فالعملية إعادة ترتيب ذاكرة لا حساب). فريمات متعدّدة
    (``len(tf_order) > 1``) تبقى على الحلقة الأصلية: محاذاة زمنية شرطية بين
    فريمات مختلفة الطول أعقد من أن تُختزل لعرض نوافذ (view) واحد بأمان.
    """
    config = CONFIG if config is None else config
    tf_order = config["tf_order"] if tf_order is None else tf_order
    window_sizes = config["window_sizes"] if window_sizes is None else window_sizes
    stride = config["stride"] if stride is None else stride

    if len(tf_order) == 1:
        return _align_single_tf_view(dfs, tf_order[0], window_sizes[tf_order[0]], stride)

    offset = config.get("higher_tf_offset", 2)
    small_tf = tf_order[0]
    df_small = dfs[small_tf]
    win_small = window_sizes[small_tf]

    if not isinstance(df_small.index, pd.DatetimeIndex):
        raise ValueError(f"DataFrame للفريم {small_tf} يجب أن يكون مفهرساً بـ DatetimeIndex")
    df_small = df_small.sort_index()
    for tf in tf_order[1:]:
        if not isinstance(dfs[tf].index, pd.DatetimeIndex):
            raise ValueError(f"DataFrame للفريم {tf} يجب أن يكون مفهرساً بـ DatetimeIndex")
        dfs[tf] = dfs[tf].sort_index()

    windows: Dict[str, List[np.ndarray]] = {tf: [] for tf in tf_order}
    end_times: List[pd.Timestamp] = []
    times_small = df_small.index

    for end_idx_small in range(win_small - 1, len(df_small), stride):
        start_idx_small = end_idx_small - win_small + 1
        end_time = times_small[end_idx_small]
        win_small_df = df_small.iloc[start_idx_small:end_idx_small + 1]
        current_wins: Dict[str, np.ndarray] = {small_tf: win_small_df.values}
        valid = True

        for tf in tf_order[1:]:
            df_tf = dfs[tf]
            win_tf = window_sizes[tf]
            pos = df_tf.index.searchsorted(end_time, side='right') - offset
            if pos < 0:
                valid = False
                break
            end_idx_tf = pos
            start_idx_tf = end_idx_tf - win_tf + 1
            if start_idx_tf < 0:
                valid = False
                break
            win_tf_df = df_tf.iloc[start_idx_tf:end_idx_tf + 1]
            if len(win_tf_df) != win_tf:
                valid = False
                break
            current_wins[tf] = win_tf_df.values

        if not valid:
            continue
        for tf in tf_order:
            windows[tf].append(current_wins[tf])
        end_times.append(end_time)

    out: Dict[str, np.ndarray] = {}
    for tf in tf_order:
        if windows[tf]:
            arr_tf = np.stack(windows[tf], axis=0)
        else:
            n_feat = dfs[tf].shape[1]
            arr_tf = np.zeros((0, window_sizes[tf], n_feat), dtype='float32')
        out[tf] = arr_tf.astype('float32')
    return out, end_times


def _align_single_tf_view(dfs: Dict[str, pd.DataFrame], tf: str, win: int,
                          stride: int) -> Tuple[Dict[str, np.ndarray], List[pd.Timestamp]]:
    """مسار فريم واحد عبر ``sliding_window_view`` — بلا أي نسخة إلا الأخيرة."""
    df = dfs[tf]
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError(f"DataFrame للفريم {tf} يجب أن يكون مفهرساً بـ DatetimeIndex")
    df = df.sort_index()
    n_feat = df.shape[1]

    if len(df) < win:
        return {tf: np.zeros((0, win, n_feat), dtype='float32')}, []

    values = df.to_numpy(dtype='float32', copy=False)          # (T, F)
    view = np.lib.stride_tricks.sliding_window_view(values, win, axis=0)  # (T-win+1, F, win) — VIEW
    view = np.moveaxis(view, -1, 1)                              # (T-win+1, win, F) — لا يزال VIEW
    sel = view[::stride]                                          # VIEW أيضاً (خطوة أساسية بلا نسخ)
    arr = np.ascontiguousarray(sel)                                # نسخة واحدة فقط هنا
    end_times = list(df.index[win - 1::stride])
    return {tf: arr}, end_times


## 11) بناء النوافذ والأهداف لأصل واحد (`windows.py` سابقاً)


In [40]:
# @title
"""
بناء النوافذ والأهداف لأصل واحد.

لكل نافذة يُنتَج لكل هدف سعري مُفعَّل رأسان:
  * ``{t}_class`` — اتجاه ±1 (هل تجاوزت القيمة المستقبلية قيمتها الحالية؟)
  * ``{t}_reg``   — القيمة المستقبلية نفسها بعد التطبيع (تُفكّ لاحقاً بـ ``* iqr + median``)

الرؤوس المُعطَّلة في ``CONFIG['enabled_heads']`` **لا تُحسب أصلاً**.
"""

#: أعمدة مصفوفة ``last_candles``
LAST_COLUMNS = ['last_high', 'last_low', 'last_close', 'timestamp',
                'future_close', 'future_low_min', 'future_high_max']

#: أعمدة الأسعار اللازمة لحساب الأهداف — تُبقى في البيانات دائماً حتى لو
#: استُبعدت من مدخلات النموذج عبر ``exclude_from_features``.
TARGET_COLUMNS = ('high', 'low', 'close')

#: فهرس عمود الطابع الزمني داخل ``last_candles``.
TS_COL = LAST_COLUMNS.index('timestamp')

#: ✅ ``last_candles`` تُخزَّن ``float64`` لا ``float32``: الطابع الزمني بالنانوثانية
#: يبلغ ~1.7e18، وهو يفقد ~12 ثانية من الدقة في ``float32``. هذا لا يضرّ على فريم
#: 4h لكنه يكسر أي تقسيم أو ربط زمني دقيق (وفريمات الدقائق تماماً). التكلفة
#: مهملة: المصفوفة (N, 7) مقابل مصفوفات X بحجم (N, T, F).
LAST_DTYPE = 'float64'


def prepare_single_asset(
    dfs: Dict[str, pd.DataFrame],
    tf_order: Optional[List[str]] = None,
    window_sizes: Optional[Dict[str, int]] = None,
    targets: Optional[List[str]] = None,
    forecast_horizon: Optional[int] = None,
    stride: Optional[int] = None,
    scaler_type: Optional[str] = None,
    base_tf: Optional[str] = None,
    config: Optional[dict] = None,
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    """يُرجع ``(X_tf, y_heads, base_params, last_candles)`` لأصل واحد.

    * ``X_tf``       — ``{tf: (N, T, F)}`` نوافذ مُطبَّعة.
    * ``y_heads``    — ``{head: (N,)}`` لكل رأس مُفعَّل.
    * ``base_params``— ``(N, 2)`` [المركز، المقياس] المشتقّان من نافذة الإغلاق.
    * ``last_candles``— ``(N, 7)`` انظر :data:`LAST_COLUMNS`.
    """
    config = CONFIG if config is None else config
    tf_order = config["tf_order"] if tf_order is None else tf_order
    window_sizes = config["window_sizes"] if window_sizes is None else window_sizes
    targets = config["targets"] if targets is None else targets
    forecast_horizon = config["forecast_horizon"] if forecast_horizon is None else forecast_horizon
    stride = config["stride"] if stride is None else stride
    scaler_type = config["scaler_type"] if scaler_type is None else scaler_type
    base_tf = base_tf or config.get("base_tf") or tf_order[0]
    features = feature_order(config)

    # ✅ فصل «أعمدة الأهداف» عن «أعمدة الإدخال»:
    #    الأهداف تحتاج high/low/close الخام دائماً، أمّا النموذج فقد لا يحتاجها
    #    كمدخلات (قيست 0.99 ترابطاً بينها بعد التطبيع — معلومتها الحقيقية هي
    #    الفرق بينها، وتلتقطه RANGE_rel و BODY_ratio و WICK_*).
    #    بهذا الفصل صار بالإمكان استبعادها عبر exclude_from_features بلا كسر
    #    حساب الأهداف، وهو ما كان مستحيلاً حين كان المصدر واحداً.
    dfs = {tf: df.sort_index() for tf, df in dfs.items()}
    price_df = dfs[base_tf]
    for col in ('high', 'low', 'close'):
        if col not in price_df.columns:
            raise ValueError(
                f"العمود '{col}' غير موجود في بيانات الفريم الأساسي '{base_tf}' — "
                "لا يمكن حساب الأهداف بدونه (وهو مطلوب حتى لو استُبعد من الميزات).")

    feat_dfs = {tf: dfs[tf][features] for tf in tf_order}

    windows, end_times = align_multi_timeframes_time_based(
        dfs=feat_dfs, tf_order=tf_order, window_sizes=window_sizes,
        stride=stride, config=config,
    )

    N = windows[base_tf].shape[0]
    if N == 0:
        print("   ⚠️ لا توجد نوافذ صالحة بعد المحاذاة الزمنية")
        return ({}, {},
                np.empty((0, 2), dtype='float32'),
                np.empty((0, len(LAST_COLUMNS)), dtype=LAST_DTYPE))

    # مقياس النافذة يُشتق من عمود الإغلاق داخل الميزات إن وُجد، وإلا من الأسعار
    # الخام مباشرةً — فيبقى المرجع السعري واحداً في الحالتين.
    i_close_feat = features.index('close') if 'close' in features else None

    target_heads = get_target_heads(targets, config)
    y_t = {h: np.zeros(N, dtype='float32') for h in target_heads}
    bases = np.zeros((N, 2), dtype='float32')
    last = np.zeros((N, len(LAST_COLUMNS)), dtype=LAST_DTYPE)
    reg_mode = config.get('reg_target_mode', 'return')
    _reg_clip_cfg = config.get('reg_target_clip')
    reg_clip = float(_reg_clip_cfg) if _reg_clip_cfg is not None else (
        1.0 if reg_mode == 'return' else 10.0)

    T_base = len(price_df)
    win_base = window_sizes[base_tf]
    raw_high = price_df['high'].values
    raw_low = price_df['low'].values
    raw_close = price_df['close'].values
    n_valid = N

    for i in range(N):
        base_win_arr = windows[base_tf][i]

        end_time = end_times[i]
        try:
            end_idx = price_df.index.get_loc(end_time)
        except KeyError:
            end_idx = price_df.index.searchsorted(end_time, side='right') - 1
            end_idx = max(0, min(T_base - 1, end_idx))

        # fs/fe تُحسب مرة واحدة وتُفحص قبل أي كتابة على المصفوفات
        fs = end_idx + 1
        fe = end_idx + 1 + forecast_horizon
        if fe > T_base:
            n_valid = i
            break

        # ✅ قيم الشمعة الأخيرة تُقرأ من الأسعار الخام لا من مصفوفة الميزات
        last_h = raw_high[end_idx]
        last_l = raw_low[end_idx]
        last_c = raw_close[end_idx]

        ts = price_df.index[end_idx].value
        future_close = raw_close[fe - 1]
        future_high_max = raw_high[fs:fe].max()
        future_low_min = raw_low[fs:fe].min()
        last[i] = [last_h, last_l, last_c, ts,
                   future_close, future_low_min, future_high_max]

        # مقياس النافذة من الإغلاق: من الميزات إن كان موجوداً، وإلا من الخام
        if i_close_feat is not None:
            close_win = base_win_arr[:, i_close_feat]
        else:
            close_win = raw_close[max(0, end_idx - win_base + 1):end_idx + 1]
        med_p, iqr_p = calc_scale_params(close_win, scaler_type)
        med_p=last_c
        bases[i] = [med_p, iqr_p]     # لتطبيع الميزات فقط الآن — راجع reg_mode أدناه لهدف الانحدار

        # كل هدف يُقارَن بنفس نوعه (high بـ high، low بـ low، close بـ close) —
        # لرأس الانحدار أيضاً منذ reg_target_mode='return'؛ سابقاً كان reg
        # يُقارَن دائماً بآخر *إغلاق* (med_p) بصرف النظر عن نوع الهدف، فيختلف
        # مرجعه عن مرجع class لنفس الهدف على high/low — يُضعف إشارة
        # head_agreement (راجع abstention_cfg) بلا داعٍ. الآن يتّفقان دائماً.
        own_last = {'high': last_h, 'low': last_l, 'close': last_c}
        own_future = {'high': future_high_max, 'low': future_low_min,
                      'close': future_close}

        for t in targets:
            if t not in own_last:
                continue
            # ✅ الرؤوس المُعطَّلة تُتخطّى بدل محاولة الكتابة في مفتاح غير موجود
            if f'{t}_class' in y_t:
                y_t[f'{t}_class'][i] = 1.0 if own_future[t] > own_last[t] else -1.0
            if f'{t}_reg' in y_t:
                if reg_mode == 'return':
                    # ✅ عائد مباشر بلا مرجع خارجي: (سعر_مستقبلي/آخر_سعر) - 1،
                    # بنفس مرجع رأس التصنيف أعلاه. بلا حاجة لـ iqr_p إطلاقاً —
                    # نافذتان بتقلّب مختلف تُنتجان نفس القيمة لنفس نسبة الحركة.
                    # في الوضع القديم (أدناه) تُقسَم نفس الحركة على IQR النافذة
                    # فتكبر في نافذة هادئة وتصغر في نافذة متقلّبة — انزياح مرجع
                    # الهدف هذا بين نافذة وأخرى هو ما كان يدفع النموذج نحو
                    # الانكماش للمتوسط (أرخص حلّ حين يتذبذب معنى الصفر).
                    denom = own_last[t] if abs(own_last[t]) > EPS else (
                        med_p if abs(med_p) > EPS else EPS)
                    raw = (own_future[t] - own_last[t]) / denom
                else:                                   # 'window_scale' القديم
                    # ✅ قصّ صريح لهدف الانحدار: حتى مع الحدّ النسبي في
                    # calc_scale_params، نافذة واحدة شاذة (سكون سعري مؤقت + حركة
                    # قوية لاحقة) تكفي لإنتاج هدف بمئات الأضعاف — قيس فعلياً
                    # ([-41, +728] بدل [-3, +3] المتوقَّع).
                    raw = scale_data(own_future[t], med_p, iqr_p)
                # ضمان أخير بغضّ النظر عن الوضع: خسارة محدودة دائماً، فلا تُفسد
                # عيّنة واحدة تدرّج الجذع المشترك لكل الرؤوس.
                y_t[f'{t}_reg'][i] = np.clip(raw, -reg_clip, reg_clip)

    if n_valid < N:
        for h in target_heads:
            y_t[h] = y_t[h][:n_valid]
        bases = bases[:n_valid]
        last = last[:n_valid]
        for tf in tf_order:
            windows[tf] = windows[tf][:n_valid]

    # ✅ التطبيع دفعةً واحدة لكل الفريمات: النسخة المتّجهة تُلغي حلقة بايثون
    #    المزدوجة (N نافذة × F عمود) التي كانت تستهلك أغلب زمن التجهيز.
    X_tf = {
        tf: process_windows(windows[tf], features, bases[:, 0], bases[:, 1],
                            scaler_type, config=config)
        for tf in tf_order
    }

    gc.collect()
    return X_tf, y_t, bases, last


def invert_reg_predictions(preds: np.ndarray, head: str,
                           last_candles: Optional[np.ndarray] = None,
                           bases: Optional[np.ndarray] = None,
                           config: Optional[dict] = None) -> np.ndarray:
    """يعكس تنبؤات رأس انحدار واحد إلى سعر حقيقي، حسب ``CONFIG['reg_target_mode']``.

    * ``'return'`` (الافتراضي): يحتاج ``last_candles`` — ``آخر_سعر_من_نفس_نوع_
      الهدف × (1 + التنبؤ)``. يُقرأ آخر سعر من عمود :data:`LAST_COLUMNS`
      المطابق (high/low/close) — لا من ``bases``، التي تخزّن آخر *إغلاق*
      دائماً بصرف النظر عن نوع الهدف (تخدم تطبيع الميزات فقط؛ راجع
      :func:`prepare_single_asset`).
    * ``'window_scale'`` (القديم): يحتاج ``bases`` — يُفوَّض لـ :func:`inverse_scale`
      كما كانت الدالة السابقة تعمل.

    ⚠️ **لا يعكس** تطبيعاً مقطعياً طُبِّق عبر :func:`cross_sectional_normalize`؛
    ذاك تحويل نسبي لأقران نفس اللحظة، وعكسه (لـ ``method='zscore'`` فقط) عبر
    :func:`invert_cross_sectional` باستخدام ``dataset['cs_norm_stats']``، **قبل**
    استدعاء هذه الدالة لا بعدها.
    """
    config = CONFIG if config is None else config
    target, kind = split_head_name(head)
    if kind != 'reg':
        raise ValueError(f"'{head}' ليس رأس انحدار (reg).")
    mode = config.get('reg_target_mode', 'return')

    if mode == 'window_scale':
        if bases is None:
            raise ValueError("mode='window_scale' يتطلب bases (median, iqr).")
        return inverse_scale(np.asarray(preds), np.asarray(bases))

    if last_candles is None:
        raise ValueError("mode='return' يتطلب last_candles (راجع LAST_COLUMNS).")
    col = {'high': 'last_high', 'low': 'last_low', 'close': 'last_close'}.get(target)
    if col is None:
        raise ValueError(f"لا مرجع سعر معروف للهدف '{target}'.")
    last_price = np.asarray(last_candles)[:, LAST_COLUMNS.index(col)]
    return last_price * (1.0 + np.asarray(preds, dtype='float64'))


## 12) أدوات التشخيص (`diagnostics.py` سابقاً)


In [41]:
# @title
"""
أدوات تشخيص البيانات قبل التدريب.

تكشف الأعطال الصامتة مبكراً: أعمدة ميزات مفقودة، أسماء أصول غير متطابقة بين
سجل الأصول والبيانات المُحمَّلة، وقيم مفقودة داخل الفريمات.
"""

def check_missing_values(data: Dict[str, Dict], tf_order: Optional[List[str]] = None,
                         config: Optional[dict] = None) -> Dict[str, dict]:
    """يبحث عن قيم مفقودة داخل كل (أصل × فريم) ويطبع تقريراً."""
    config = CONFIG if config is None else config
    tf_order = config["tf_order"] if tf_order is None else tf_order
    problems: Dict[str, dict] = {}

    for asset_name, tf_dict in data.items():
        if not isinstance(tf_dict, dict):
            continue
        for tf, df in tf_dict.items():
            if tf_order and tf not in tf_order:
                continue
            missing = df.isnull().sum()
            missing = missing[missing > 0]
            if len(missing):
                problems[f"{asset_name}/{tf}"] = dict(missing)
                print(f"⚠️ {asset_name}/{tf}: أعمدة فيها قيم مفقودة → {dict(missing)}")

    if not problems:
        print("✅ لا توجد قيم مفقودة في البيانات المُحمَّلة")
    return problems


def diagnose_feature_availability(data: Dict[str, Dict],
                                  tf_order: Optional[List[str]] = None,
                                  desired_features: Optional[List[str]] = None,
                                  config: Optional[dict] = None) -> List[str]:
    """يتحقق من توفر كل عمود في ``feature_order`` داخل البيانات الفعلية.

    يُرجع قائمة الأعمدة المشتركة فعلياً بين كل الأصول/الفريمات، لضمان شكل
    مصفوفات موحّد عند الدمج. أسنِد الناتج إلى ``CONFIG['feature_order']``.
    """
    config = CONFIG if config is None else config
    tf_order = config["tf_order"] if tf_order is None else tf_order
    if desired_features is None:
        desired_features = feature_order(config)

    common = set(desired_features)
    per_asset_missing: Dict[str, set] = {}
    checked = 0

    for asset, tf_dict in data.items():
        if not isinstance(tf_dict, dict):
            continue
        for tf in tf_order:
            if tf not in tf_dict:
                continue
            checked += 1
            cols = set(tf_dict[tf].columns)
            missing = set(desired_features) - cols
            if missing:
                per_asset_missing[f"{asset}/{tf}"] = missing
            common &= cols

    missing_from_all = set(desired_features) - common
    print(f"\n🔎 فحص توفر الميزات: تحقّقت من {checked} (أصل × فريم)")
    if missing_from_all:
        print(f"⚠️ {len(missing_from_all)} عمود من أصل {len(desired_features)} غير متوفر "
              f"إطلاقاً في البيانات (سيُستبعد تلقائياً):")
        print("   ", sorted(missing_from_all))
        print("   → على الأرجح: البيانات المخزَّنة أُنشئت بإعدادات add_features أو "
              "بنسخة pandas_ta مختلفة عن الحالية.")

    inconsistent = {k: v for k, v in per_asset_missing.items() if v - missing_from_all}
    if inconsistent:
        print(f"🚨 تحذير أخطر: {len(inconsistent)} (أصل/فريم) ناقصة أعمدة *إضافية* غير "
              f"مفقودة من البقية — سيُنتج شكل بيانات غير متّسق عند الدمج!")
        for k, v in list(inconsistent.items())[:5]:
            print(f"   {k}: مفقود فقط هنا → {sorted(v - missing_from_all)}")

    available = [c for c in desired_features if c in common]
    print(f"✅ سيُستخدَم فعلياً {len(available)} عمود من أصل {len(desired_features)}\n")
    return available


def _strip_quote_suffix(s) -> str:
    s = str(s).strip().upper()
    for suf in ("USDT", "USDC", "BUSD", "-USDT", "/USDT", "_USDT"):
        if s.endswith(suf):
            return s[: -len(suf)]
    return s


def diagnose_data_vs_configs(data: dict, configs: list, n_preview: int = 15) -> dict:
    """مقارنة أسماء الأصول بين ``configs`` و``data`` بعدة استراتيجيات مطابقة.

    السبب الأشيع لخروج مجموعة بيانات فارغة هو اختلاف التسمية (حالة أحرف،
    مسافات، لاحقة USDT). هذه الدالة تكشفه فوراً.
    """
    print("=" * 90)
    print("1) بنية data")
    print("=" * 90)
    print(f"النوع: {type(data).__name__} | عدد المفاتيح: {len(data)}")

    data_keys = list(data.keys())
    print(f"\nأول {n_preview} مفتاحاً (repr لكشف المسافات والرموز المخفية):")
    for k in data_keys[:n_preview]:
        print(f"   {repr(k)}")

    if data_keys:
        k0 = data_keys[0]
        v0 = data[k0]
        print(f"\nنوع القيمة لأول مفتاح {repr(k0)}: {type(v0).__name__}")
        if isinstance(v0, dict):
            print(f"مفاتيح المستوى الثاني (الفريمات): {list(v0.keys())}")
            if v0:
                k1 = list(v0.keys())[0]
                v1 = v0[k1]
                if hasattr(v1, "shape"):
                    print(f"شكل الفريم {repr(k1)}: {v1.shape}")
                if hasattr(v1, "columns"):
                    print(f"أول 10 أعمدة: {list(v1.columns)[:10]}")
        else:
            print("⚠️ القيمة ليست dict — هل البنية data[tf][asset] بدل data[asset][tf]؟")
            print(f"   محتوى مختصر: {str(v0)[:300]}")

    print("\n" + "=" * 90)
    print("2) أسماء configs")
    print("=" * 90)
    print(f"عدد configs: {len(configs)}")
    config_names = [c.get("name") if isinstance(c, dict) else c for c in configs]
    for n in config_names[:n_preview]:
        print(f"   {repr(n)}")

    print("\n" + "=" * 90)
    print("3) استراتيجيات المطابقة")
    print("=" * 90)
    cfg_set = {n for n in config_names if n is not None}
    data_set = set(data_keys)

    exact = cfg_set & data_set
    print(f"✅ تطابق تام: {len(exact)} / {len(cfg_set)}")

    cfg_stripped = {str(n).strip(): n for n in cfg_set}
    data_stripped = {str(k).strip(): k for k in data_keys}
    stripped_match = set(cfg_stripped) & set(data_stripped)
    print(f"🔤 بعد strip(): {len(stripped_match)} / {len(cfg_stripped)}")

    cfg_lower = {str(n).strip().lower(): n for n in cfg_set}
    data_lower = {str(k).strip().lower(): k for k in data_keys}
    lower_match = set(cfg_lower) & set(data_lower)
    print(f"🔡 بتجاهل حالة الأحرف: {len(lower_match)} / {len(cfg_lower)}")

    cfg_base = {_strip_quote_suffix(n): n for n in cfg_set}
    data_base = {_strip_quote_suffix(k): k for k in data_keys}
    base_match = set(cfg_base) & set(data_base)
    print(f"💱 بعد إزالة لاحقة USDT/USDC/BUSD: {len(base_match)} / {len(cfg_base)}")
    if base_match and not exact:
        print("   أمثلة:")
        for b in list(base_match)[:8]:
            print(f"      config: {repr(cfg_base[b])}  ↔  data: {repr(data_base[b])}")

    if not exact and not base_match:
        print("\n   ⚠️ لا يوجد تطابق يُذكر — راجع مصدر data أو بنيتها (القسم 1).")

    return {
        "data_keys": data_keys,
        "config_names": config_names,
        "exact_match": exact,
        "stripped_match": stripped_match,
        "lower_match": lower_match,
        "base_match": base_match,
    }


def summarize_dataset(dataset: dict) -> dict:
    """ملخّص رقمي لمجموعة بيانات مُجهَّزة — يُستخدم كفحص أخير قبل الحفظ."""
    print("═" * 66)
    print("📦 ملخّص مجموعة البيانات")
    print("═" * 66)
    n = len(dataset['base_params'])
    print(f"  العيّنات      : {n:,}")
    print(f"  الفريمات      : {dataset['timeframes']}")
    print(f"  الرؤوس        : {dataset['targets']}")
    print(f"  الميزات       : {len(dataset['feature_order'])}")
    print(f"  عدد الأصول    : {len(dataset.get('asset_bounds', []))}")

    stats = {}
    for tf in dataset['timeframes']:
        X = dataset[f'X_{tf}']
        nan = int(np.isnan(X).sum())
        inf = int(np.isinf(X).sum())
        stats[f'X_{tf}'] = {'shape': X.shape, 'nan': nan, 'inf': inf}
        flag = "✅" if (nan == 0 and inf == 0) else "🚨"
        print(f"  {flag} X_{tf}: {X.shape} | NaN={nan} | Inf={inf}")

    for h in dataset['targets']:
        y = dataset[f'y_{h}']
        uniq = np.unique(y)
        if len(uniq) <= 3:
            counts = {float(u): int((y == u).sum()) for u in uniq}
            balance = ", ".join(f"{k:+.0f}: {v / len(y):.1%}" for k, v in counts.items())
            print(f"  📊 y_{h}: توازن الفئات → {balance}")
            stats[f'y_{h}'] = counts
        else:
            print(f"  📊 y_{h}: متوسط={y.mean():+.4f} | انحراف={y.std():.4f} "
                  f"| المدى=[{y.min():+.3f}, {y.max():+.3f}]")
            stats[f'y_{h}'] = {'mean': float(y.mean()), 'std': float(y.std())}
    print("═" * 66)
    return stats


## 13) التحميل والمعالجة المتوازية (`parallel.py` سابقاً)


In [42]:
# @title
"""
تحميل ومعالجة متوازية للأصول.

**لماذا يفيد التوازي هنا؟** المسار التسلسلي يقضي أغلب وقته منتظراً الشبكة:
كل عملة تعني طلب ``read_csv`` مستقلاً من Google Drive. مع 77 عملة يصبح ذلك
77 انتظاراً متتابعاً. والخيوط (threads) كافية تماماً — لا حاجة لعمليات
منفصلة — لأن كلاً من انتظار الشبكة، وتحليل CSV في pandas، وحسابات numpy
داخل ``prepare_single_asset`` **تُحرِّر قفل المفسّر (GIL)** أثناء عملها.

**الترتيب محفوظ.** :func:`imap_ordered` تُرجع النتائج بترتيب الإدخال مهما
اختلف ترتيب اكتمالها، فيبقى ``asset_bounds`` وترتيب الصفوف في مجموعة البيانات
**قابلاً لإعادة الإنتاج بالضبط** — وهذا شرط لمقارنة تجربتين بأمان.

**الذاكرة مضبوطة.** ``prefetch`` يحدّ عدد المهام الطائرة في وقت واحد، فلا
تُحمَّل كل العملات دفعةً في الذاكرة إن لم تُرِد ذلك (مهم على Colab).
"""

def _system_ram_mb() -> Optional[float]:
    """رام النظام الكلي بالميجابايت من ``/proc/meminfo`` (لينكس — بيئة Colab
    القياسية). ``None`` إن تعذّرت القراءة (نظام غير لينكس مثلاً) — لا يُعطَّل
    شيء عندها، فقط يُفقَد هذا السقف الاحترازي الإضافي."""
    try:
        with open('/proc/meminfo') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    return int(line.split()[1]) / 1024.0
    except Exception:                                          # noqa: BLE001
        pass
    return None


def default_workers(n_items: int = 0) -> int:
    """عدد خيوط افتراضي معقول: مقيَّد بالمهام المتاحة، وعدد الأنوية، ورام
    النظام الكلي.

    ✅ سقف رام إضافي (جيجا واحد متاح لكل خيط تقريباً، تقدير محافظ): على آلة
    بأنوية كثيرة لكن رام محدودة (شائع في Colab المجانية، ~12 جيجا) كان عدد
    الخيوط يُشتقّ من الأنوية وحدها فيُفرط في التوازي، وكل خيط يحمل ذروته
    الخاصة من `align_multi_timeframes_time_based`/`process_windows` —
    تراكمها معاً هو ما سبّب استهلاك رام أكبر من حجم البيانات النهائي بأضعاف.
    هذا السقف احترازي إضافي فوق إصلاح الذروة نفسها في الدالتين أعلاه، لا بديل
    عنه (لا يعرف حجم كل أصل مسبقاً، فهو تقدير خشن لا مضبوط بدقة).
    """
    cpu = os.cpu_count() or 4
    workers = min(100, max(4, cpu * 2))
    ram_mb = _system_ram_mb()
    if ram_mb is not None:
        ram_cap = max(2, int(ram_mb // 1024))
        workers = min(workers, ram_cap)
    return max(1, min(workers, n_items)) if n_items else workers


def imap_ordered(fn: Callable[[T], R],
                 items: Iterable[T],
                 max_workers: int = 8,
                 prefetch: Optional[int] = None) -> Iterator[R]:
    """``map`` متوازٍ يُرجع النتائج **بترتيب الإدخال** مع تقييد المهام الطائرة.

    Args:
        max_workers: عدد الخيوط. ``<= 1`` ينفّذ تسلسلياً بلا أي تجمّع خيوط.
        prefetch: أقصى عدد مهام طائرة معاً (افتراضياً ``max_workers * 2``).
            يحدّ الذاكرة: لا تُنجَز نتائج أكثر من هذا قبل استهلاكها.
    """
    items = list(items)
    if max_workers <= 1 or len(items) <= 1:
        for it in items:
            yield fn(it)
        return

    prefetch = prefetch or max_workers * 2
    prefetch = max(prefetch, max_workers)

    with ThreadPoolExecutor(max_workers=max_workers,
                            thread_name_prefix="cm_asset") as pool:
        pending: Dict[int, object] = {}
        next_in = 0
        next_out = 0

        while next_out < len(items):
            # املأ نافذة المهام الطائرة
            while next_in < len(items) and len(pending) < prefetch:
                pending[next_in] = pool.submit(fn, items[next_in])
                next_in += 1

            # سلّم كل ما جهز **بالترتيب** من مقدّمة الطابور
            if next_out in pending and pending[next_out].done():
                yield pending.pop(next_out).result()
                next_out += 1
                continue

            # انتظر اكتمال أي مهمة ثم أعد المحاولة
            wait([pending[next_out]] if next_out in pending
                 else list(pending.values()),
                 return_when=FIRST_COMPLETED)
            if next_out in pending:
                yield pending.pop(next_out).result()
                next_out += 1


def load_assets(configs: List[Dict],
                load_asset_fn: Callable,
                max_workers: Optional[int] = None,
                prefetch: Optional[int] = None,
                tail: int = 0,
                verbose: bool = True) -> Dict[str, pd.DataFrame]:
    """يُحمّل **كل** العملات دفعةً واحدة بالتوازي ويُرجع ``{name: DataFrame}``.

    مفيد حين تريد البيانات الخام في الذاكرة لفحصها أو لإعادة استخدامها في عدة
    تجارب بلا إعادة تنزيل. للبناء المباشر لمجموعة بيانات فضّل
    :func:`build_dataset_from_loader` بمعامل ``max_workers`` — فهي تُعالج
    أثناء التنزيل ولا تحتفظ بكل شيء في الذاكرة.

    الأصول التي يفشل تحميلها تُتخطّى مع تقرير، ولا تُوقف البقية.
    """
    max_workers =    max_workers or default_workers(len(configs))

    def _one(cfg: Dict) -> Tuple[str, Optional[pd.DataFrame], str]:
        name = cfg['name'] if isinstance(cfg, dict) else str(cfg)
        try:
            df = load_asset_fn(cfg.get('file_id'), name)
            if tail:
                df = df[-tail:]
            return name, df, ''
        except Exception as exc:                       # noqa: BLE001
            return name, None, str(exc)

    out: Dict[str, pd.DataFrame] = {}
    failed: List[Tuple[str, str]] = []

    if verbose:
        print(f"⏬ تحميل {len(configs)} عملة بـ {max_workers} خيطاً...")

    for i, (name, df, err) in enumerate(
            imap_ordered(_one, configs, max_workers, prefetch), start=1):
        if df is None:
            failed.append((name, err))
            if verbose:
                print(f"   [{i}/{len(configs)}] ⚠️ {name}: {err[:70]}")
            continue
        out[name] = df
        if verbose:
            print(f"   [{i}/{len(configs)}] ✅ {name}: {len(df):,} صف")

    if verbose:
        print(f"✅ حُمّلت {len(out)} من {len(configs)} عملة")
        if failed:
            print(f"   ⚠️ فشل {len(failed)}: {[n for n, _ in failed][:8]}")
    return out


## 14) مصادر البيانات التاريخية (`sources.py` سابقاً) — Google Drive المُركَّب حصراً

**تبسيط جوهري عن النسخة الأصلية:** الملف الأصلي كان يدعم طريقتين للتحميل
(Drive مُركَّب، أو رابط عام عبر `gdown`) مع منطق تبديل تكيّفي بينهما. هنا
**طريقة واحدة فقط**: Drive المُركَّب. سجل الأصول أيضاً يُقرأ من Drive
(`CONFIG['asset_registry_path']`) لا من رابط عام.

> ℹ️ `ALL_GLOBAL` مُعرَّفة هنا (لا في قسم الإعدادات) لأنها تنتمي أصلاً لهذا
> الملف في الحزمة الأصلية — تعني "لا تصفية فئات، خُذ سجل الأصول كاملاً".


In [43]:
# @title
"""
مصادر البيانات التاريخية: سجل الأصول، تحميل العملات، وإعادة أخذ العينات.

هذه هي الطبقة الوحيدة التي تعرف **من أين** تأتي البيانات التاريخية؛ بقية
الحزمة تتعامل مع ``DataFrame`` مفهرس زمنياً فقط. (للبيانات **الحيّة** من
Binance، انظر القسم اللاحق "تحضير بيانات حيّة".)
"""

#: قيمة خاصة لـ categories في flatten_coins_by_category: خُذ كل عملات سجل
#: الأصول بلا أي تصفية بالفئات.
ALL_GLOBAL = 'all_global'

OHLCV_AGG = {'open': 'first', 'high': 'max', 'low': 'min',
             'close': 'last', 'volume': 'sum'}


# ══════════════════════════════════════════════════════════════════════════
# سجل الأصول
# ══════════════════════════════════════════════════════════════════════════
def load_asset_registry(path: Optional[str] = None,
                        config: Optional[dict] = None) -> List[Dict]:
    """سجل العملات ``[{'name':..., ...}, ...]`` — من CSV محلي صريح (``path``)،
    أو من Google Drive المُركَّب حسب ``CONFIG['asset_registry_path']``
    (مسار نسبي داخل ``MyDrive``). لا رابط عام في أي من الحالتين."""
    config = CONFIG if config is None else config
    if path:
        df = pd.read_csv(path)
    else:
        rel = config.get("asset_registry_path")
        if not rel:
            raise ValueError("مرّر path أو عيّن CONFIG['asset_registry_path'].")
        root = mount_drive(config=config)
        if root is None:
            raise RuntimeError("تعذّر تركيب Google Drive — تأكد من العمل داخل Colab.")
        df = pd.read_csv(root / rel)
    print(f"📒 سجل الأصول: {len(df)} عملة")
    return df.to_dict('records')


def flatten_coins_by_category(category_dict: Optional[Dict[str, List[str]]] = None,
                              categories: Optional[List[str]] = None) -> List[str]:
    """تسطيح قاموس الفئات إلى قائمة أسماء واحدة.

    Args:
        categories: الفئات المطلوبة (افتراضياً :data:`DEFAULT_ENABLED_CATEGORIES`
            — ``["Large_Caps"]`` بالإعداد الافتراضي).
            ``'all'`` أو ``[]`` → كل الفئات المُعرَّفة في :data:`COINS_BY_CATEGORY`.
            ``'all_global'`` (أي :data:`ALL_GLOBAL`) → لا تصفية إطلاقاً (تُعالَج
            في :func:`filter_desired_coins`، فتُؤخذ كل عملة في السجل حتى لو لم
            تُذكر في أي فئة).
    """
    category_dict = COINS_BY_CATEGORY if category_dict is None else category_dict
    if categories is None:
        categories = DEFAULT_ENABLED_CATEGORIES
    if categories == ALL_GLOBAL:
        print("🌍 all_global: كل عملات السجل بلا تصفية")
        return []
    if categories == 'all' or not categories:
        selected = list(category_dict.keys())
    else:
        selected = [c for c in categories if c in category_dict]
        unknown = set(categories) - set(category_dict)
        if unknown:
            print(f"⚠️ فئات غير معروفة تُتجاهَل: {sorted(unknown)}")

    flat: List[str] = []
    for cat in selected:
        flat.extend(category_dict[cat])
    print(f"🎯 الفئات المختارة: {selected} → {len(flat)} عملة")
    return flat


def filter_desired_coins(full_registry: List[Dict],
                         desired_coins: Optional[List[str]] = None,
                         verbose: bool = True) -> List[Dict]:
    """تصفية سجل الأصول على العملات المطلوبة، مع تقرير عن غير الموجود.

    ``desired_coins`` فارغة أو ``None`` → **لا تصفية**: يُؤخذ السجل كاملاً
    (سلوك ``categories=ALL_GLOBAL``).
    """
    if not desired_coins:
        usable = [c for c in full_registry if c.get('name')]
        if verbose:
            print(f"🌍 بلا تصفية: {len(usable)} عملة من السجل كاملاً")
            dropped = len(full_registry) - len(usable)
            if dropped:
                print(f"   ⚠️ {dropped} سطراً بلا عمود 'name' صالح تُتجاهَل")
        return usable

    desired_set = set(desired_coins)
    selected = [coin for coin in full_registry if coin.get('name') in desired_set]
    if verbose:
        found = {c['name'] for c in selected}
        missing = sorted(desired_set - found)
        print(f"✅ وُجد {len(selected)} من أصل {len(desired_set)} عملة مطلوبة")
        if missing:
            print(f"   ⚠️ غير موجودة في السجل: {missing[:10]}"
                  f"{'...' if len(missing) > 10 else ''}")
    return selected


# ══════════════════════════════════════════════════════════════════════════
# استبعاد عملات بعينها
# ══════════════════════════════════════════════════════════════════════════
def _coin_key(name: Any) -> str:
    """مفتاح المطابقة: أحرف كبيرة بلا مسافات طرفية — ``' btcusdt '`` = ``'BTCUSDT'``."""
    return str(name).strip().upper()


def _coin_name(item: Any) -> str:
    """اسم العملة من عنصر ``configs`` (قاموس فيه ``name``) أو من نص مباشر."""
    return str(item.get('name', '')) if isinstance(item, dict) else str(item)


def split_excluded_coins(configs: Iterable[Any],
                         excluded: Optional[Iterable[str]] = None,
                         config: Optional[dict] = None
                         ) -> Tuple[List[Any], List[str], List[str]]:
    """يفصل ``configs`` إلى ``(المُبقاة، أسماء المُستبعَدة فعلاً، أسماء لم تُوجد)``.

    Args:
        excluded: أسماء العملات المطلوب تخطّيها. ``None`` → تُؤخذ من
            ``CONFIG['excluded_coins']``؛ قائمة صريحة (حتى الفارغة) تحلّ محلّها
            ولا تُدمج معها. نصّ واحد (``'BTCUSDT'``) يُعامَل كاسم واحد لا كأحرف.

    المطابقة على **الاسم الكامل** بلا حساسية لحالة الأحرف. لا تُطابَق بالجذر
    (``'BTC'`` لا تستبعد ``BTCUSDT``) عمداً: الجذر الواحد قد يجمع عملات مختلفة
    (``BTCUSDT``/``BTCDOMUSDT``)، والاستبعاد الخاطئ الصامت أسوأ من رفض صريح.
    الأسماء التي لم تُوجد في ``configs`` تُرجَع للتحذير منها (خطأ إملائي غالباً).
    ``configs`` نفسها لا تُعدَّل.
    """
    config = CONFIG if config is None else config
    if excluded is None:
        excluded = config.get('excluded_coins')
    if excluded is None:
        excluded = []
    elif isinstance(excluded, str):
        excluded = [excluded]

    wanted: Dict[str, str] = {}
    for n in excluded:
        if str(n).strip():
            wanted.setdefault(_coin_key(n), str(n).strip())

    kept: List[Any] = []
    removed: List[str] = []
    seen = set()
    for item in configs:
        key = _coin_key(_coin_name(item))
        if key in wanted:
            removed.append(_coin_name(item))
            seen.add(key)
        else:
            kept.append(item)
    not_found = [orig for key, orig in wanted.items() if key not in seen]
    return kept, removed, not_found


def exclude_coins(configs: Iterable[Any],
                  excluded: Optional[Iterable[str]] = None,
                  config: Optional[dict] = None,
                  verbose: bool = True) -> List[Any]:
    """يُرجع ``configs`` بعد حذف العملات المُستبعَدة، فتُتخطّى في كل ما يليها.

    >>> configs = exclude_coins(configs, ['TSLAUSDT', 'XAUUSDT'])

    ``build_dataset*`` تُطبّق ``CONFIG['excluded_coins']`` تلقائياً، فلا حاجة
    لاستدعائها يدوياً إلا لتصفية قائمة قبل بناء الدفعة (أو لمرة واحدة بلا
    لمس ``CONFIG``). تفاصيل المطابقة: :func:`split_excluded_coins`.
    """
    kept, removed, not_found = split_excluded_coins(configs, excluded, config)
    if verbose:
        if removed:
            print(f"🚫 استُبعدت {len(removed)} عملة: {removed[:10]}"
                  f"{'...' if len(removed) > 10 else ''}")
        if not_found:
            print(f"   ⚠️ أسماء استبعاد غير موجودة في configs — تحقّق من الإملاء: "
                  f"{not_found[:10]}{'...' if len(not_found) > 10 else ''}")
    return kept


# ══════════════════════════════════════════════════════════════════════════
# تحميل عملة واحدة — Google Drive المُركَّب حصراً
# ══════════════════════════════════════════════════════════════════════════
def load_asset(file_id: Optional[str], name: str = "",
               config: Optional[dict] = None) -> pd.DataFrame:
    """تحميل عملة واحدة كـ ``DataFrame`` مفهرس بـ ``timestamp`` ومرتّب زمنياً —
    **حصراً** من Google Drive المُركَّب (``drive.mount``). لا رابط عام
    (``uc?id=``) ولا ``gdown`` في أي مسار من مسارات هذه الدالة.

    ``name`` (أو ``file_id`` إن لم يُحدَّد ``name``) يُستخدَم لبناء اسم الملف
    داخل ``CONFIG['drive_raw_dir']`` حسب نمط ``CONFIG['drive_raw_pattern']``.

    الإمضاء ``(file_id, name, config=None)`` مطابق عمداً لما تتوقّعه
    :func:`build_dataset_from_loader` (تستدعي ``load_asset_fn(cfg.get('file_id'), name)``)،
    حتى لو لم يعد ``file_id`` يحمل أي معنى فعلي هنا.
    """
    config = CONFIG if config is None else config
    key = name or file_id
    if not key:
        raise ValueError("يلزم name أو file_id لتحديد اسم ملف العملة.")

    raw_dir = config.get("drive_raw_dir")
    if not raw_dir:
        raise ValueError("عيّن CONFIG['drive_raw_dir'].")
    pattern = config.get("drive_raw_pattern", "{name}.csv")

    root = mount_drive(config=config)
    if root is None:
        raise RuntimeError("تعذّر تركيب Google Drive — تأكد من العمل داخل Colab.")

    path = root / raw_dir / pattern.format(name=key)
    if not path.exists():
        raise FileNotFoundError(f"[{key}] الملف غير موجود: {path}")

    df = pd.read_csv(path)
    df.columns = df.columns.str.lower()

    # ملفات history_1d تحمل ``datetime_utc`` (وقتاً مقروءاً) وقد يحمل الملف أيضاً
    # ``timestamp`` كأرقام ms — تفسيرها كنص وقت يعطي تواريخ 1970. فيُفضَّل
    # ``datetime_utc``؛ وبقاء ``timestamp`` احتياطاً يُبقي ملفات 1h القديمة تعمل.
    ts_col = 'datetime_utc' if 'datetime_utc' in df.columns else 'timestamp'
    if ts_col not in df.columns:
        raise ValueError(
            f"[{key}] لا عمود 'datetime_utc' ولا 'timestamp' — "
            f"الأعمدة المتاحة: {list(df.columns)}")

    df['timestamp'] = pd.to_datetime(df[ts_col], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp']).set_index('timestamp').sort_index()
    return df


#: اسم متوافق مع الكود القديم الذي كان يستدعي load_asset2 صراحةً — نفس الدالة
#: الوحيدة الآن.
load_asset2 = load_asset


# ══════════════════════════════════════════════════════════════════════════
# إعادة أخذ العينات
# ══════════════════════════════════════════════════════════════════════════
def resample_timeframes(df: pd.DataFrame,
                        target_tfs: Optional[List[str]] = None,
                        base_tf: Optional[str] = None,
                        with_features: bool = True,
                        config: Optional[dict] = None) -> Dict[str, pd.DataFrame]:
    """تحويل ``df`` إلى كل الفريمات المطلوبة، مع إضافة المؤشرات لكل فريم.

    Args:
        target_tfs: الفريمات المطلوبة بصيغة pandas offset (``['1h','4h','1D']``).
        base_tf: فريم ``df`` الأصلي؛ إن لم يُعطَ يُفترض أنه أصغر فريم مطلوب.
        with_features: True يستدعي :func:`add_features` على كل فريم — اتركها True
            وإلا لن تطابق الأعمدةُ ``feature_order``.
    """
    config = CONFIG if config is None else config
    target_tfs = config["tf_order"] if target_tfs is None else target_tfs
    df = df.sort_index()

    smallest = target_tfs[0]
    if base_tf != smallest:
        df = df.resample(smallest).agg(OHLCV_AGG).dropna(how='all')
        base_tf = smallest

    dfs: Dict[str, pd.DataFrame] = {}
    for tf in target_tfs:
        frame = df if tf == base_tf else df.resample(tf).agg(OHLCV_AGG).dropna(how='all')
        dfs[tf] = add_features(frame.copy(), config=config) if with_features else frame.copy()
    return dfs


def make_resample_fn(config: Optional[dict] = None):
    """يُنتج ``resample_fn`` مربوطة بـ ``config`` — جاهزة لـ ``build_dataset_from_loader``
    ولخط الأنابيب الحيّ أيضاً (القسم اللاحق)."""
    config = CONFIG if config is None else config

    def _fn(df: pd.DataFrame, tf_order: List[str]) -> Dict[str, pd.DataFrame]:
        return resample_timeframes(df, tf_order, config=config)

    return _fn


## 15) خط أنابيب تجميع البيانات عبر كل الأصول (`pipeline.py` سابقاً)


## 14-ب) سياق سوقي عابر للأصول (Market Context)


In [44]:
# @title
"""
سياق سوقي عابر للأصول (Cross-Asset Market Context).

كل نافذة كانت تُبنى بمعزل تام عن باقي السوق — لا ميزة تخبر النموذج "هل BTC
يرتفع أم ينخفض الآن؟" رغم أن حركة أغلب العملات مرتبطة به جزئياً. الحل هنا
مستوحى من ``include_corr_pairlist`` في FreqAI (إطار تعلّم آلي لبوت التداول
مفتوح المصدر freqtrade): تُضاف ميزات عملة مرجعية (BTC افتراضياً) كأعمدة إدخال
إضافية (بادئة ``MKT_``) لكل عملة أخرى، بعد محاذاتها زمنياً.

**للعملة المرجعية نفسها**: القيمة صفر دائماً — عائد BTC نسبة لنفسه صفر
بالتعريف، وأي قيمة أخرى كانت ستُسرِّب معلومة دائرية (الهدف يتنبأ بنفسه جزئياً
عبر ميزة مطابقة تقريباً).
"""


def market_context_columns(config: Optional[dict] = None) -> List[str]:
    """أسماء أعمدة السياق السوقي — قائمة ثابتة تُحسب من الإعدادات وحدها
    (بلا بيانات فعلية)، لتُدرَج في feature_order تلقائياً عبر infer_feature_columns."""
    config = CONFIG if config is None else config
    mc = config.get('market_context') or {}
    if not mc.get('enabled', False):
        return []
    periods = mc.get('return_periods') or [1]
    return [f'MKT_ret_{p}' for p in periods]


def _market_return_frame(price_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """عوائد الإغلاق للعملة المرجعية على فتراتها المُعدَّة — إطار خفيف
    (عمود واحد لكل فترة) يُحاذى لاحقاً بفهرس كل عملة أخرى."""
    mc = config.get('market_context') or {}
    periods = mc.get('return_periods') or [1]
    close = price_df['close']
    out = pd.DataFrame(index=price_df.index)
    for p in periods:
        out[f'MKT_ret_{p}'] = close.pct_change(int(p))
    return out


def build_market_context(load_asset_fn: Optional[Callable] = None,
                         resample_fn: Optional[Callable] = None,
                         data: Optional[Dict] = None,
                         config: Optional[dict] = None) -> Optional[Dict[str, pd.DataFrame]]:
    """يبني (مرّة واحدة قبل حلقة الأصول المتوازية) عوائد العملة المرجعية لكل
    فريم في ``tf_order`` — تُمرَّر بعدها إلى :func:`add_market_context` لكل
    عملة. ``None`` إن كان ``market_context`` معطَّلاً، أو تعذّر تحميل المرجع
    (يُحذَّر ولا يُوقِف البناء — كل العملات تحصل على سياق صفري بدلاً منه).

    مرّر ``data`` للمصدر المُجهَّز مسبقاً، أو ``load_asset_fn``+``resample_fn``
    لتحميل المرجع مباشرة من Binance/الملفات الخام — بصرف النظر عمّا إذا كانت
    العملة المرجعية ضمن ``configs`` المطلوبة فعلياً أم لا.
    """
    config = CONFIG if config is None else config
    mc = config.get('market_context') or {}
    if not mc.get('enabled', False):
        return None
    ref = mc.get('reference_symbol', 'BTCUSDT')
    tf_order = config['tf_order']
    try:
        if data is not None:
            if ref not in data:
                raise KeyError(f"'{ref}' غير موجودة في data المُجهَّزة مسبقاً")
            raw = {tf: data[ref][tf] for tf in tf_order if tf in data[ref]}
        else:
            if load_asset_fn is None or resample_fn is None:
                raise ValueError("يلزم load_asset_fn+resample_fn أو data")
            df = load_asset_fn(None, ref)
            raw = resample_fn(df, tf_order)
        return {tf: _market_return_frame(raw[tf], config) for tf in tf_order if tf in raw}
    except Exception as exc:                                   # noqa: BLE001
        print(f"   ⚠️ تعذّر بناء السياق السوقي من '{ref}': {exc} — "
              f"سيُترَك بلا سياق سوقي (أعمدة MKT_ = صفر للجميع).")
        return None


def add_market_context(dfs: Dict[str, pd.DataFrame],
                       market_dfs: Optional[Dict[str, pd.DataFrame]],
                       is_reference: bool,
                       config: Optional[dict] = None) -> Dict[str, pd.DataFrame]:
    """يُلحق أعمدة MKT_ إلى ``dfs`` (لكل فريم) — محاذاة زمنية بـ
    ``reindex(..., method='ffill')`` على فهرس كل عملة، فلا تحتاج العملتان
    نفس الطول أو نفس أوقات الشموع بالضبط. عدم توفّر ``market_context`` أصلاً
    (أعمدته فارغة) يُعيد ``dfs`` كما هي بلا نسخ إضافي.
    """
    cols = market_context_columns(config)
    if not cols:
        return dfs
    out = {}
    for tf, df in dfs.items():
        d = df.copy()
        if is_reference or market_dfs is None or tf not in market_dfs:
            for c in cols:
                d[c] = 0.0
        else:
            aligned = market_dfs[tf].reindex(d.index, method='ffill')
            for c in cols:
                d[c] = aligned[c].fillna(0.0) if c in aligned.columns else 0.0
        out[tf] = d
    return out


In [45]:
# @title
"""
نقاط استئناف لكل أصل — تحمي من فقدان بيانات مُعالَجة عند انقطاع الجلسة
(نفاد الرام، انقطاع الاتصال، إعادة تشغيل يدوية) في ``build_dataset_from_loader``.

**الفكرة:** كل أصل يُحفَظ على القرص (Drive عادة) فور معالجته، لا فقط في نهاية
البناء كله. عند إعادة التشغيل، الأصول المحفوظة سابقاً تُحمَّل من القرص مباشرة
(بلا شبكة ولا إعادة حساب)، ويُعالَج فقط ما تبقّى.

⚠️ **الحماية من التقادم:** نقطة استئناف محفوظة بإعدادات مختلفة (نافذة، ميزات،
أهداف...) عن التشغيل الحالي خطر حقيقي — دمج بيانات بشكل قديم مع أخرى بشكل
جديد صامتاً. لهذا تُحفَظ **بصمة** الإعدادات المؤثّرة مع كل نقطة استئناف
وتُقارَن عند التحميل: أي اختلاف يُسقط نقطة الاستئناف (تُعاد معالجة ذلك الأصل)
بدل دمج غير متجانس بصمت.
"""


def _checkpoint_fingerprint(config: dict, features: List[str], tail: int = 0) -> str:
    """بصمة كل ما يؤثّر في شكل/معنى بيانات عملة واحدة مُعالَجة — تُحفَظ مع كل
    نقطة استئناف. أي تغيير في هذه القيم يُبطل نقاط الاستئناف القديمة تلقائياً."""
    import hashlib
    parts = {
        'feature_order': list(features),
        'tf_order': list(config['tf_order']),
        'window_sizes': {tf: config['window_sizes'][tf] for tf in config['tf_order']},
        'forecast_horizon': config['forecast_horizon'],
        'targets': list(config['targets']),
        'enabled_heads': config.get('enabled_heads', {}),
        'scaler_type': config['scaler_type'],
        'stride': config['stride'],
        'reg_target_mode': config.get('reg_target_mode', 'return'),
        'market_context': config.get('market_context', {}),
        'funding_rate': config.get('funding_rate', {}),
        'open_interest': config.get('open_interest', {}),
        'tail': tail,
    }
    blob = json.dumps(parts, sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha256(blob).hexdigest()[:16]


def _checkpoint_paths(checkpoint_dir, name: str) -> Tuple[Path, Path]:
    d = Path(checkpoint_dir)
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{name}.npz", d / f"{name}.meta.json"


def _save_asset_checkpoint(checkpoint_dir, name: str, X_tf: Dict[str, np.ndarray],
                           y_t: Dict[str, np.ndarray], bases: np.ndarray,
                           last: np.ndarray, fingerprint: str) -> None:
    """يحفظ نتيجة أصل واحد على القرص — كتابة ذرّية (ملف مؤقت ثم إعادة تسمية)
    فلا يُخلَّف ملف نصفي تالف لو انقطعت الجلسة أثناء الحفظ نفسه."""
    npz_path, meta_path = _checkpoint_paths(checkpoint_dir, name)
    payload = {f"X__{tf}": arr for tf, arr in X_tf.items()}
    payload.update({f"y__{h}": arr for h, arr in y_t.items()})
    payload["bases"] = bases
    payload["last"] = last

    # ✅ الاسم المؤقّت ينتهي بـ .npz أيضاً: np.savez تُلحق ".npz" تلقائياً إن لم
    # ينتهِ الاسم بها فعلاً — ".npz.tmp" كان سيُصبح فعلياً ".npz.tmp.npz"،
    # فيفشل replace() لاحقاً (يبحث عن اسم لم يُكتَب قط).
    tmp_npz = npz_path.with_name(f"{name}.tmp.npz")
    np.savez(tmp_npz, **payload)
    tmp_npz.replace(npz_path)

    tmp_meta = meta_path.with_suffix(".json.tmp")
    tmp_meta.write_text(json.dumps({"fingerprint": fingerprint, "n_samples": int(bases.shape[0])}))
    tmp_meta.replace(meta_path)


def _load_asset_checkpoint(checkpoint_dir, name: str, fingerprint: str):
    """يحمّل نقطة استئناف أصل — ``None`` إن لم تُوجد، أو وُجدت ببصمة مختلفة
    (إعدادات تغيّرت)، أو تعذّرت قراءتها (ملف تالف من انقطاع سابق)."""
    npz_path, meta_path = _checkpoint_paths(checkpoint_dir, name)
    if not (npz_path.exists() and meta_path.exists()):
        return None
    try:
        meta = json.loads(meta_path.read_text())
        if meta.get("fingerprint") != fingerprint:
            return None
        with np.load(npz_path) as z:
            X_tf = {k[3:]: z[k] for k in z.files if k.startswith("X__")}
            y_t = {k[3:]: z[k] for k in z.files if k.startswith("y__")}
            bases = z["bases"]
            last = z["last"]
        return X_tf, y_t, bases, last
    except Exception:                                            # noqa: BLE001
        return None


def clear_checkpoint(checkpoint_dir, names: Optional[List[str]] = None) -> int:
    """يحذف نقاط استئناف محفوظة — كل الأصول (``names=None``) أو أصول بعينها.
    يُرجع عدد الأصول المحذوفة.

    بصمة الإعدادات (:func:`_checkpoint_fingerprint`) تُبطل نقاط الاستئناف
    تلقائياً عند تغيّر ما يؤثّر في البيانات — هذه الدالة للتحكّم اليدوي
    الإضافي (مثلاً: إعادة تحميل عملة واحدة تشكّ ببياناتها الخام).
    """
    d = Path(checkpoint_dir)
    if not d.exists():
        return 0
    removed = 0
    for npz in d.glob("*.npz"):
        name = npz.stem
        if names is not None and name not in names:
            continue
        npz.unlink(missing_ok=True)
        meta = d / f"{name}.meta.json"
        if meta.exists():
            meta.unlink()
        removed += 1
    return removed


In [46]:
# @title
"""
خط أنابيب تجميع البيانات عبر كل الأصول.

مصدران للبيانات، ومنطق تجميع واحد مشترك:

  * :func:`build_dataset_from_loader`    — تحميل كل عملة من ملف خام ثم resample.
  * :func:`build_dataset_from_preloaded` — بيانات مُجهَّزة مسبقاً ``{asset: {tf: df}}``.

كلاهما يُفوّض المنطق الفعلي إلى :func:`prepare_single_asset` فلا يوجد أي
تكرار أو اختلاف في السلوك بينهما.
"""

SKIP_LABELS = {
    "name_not_in_data": "اسم غير مطابق",
    "missing_tf": "فريم مفقود",
    "insufficient_length": "بيانات غير كافية",
    "no_valid_samples": "لا عينات بعد المحاذاة",
    "load_failed": "فشل التحميل",
    "excluded": "مستبعدة يدوياً",
}

#: كل كم عملة نُجري تنظيف ذاكرة (gc مكلف، فلا يُستدعى لكل عملة).
GC_EVERY = 10


class _AssetResult:
    """نتيجة معالجة عملة واحدة — تُمرَّر من الخيط العامل إلى الخيط الرئيسي."""

    __slots__ = ('name', 'status', 'payload', 'note', 'n_candles')

    def __init__(self, name: str, status: str, payload=None,
                 note: str = '', n_candles: int = 0):
        self.name = name
        self.status = status
        self.payload = payload
        self.note = note
        self.n_candles = n_candles


def _consume(acc: '_Accumulator', process_fn, configs: List[Dict],
             workers: int, prefetch: Optional[int], tf_order: List[str]) -> None:
    """يُشغّل ``process_fn`` على كل العملات (متوازياً) ويُجمّع النتائج بالترتيب.

    التجميع يبقى في الخيط الرئيسي وحده، فلا يحتاج ``_Accumulator`` أي أقفال،
    ويظل ترتيب الصفوف مطابقاً لترتيب ``configs`` في كل تشغيل.
    """
    total = len(configs)
    for i, res in enumerate(
            imap_ordered(process_fn, configs, workers, prefetch), start=1):
        if res.status != "ok":
            acc.skip(res.status, res.name)
            print(f"   [{i}/{total}] ⚠️ {res.name}: "
                  f"{SKIP_LABELS.get(res.status, res.status)}"
                  f"{f' — {res.note}' if res.note else ''}")
            continue

        n = acc.add(res.name, *res.payload)
        if n:
            print(f"   [{i}/{total}] ✅ {res.name}: {res.n_candles:,} شمعة "
                  f"→ {n:,} نافذة")
        else:
            print(f"   [{i}/{total}] ⚠️ {res.name}: لا عيّنات صالحة بعد المحاذاة")

        res.payload = None                 # حرّر مراجع المصفوفات فوراً
        if i % GC_EVERY == 0:
            gc.collect()
    gc.collect()


class _Accumulator:
    """يُجمِّع مخرجات كل الأصول ثم يدمجها في قاموس مجموعة بيانات واحد."""

    def __init__(self, tf_order: List[str], target_heads: List[str]):
        self.tf_order = tf_order
        self.target_heads = target_heads
        self.X = {tf: [] for tf in tf_order}
        self.y = {h: [] for h in target_heads}
        self.bases: list = []
        self.last: list = []
        self.bounds: List[Dict] = []
        self.offset = 0
        self.skips = {k: 0 for k in SKIP_LABELS}
        #: أسماء العملات المتخطّاة لكل سبب — تُتيح نسخها مباشرة إلى excluded_coins.
        self.skipped: Dict[str, List[str]] = {k: [] for k in SKIP_LABELS}

    def skip(self, reason: str, name: Optional[str] = None) -> None:
        self.skips[reason] = self.skips.get(reason, 0) + 1
        if name is not None:
            self.skipped.setdefault(reason, []).append(name)

    def add(self, name: str, X_tf, y_t, bases, last) -> int:
        n = bases.shape[0]
        if n == 0:
            self.skip("no_valid_samples", name)
            return 0
        self.bounds.append({'name': name, 'start': self.offset, 'end': self.offset + n})
        self.offset += n
        for tf in self.tf_order:
            self.X[tf].append(X_tf[tf])
        for h in self.target_heads:
            self.y[h].append(y_t[h])
        self.bases.append(bases)
        self.last.append(last)
        return n

    def report(self, requested: int) -> None:
        print(f"\n\n📋 ملخص: {len(self.bounds)} عملة صالحة من أصل {requested} مطلوبة")
        if sum(self.skips.values()):
            parts = [f"{SKIP_LABELS[k]}={v}" for k, v in self.skips.items() if v]
            print(f"   أسباب التخطي: {' | '.join(parts)}")
        failed = self.skipped.get('load_failed') or []
        if failed:
            # ملف غير موجود عادةً (أسهم/سلع/عملات لم تُنزَّل) — لا تُجدي إعادة المحاولة.
            print(f"   💡 تعذّر تحميل {len(failed)} عملة. لتخطّيها مباشرة في التشغيل القادم:\n"
                  f"      update_config(excluded_coins={failed})")

    def finalize(self, config: dict, scaler_type: str, base_tf: str,
                 features: List[str]) -> Dict:
        if not self.bases:
            raise ValueError(
                "❌ لم تُجمَّع أي عملة بنجاح — راجع 'أسباب التخطي' أعلاه.\n"
                "   الاحتمال الأكثر شيوعاً: أسماء configs لا تطابق مفاتيح data "
                "(استخدم diagnose_data_vs_configs للمقارنة)."
            )

        print("\n🔗 دمج...", end=" ")
        X_final = {tf: np.concatenate(self.X[tf], axis=0)
                   for tf in self.tf_order if self.X[tf]}
        y_final = {h: np.concatenate(self.y[h], axis=0)
                   for h in self.target_heads if self.y[h]}
        bases_final = np.concatenate(self.bases, axis=0)
        last_final = np.concatenate(self.last, axis=0)
        self.X, self.y, self.bases, self.last = {}, {}, [], []
        gc.collect()

        print(f"✅ {len(bases_final):,} sequences")
        for tf in self.tf_order:
            print(f"   X_{tf}: {X_final[tf].shape}")
        for h in self.target_heads:
            print(f"   y_{h}: {y_final[h].shape}")

        out: Dict = {
            'base_params': bases_final,
            'last_candles': last_final,
            'mode': f"{config.get('target_mode', 'direction')}_{scaler_type}",
            'scaler_type': scaler_type,
            'feature_order': features,
            'timeframes': self.tf_order,
            'targets': self.target_heads,      # أسماء الرؤوس المُفعَّلة
            'price_targets': list(config['targets']),
            'asset_bounds': self.bounds,
            'base_tf': base_tf,
            'window_sizes': dict(config['window_sizes']),
            'stride': config['stride'],
            'forecast_horizon': config['forecast_horizon'],
            # أسماء المتخطّاة لكل سبب (فارغة الأسباب محذوفة) — للتدقيق وللنسخ.
            'skipped_assets': {k: list(v) for k, v in self.skipped.items() if v},
        }
        for tf in self.tf_order:
            out[f'X_{tf}'] = X_final[tf]
        for h in self.target_heads:
            out[f'y_{h}'] = y_final[h]
        return out


def _apply_exclusions(configs: List[Dict], exclude, config: dict,
                     acc: '_Accumulator') -> List[Dict]:
    """يُطبّق الاستبعاد اليدوي ويُسجّله في المُجمِّع، ويُرجع configs المتبقّية."""
    kept, removed, not_found = split_excluded_coins(configs, exclude, config)
    for name in removed:
        acc.skip("excluded", name)
    if removed:
        print(f"🚫 مستبعدة يدوياً ({len(removed)}): {removed[:8]}"
              f"{'...' if len(removed) > 8 else ''}")
    if not_found:
        print(f"   ⚠️ أسماء استبعاد غير موجودة في configs — تحقّق من الإملاء: "
              f"{not_found[:8]}{'...' if len(not_found) > 8 else ''}")
    return kept


def _resolve(config: Optional[dict], tf_order, window_sizes, targets,
             forecast_horizon, scaler_type, stride, base_tf):
    config = CONFIG if config is None else config
    tf_order = config["tf_order"] if tf_order is None else tf_order
    window_sizes = config["window_sizes"] if window_sizes is None else window_sizes
    targets = config["targets"] if targets is None else targets
    forecast_horizon = (config["forecast_horizon"]
                        if forecast_horizon is None else forecast_horizon)
    scaler_type = config["scaler_type"] if scaler_type is None else scaler_type
    stride = config["stride"] if stride is None else stride
    base_tf = base_tf or config.get("base_tf") or tf_order[0]
    return (config, tf_order, window_sizes, targets, forecast_horizon,
            scaler_type, stride, base_tf)


# ══════════════════════════════════════════════════════════════════════════
# المصدر 1: ملفات خام + resample
# ══════════════════════════════════════════════════════════════════════════
def build_dataset_from_loader(
    configs: List[Dict],
    load_asset_fn: Callable,
    resample_fn: Callable,
    tf_order: Optional[List[str]] = None,
    window_sizes: Optional[Dict[str, int]] = None,
    targets: Optional[List[str]] = None,
    forecast_horizon: Optional[int] = None,
    scaler_type: Optional[str] = None,
    stride: Optional[int] = None,
    base_tf: Optional[str] = None,
    tail: int = 0,
    max_workers: Optional[int] = None,
    prefetch: Optional[int] = None,
    exclude: Optional[Iterable[str]] = None,
    checkpoint_dir: Optional[str] = None,
    config: Optional[dict] = None,
) -> Dict:
    """بناء مجموعة البيانات بتحميل كل عملة من مصدرها الخام — **متوازياً**.

    التنزيل والمؤشرات وبناء النوافذ تجري كلها داخل خيط العامل، فيتداخل حساب
    عملة مع تنزيل أخرى بدل الانتظار عاطلين على الشبكة.

    Args:
        configs: ``[{'name': 'BTCUSDT', 'file_id': '...'}, ...]``
        load_asset_fn: ``callable(file_id, name) -> DataFrame`` بأعمدة OHLCV.
        resample_fn: ``callable(df, tf_order) -> {tf: DataFrame}`` — يجب أن
            تستدعي ``add_features`` على كل فريم.
        tail: إن كان > 0، يُقتصر على آخر ``tail`` شمعة من كل عملة (للتجارب السريعة).
        max_workers: عدد الخيوط. ``None`` يختار تلقائياً، و``1`` يُعيد السلوك
            التسلسلي القديم بالضبط.
        prefetch: أقصى عدد عملات قيد المعالجة معاً — يحدّ الذاكرة على Colab
            (افتراضياً ``max_workers * 2``).
        exclude: أسماء عملات تُتخطّى قبل التحميل (``None`` → ``CONFIG['excluded_coins']``).
            تظهر في ``dataset['skipped_assets']['excluded']``.
        checkpoint_dir: مسار (على Drive عادة) لحفظ نتيجة كل عملة فور معالجتها،
            واستئناف ما سبق حفظه بدل إعادة معالجته — يحمي من فقدان عمل ساعات
            عند انقطاع الجلسة. ``None`` (الافتراضي) يُعطّل هذا كلياً (لا تغيير
            في السلوك). راجع :func:`clear_checkpoint` للتحكّم اليدوي في نقاط
            الاستئناف، وتحذير التقادم في رأس هذا القسم.

    الناتج **مطابق** للوضع التسلسلي مهما تغيّر عدد الخيوط: الترتيب محفوظ —
    بما في ذلك عند استئناف جزئي (أصل مُستأنَف من القرص أو مُعالَج لتوّه، كلاهما
    يمرّ عبر نفس ``_process``/``imap_ordered`` بترتيب ``configs`` نفسه).
    """
    (config, tf_order, window_sizes, targets, forecast_horizon,
     scaler_type, stride, base_tf) = _resolve(
        config, tf_order, window_sizes, targets, forecast_horizon,
        scaler_type, stride, base_tf)

    # ✅ تُشتق مرة واحدة **قبل** إطلاق الخيوط: لو تُركت لأول عامل يطلبها لتسابقت
    #    الخيوط على تعبئة config['feature_order'] معاً.
    features = feature_order(config)
    target_heads = get_target_heads(targets, config)
    acc = _Accumulator(tf_order, target_heads)
    requested = len(configs)
    configs = _apply_exclusions(configs, exclude, config, acc)
    min_len = window_sizes[tf_order[0]]
    workers = default_workers(len(configs)) if max_workers is None else max_workers
    ref_symbol = (config.get('market_context') or {}).get('reference_symbol', 'BTCUSDT')
    market_dfs = build_market_context(load_asset_fn=load_asset_fn, resample_fn=resample_fn,
                                      config=config)
    ckpt_fp = _checkpoint_fingerprint(config, features, tail) if checkpoint_dir is not None else None

    def _process(cfg: Dict) -> _AssetResult:
        """السلسلة الكاملة لعملة واحدة — تعمل داخل خيط عامل.

        نضع التنزيل والمؤشرات وبناء النوافذ معاً في العامل، فيتداخل حساب عملة
        مع تنزيل أخرى بدل انتظار الشبكة عاطلين.
        """
        name = cfg['name']

        # ✅ مسار الاستئناف: تحقّق أولاً من نقطة حفظ صالحة (نفس البصمة) قبل
        # أي شبكة أو حساب — إن وُجدت، هذه العملة "تنجح" فوراً بلا تنزيل.
        if checkpoint_dir is not None:
            loaded = _load_asset_checkpoint(checkpoint_dir, name, ckpt_fp)
            if loaded is not None:
                n = int(loaded[2].shape[0])          # bases.shape[0] = عدد النوافذ
                return _AssetResult(name, "ok", payload=loaded, n_candles=n)

        try:
            df = load_asset_fn(cfg.get('file_id'), name)
        except Exception as exc:                        # noqa: BLE001
            return _AssetResult(name, "load_failed", note=str(exc)[:80])

        if tail:
            df = df[-tail:]
        if len(df) < min_len * 2:
            return _AssetResult(name, "insufficient_length",
                                note=f"{len(df):,} صف فقط")

        dfs = resample_fn(df, tf_order)
        missing = [tf for tf in tf_order if tf not in dfs]
        if missing:
            return _AssetResult(name, "missing_tf", note=f"مفقود: {missing}")
        dfs = add_market_context(dfs, market_dfs, name == ref_symbol, config)
        dfs = add_funding_oi_features(dfs, name, config)
        n_candles = len(dfs[tf_order[0]])
        if n_candles < min_len * 1.5:
            return _AssetResult(name, "insufficient_length",
                                note=f"{n_candles:,} شمعة فقط")

        payload = prepare_single_asset(
            dfs=dfs, tf_order=tf_order, window_sizes=window_sizes, targets=targets,
            forecast_horizon=forecast_horizon, scaler_type=scaler_type,
            base_tf=base_tf, stride=stride, config=config)

        if checkpoint_dir is not None:
            try:
                _save_asset_checkpoint(checkpoint_dir, name, *payload, fingerprint=ckpt_fp)
            except Exception as exc:                    # noqa: BLE001
                print(f"   ⚠️ [{name}] تعذّر حفظ نقطة استئناف ({exc}) — "
                      f"ستُعاد معالجتها عند أي استئناف لاحق.")
        return _AssetResult(name, "ok", payload=payload, n_candles=n_candles)

    print(f"⚙️ معالجة {len(configs)} عملة "
          f"({'تسلسلياً' if workers <= 1 else f'بـ {workers} خيطاً متوازياً'})...")
    _consume(acc, _process, configs, workers, prefetch, tf_order)

    acc.report(requested)
    return acc.finalize(config, scaler_type, base_tf, features)


# ══════════════════════════════════════════════════════════════════════════
# المصدر 2: بيانات مُجهَّزة مسبقاً {asset: {tf: DataFrame}}
# ══════════════════════════════════════════════════════════════════════════
def build_dataset_from_preloaded(
    configs: List[Dict],
    data: Dict[str, Dict[str, pd.DataFrame]],
    tf_order: Optional[List[str]] = None,
    window_sizes: Optional[Dict[str, int]] = None,
    targets: Optional[List[str]] = None,
    forecast_horizon: Optional[int] = None,
    scaler_type: Optional[str] = None,
    stride: Optional[int] = None,
    base_tf: Optional[str] = None,
    max_workers: Optional[int] = None,
    prefetch: Optional[int] = None,
    exclude: Optional[Iterable[str]] = None,
    config: Optional[dict] = None,
) -> Dict:
    """بناء مجموعة البيانات من قاموس مُحمَّل مسبقاً — **متوازياً**.

    لا شبكة هنا، لكن ``prepare_single_asset`` حسابات numpy ثقيلة تُحرِّر GIL،
    فالتوازي يفيد أيضاً. ``max_workers=1`` يُعيد السلوك التسلسلي.
    """
    if isinstance(data, dict) and 'base_params' in data:
        # خطأ شائع: تمرير مخرج build_dataset (مجموعة بيانات جاهزة) هنا بدل
        # {asset: {tf: DataFrame}} — يفشل لاحقاً بـ AttributeError غامض على
        # tf_dict.keys() لأن القيمة تكون مصفوفة numpy لا قاموس فريمات.
        raise TypeError(
            "data تحمل 'base_params' — تبدو مجموعة بيانات جاهزة (مخرج "
            "build_dataset) لا بيانات خام {asset: {tf: DataFrame}}. "
            "استخدمها مباشرة مع split_data() بدل تمريرها هنا.")

    (config, tf_order, window_sizes, targets, forecast_horizon,
     scaler_type, stride, base_tf) = _resolve(
        config, tf_order, window_sizes, targets, forecast_horizon,
        scaler_type, stride, base_tf)

    features = feature_order(config)
    target_heads = get_target_heads(targets, config)
    acc = _Accumulator(tf_order, target_heads)
    requested = len(configs)
    configs = _apply_exclusions(configs, exclude, config, acc)
    min_len = window_sizes[tf_order[0]]

    workers = default_workers(len(configs)) if max_workers is None else max_workers
    # ✅ نُبقي أعمدة الأهداف مع الميزات: prepare_single_asset يحتاج
    #    high/low/close الخام حتى لو كانت مستبعَدة من مدخلات النموذج.
    keep = list(features) + [c for c in TARGET_COLUMNS if c not in features]
    ref_symbol = (config.get('market_context') or {}).get('reference_symbol', 'BTCUSDT')
    market_dfs = build_market_context(data=data, config=config)

    def _process(cfg: Dict) -> _AssetResult:
        name = cfg['name'] if isinstance(cfg, dict) else str(cfg)
        if name not in data:
            return _AssetResult(name, "name_not_in_data",
                                note="راجع diagnose_data_vs_configs")

        tf_dict = data[name]
        missing = [tf for tf in tf_order if tf not in tf_dict]
        if missing:
            return _AssetResult(name, "missing_tf",
                                note=f"مفقود {missing}، متوفر {list(tf_dict)}")

        raw_dfs = {tf: tf_dict[tf] for tf in tf_order}
        raw_dfs = add_market_context(raw_dfs, market_dfs, name == ref_symbol, config)
        raw_dfs = add_funding_oi_features(raw_dfs, name, config)
        dfs = {tf: extract_features(raw_dfs[tf], keep) for tf in tf_order}
        n_candles = len(dfs[tf_order[0]])
        if n_candles < min_len * 1.5:
            return _AssetResult(name, "insufficient_length",
                                note=f"{n_candles:,} شمعة فقط")

        payload = prepare_single_asset(
            dfs=dfs, tf_order=tf_order, window_sizes=window_sizes, targets=targets,
            forecast_horizon=forecast_horizon, scaler_type=scaler_type,
            base_tf=base_tf, stride=stride, config=config)
        return _AssetResult(name, "ok", payload=payload, n_candles=n_candles)

    print(f"⚙️ معالجة {len(configs)} عملة "
          f"({'تسلسلياً' if workers <= 1 else f'بـ {workers} خيطاً متوازياً'})...")
    _consume(acc, _process, configs, workers, prefetch, tf_order)

    acc.report(requested)
    return acc.finalize(config, scaler_type, base_tf, features)


def build_dataset(configs: List[Dict], *,
                  data: Optional[Dict] = None,
                  load_asset_fn: Optional[Callable] = None,
                  resample_fn: Optional[Callable] = None,
                  **kwargs) -> Dict:
    """واجهة موحّدة: تختار المصدر تلقائياً حسب ما يُمرَّر.

    مرّر ``data`` للبيانات المُجهَّزة مسبقاً، أو ``load_asset_fn`` + ``resample_fn``
    للتحميل من الملفات الخام (هنا: ``load_asset`` + ``make_resample_fn(CONFIG)``).
    """
    if data is not None:
        return build_dataset_from_preloaded(configs, data, **kwargs)
    if load_asset_fn is not None and resample_fn is not None:
        return build_dataset_from_loader(configs, load_asset_fn, resample_fn, **kwargs)
    raise ValueError(
        "حدّد مصدر البيانات: إمّا data={asset: {tf: df}} أو "
        "load_asset_fn + resample_fn.")


#: اسم متوافق مع الدفاتر القديمة.
prepare_regression_multiframe_multiasset = build_dataset_from_loader


## 15-ب) تطبيع مقطعي عبر الأصول (اختياري — بروح Qlib CSZScoreNorm/CSRankNorm)


In [47]:
# @title
"""
تطبيع مقطعي (cross-sectional) لأهداف الانحدار عبر الأصول — بروح معالجات
Qlib ``CSZScoreNorm``/``CSRankNorm`` على حقل ``label`` (ليست محاكاة حرفية
لكودها، بل نفس الفكرة مُعاد بناؤها هنا).

**الفرق عن كل تطبيع آخر في هذا الدفتر:** التطبيع في خلايا التطبيع (٩) وخلية
النوافذ (١١) **زمني** — كل عيّنة تُطبَّع بإحصاء *تاريخها هي* (نافذتها أو آخر
إغلاقها). هذا التطبيع **مقطعي**: يجمع عيّنات *كل الأصول* التي تشارك نفس
الطابع الزمني، ويقيس كل عيّنة نسبة إلى توزيع عوائد *أقرانها في تلك اللحظة*.

فرق عملي: عملة بتقلّب يومي 1% وأخرى بتقلّب 8% — عائد خام 2% يعني حركة قوية
جداً للأولى وعادية للثانية، لكن العائد الخام وحده (reg_target_mode='return')
لا يميّز بينهما. التطبيع المقطعي يقيس كلتيهما نسبة إلى توزيع العوائد الفعلي
في نفس اليوم عبر كل الأصول، فتصير "1.2" تعني «أفضل من الأقران بمقدار 1.2
انحراف معياري ذلك اليوم» بصرف النظر عن مستوى تقلّب العملة نفسها.
"""


def cross_sectional_normalize(dataset: Dict, heads: Optional[List[str]] = None,
                              method: Optional[str] = None,
                              min_assets: Optional[int] = None,
                              clip: Optional[float] = None,
                              config: Optional[dict] = None,
                              verbose: bool = True) -> Dict:
    """يُطبِّع رؤوس انحدار عائدية مقطعياً عبر الأصول عند كل لحظة زمنية.

    ⚠️ **تغيير دلالي، لا مجرد تحسين مقياس:** الرأس بعد هذا التطبيع يتنبأ بأداء
    *نسبي* (تفوّق/تأخّر عن الأقران في نفس اللحظة) لا بعائد *مطلق* — مناسب
    لاستراتيجية اختيار الأفضل بين عدة عملات في نفس اللحظة، لا للسؤال المطلق
    "كم سيتحرّك BTC غداً؟". للتفسير المطلق استخدم عوائد ``reg_target_mode=
    'return'`` وحدها بلا هذه الخطوة.

    ⚠️ **غير متاح حرفياً في التنبؤ الحيّ:** حتى مع ``method='zscore'``
    (القابل للعكس تاريخياً عبر :func:`invert_cross_sectional`)، عكس تنبؤ
    *مستقبلي* إلى عائد مطلق يحتاج متوسط/انحراف عوائد الأقران في تلك اللحظة —
    وهي غير معروفة قبل إغلاقها. هذا تحديداً سبب كون الرأس بعد هذا التطبيع
    إشارة ترتيب حيّة، لا توقعاً مطلقاً حيّاً.

    Args:
        heads: رؤوس الانحدار المطلوب تطبيعها (افتراضياً كل رؤوس reg المُفعَّلة).
        method: ``'zscore'`` (افتراضي، قابل للعكس عبر :func:`invert_cross_sectional`)
            أو ``'rank'`` (رتبة موحّدة إلى ``[-1, 1]``؛ أقوى أمام قيمة شاذة
            واحدة داخل اليوم، لكن بلا عكس دقيق لعائد مطلق — ترتيبي بالتصميم).
        min_assets: أقل عدد أصول عند نفس اللحظة ليُحسب التطبيع؛ دون ذلك يبقى
            العائد الخام كما هو — مجموعة أقران أصغر من هذا لا تعطي إحصاءً
            موثوقاً (انحراف معياري على عيّنتين أو ثلاث ضجيج بحت لا إشارة).
        clip: قصّ بعد ``'zscore'`` فقط (``'rank'`` محصورة أصلاً في ``[-1, 1]``).

    Returns:
        نسخة من ``dataset`` (**لا يُعدَّل الأصل**): رؤوس ``heads`` مُستبدَلة
        بالقيم المُطبَّعة، مع ``dataset['cs_norm_stats']`` — لكل رأس: الطريقة
        والحدّ الأدنى والقصّ، وإحصاء كل لحظة طُبِّعت (لازم لعكس ``'zscore'``
        عبر :func:`invert_cross_sectional`؛ اللحظات التي لم تُطبَّع — أصول
        أقل من ``min_assets`` — غائبة عن الإحصاء عمداً، فتبقى عائداً خاماً
        عند العكس أيضاً).

    يفترض ``reg_target_mode='return'`` (العائد المباشر) — تطبيقه على
    ``'window_scale'`` يخلط مرجعين مختلفي المعنى (IQR نافذة الأصل الواحد +
    توزيع الأقران في نفس اللحظة) فيُرفَض بخطأ صريح بدل نتيجة مضلِّلة بصمت.
    """
    config = CONFIG if config is None else config
    if config.get('reg_target_mode', 'return') != 'return':
        raise ValueError(
            "cross_sectional_normalize يتطلب reg_target_mode='return' — "
            "تطبيقه على 'window_scale' يخلط مرجع IQR النافذة بتوزيع الأقران.")

    cs_cfg = config.get('cs_norm', {}) or {}
    heads = list(heads) if heads is not None else get_reg_heads(config)
    method = method or cs_cfg.get('method', 'zscore')
    min_assets = int(min_assets if min_assets is not None else cs_cfg.get('min_assets', 5))
    clip = float(clip if clip is not None else cs_cfg.get('clip', 5.0))
    if method not in ('zscore', 'rank'):
        raise ValueError(f"method غير معروفة: '{method}' (المتاح: zscore, rank)")

    heads = [h for h in heads if h in dataset]
    out = dict(dataset)
    if not heads or 'last_candles' not in dataset or len(dataset['last_candles']) == 0:
        out['cs_norm_stats'] = dict(dataset.get('cs_norm_stats', {}))
        return out

    ts_all = np.asarray(dataset['last_candles'])[:, TS_COL].astype('int64')
    order = np.argsort(ts_all, kind='stable')
    ts_sorted = ts_all[order]
    uniq_ts, start_pos, counts = np.unique(ts_sorted, return_index=True, return_counts=True)

    all_stats: Dict[str, dict] = dict(dataset.get('cs_norm_stats', {}))
    for h in heads:
        raw = np.asarray(dataset[h], dtype='float64')
        y_sorted = raw[order]
        normed = y_sorted.copy()
        by_ts: Dict[int, Tuple[float, float, int]] = {}
        n_small = 0
        for t, pos, cnt in zip(uniq_ts, start_pos, counts):
            sl = slice(int(pos), int(pos) + int(cnt))
            grp = y_sorted[sl]
            if cnt < min_assets:
                n_small += cnt
                continue
            if method == 'rank':
                ranks = grp.argsort(kind='stable').argsort(kind='stable').astype('float64')
                denom = max(cnt - 1, 1)
                normed[sl] = (ranks / denom) * 2.0 - 1.0        # [-1, 1]، بلا حاجة لقصّ
            else:
                mu, sd = float(grp.mean()), float(grp.std())
                if sd < EPS:
                    normed[sl] = 0.0
                else:
                    normed[sl] = np.clip((grp - mu) / sd, -clip, clip)
                by_ts[int(t)] = (mu, sd, int(cnt))
        result = np.empty_like(raw)
        result[order] = normed
        out[h] = result.astype('float32')
        all_stats[h] = {'method': method, 'min_assets': min_assets, 'clip': clip, 'by_ts': by_ts}
        if verbose:
            total = len(raw)
            print(f"📐 [{h}] تطبيع مقطعي ({method}): "
                  f"{total - n_small:,}/{total:,} عيّنة طُبِّعت نسبة لأقرانها "
                  f"(الحدّ الأدنى {min_assets} أصلاً) | {n_small:,} بقيت عائداً خاماً "
                  f"(لحظات بأصول أقل من الحدّ)")
    out['cs_norm_stats'] = all_stats
    return out


def invert_cross_sectional(preds: np.ndarray, timestamps: np.ndarray, head: str,
                           dataset: Dict) -> np.ndarray:
    """يعكس تنبؤات مُطبَّعة مقطعياً (``method='zscore'`` فقط) إلى عائد خام.

    يستخدم إحصاء *نفس اللحظات* المحفوظ في ``dataset['cs_norm_stats'][head]``
    عند بناء بيانات التدريب — مناسب لتقييم نموذج على بيانات تاريخية (الإحصاء
    معروف بأثر رجعي)، **لا للتنبؤ الحيّ**: عائد الأقران الفعلي "اليوم" غير
    معروف قبل إغلاق اليوم نفسه — راجع تنبيه :func:`cross_sectional_normalize`.

    لحظة لم تُطبَّع أصلاً (أصول أقل من ``min_assets`` عند البناء) تُعاد كما
    هي (كانت عائداً خاماً بلا تطبيع، فتبقى كذلك). ``method='rank'`` ترتيبي
    بلا عكس دقيق فيُرفَض بخطأ صريح.
    """
    stats = dataset.get('cs_norm_stats', {}).get(head)
    if stats is None:
        raise ValueError(f"لا إحصاء تطبيع مقطعي محفوظ للرأس '{head}' — "
                         "طبّق cross_sectional_normalize أولاً.")
    if stats['method'] != 'zscore':
        raise ValueError(
            f"عكس دقيق متاح فقط لـ method='zscore' (المحفوظ: '{stats['method']}'). "
            "'rank' ترتيبي بالتصميم، بلا معنى لعكسه إلى عائد مطلق.")
    by_ts = stats['by_ts']
    preds = np.asarray(preds, dtype='float64')
    ts = np.asarray(timestamps).astype('int64')
    out = preds.copy()
    for i, t in enumerate(ts):
        rec = by_ts.get(int(t))
        if rec is not None:
            mu, sd, _ = rec
            out[i] = preds[i] * sd + mu
    return out


## 16) التقسيم الزمني train/val/test (`split.py` سابقاً)


In [48]:
# @title
"""
تقسيم زمني (بدون خلط) مع فجوة عزل (embargo).

وضعان:

``global_time`` (الافتراضي، المُوصى به)
    حدود زمنية **مطلقة مشتركة** بين كل العملات: كل عيّنة تاريخها قبل
    ``train_end`` تدخل التدريب، ومهما اختلفت أطوال تواريخ العملات يبقى
    ``test`` لاحقاً زمنياً لكل ما رآه النموذج.

``per_asset`` (السلوك القديم)
    كل عملة تُقسَّم 90/5/5 على مدى تاريخها هي.
    ⚠️ **يُسرِّب** حين تختلف أطوال تواريخ العملات: عملة مُدرجة في 2024 يقع
    تدريبها كله بعد فترة اختبار البيتكوين، فيتدرّب النموذج على مستقبلٍ
    بالنسبة لبعض عيّنات الاختبار. مُتاح للتوافق مع نتائج قديمة فقط.

في الوضعين تُطبَّق فجوة عزل، لأن النوافذ متراكبة بشدة (نافذة كبيرة، stride
صغير): بدونها تتشارك آخر عيّنة تدريب وأول عيّنة تحقق معظم شموعهما.
الفجوة الافتراضية = ``window_size(base_tf) + forecast_horizon`` شمعة.

**حدّا التقسيم في ``global_time`` يُشتقّان من العيّنات المُبقاة لا من كل العيّنات.**
فجوة العزل تُلقي عيّنات (تُهمَل) بين كل قسمين. لو اشتُقّ الحدّ الزمني من نسبة
90/5/5 على *كل* العيّنات، لقصر الزمن المتبقي بعد train عن فجوتين حين تتكدّس
العيّنات في الفترة الأخيرة (عملات كثيرة حديثة الإدراج): فيخرج val و test فارغين
— وهذا ما حدث فعلاً على بيانات ٤٢٢ عملة (train_end قبل النهاية بـ ٦٤ يوماً،
وفجوة العزل ٣٣). الآن يُبحث عن الحدّين بحيث تقترب **نِسَب العيّنات المُبقاة**
من المطلوب، وإن استحال (زمن لا يكفي فجوتين + الحدّ الأدنى للعيّنات) يُرفع
خطأ صريح بدل قسم فارغ. البديل المرفوض: تقليص فجوة العزل حتى يتّسع الزمن — يُعيد
التسرّب الذي وُجدت الفجوة لمنعه.
"""

#: بادئة مفاتيح الأهداف داخل أقسام التدريب — يتوقّعها Keras لمطابقة أسماء المخرجات.
Y_PREFIX = 'y_'

#: أوضاع التقسيم المدعومة.
SPLIT_MODES = ('global_time', 'per_asset')


def embargo_candles(data: Dict, config: Optional[dict] = None) -> int:
    """طول فجوة العزل **بالشموع** — الأساس الذي تُشتق منه بقية الصيغ."""
    config = CONFIG if config is None else config
    candles = config.get('embargo_candles')
    if candles is None:
        base_tf = data.get('base_tf', config['base_tf'])
        window = data.get('window_sizes', config['window_sizes'])[base_tf]
        candles = window + data.get('forecast_horizon', config['forecast_horizon'])
    return int(candles)


def embargo_size(data: Dict, config: Optional[dict] = None) -> int:
    """فجوة العزل بعدد **العيّنات** (تُستخدم في وضع ``per_asset``)."""
    config = CONFIG if config is None else config
    stride = max(data.get('stride', config.get('stride', 1)), 1)
    return max(1, int(math.ceil(embargo_candles(data, config) / stride)))


def _tf_to_timedelta(tf: str) -> pd.Timedelta:
    """``'4h'`` → ``Timedelta('4h')``. يقبل صيغ pandas المعتادة (``1D``, ``15T``...)."""
    return pd.Timedelta(tf)


def embargo_duration(data: Dict, config: Optional[dict] = None) -> pd.Timedelta:
    """فجوة العزل كمدة **زمنية** (تُستخدم في وضع ``global_time``)."""
    config = CONFIG if config is None else config
    base_tf = data.get('base_tf', config['base_tf'])
    return embargo_candles(data, config) * _tf_to_timedelta(base_tf)


def sample_timestamps(split_or_data: Dict) -> pd.DatetimeIndex:
    """الطابع الزمني لكل عيّنة، مأخوذاً من عمود ``timestamp`` في ``last_candles``."""
    ts = np.asarray(split_or_data['last_candles'])[:, TS_COL]
    return pd.to_datetime(ts.astype('int64'), utc=True)


def _new_accumulator(timeframes, targets) -> dict:
    return {'X': {tf: [] for tf in timeframes},
            'y': {t: [] for t in targets},
            'base_params': [], 'last_candles': []}


def _append_range(data, s, e, acc, timeframes, targets) -> None:
    if e <= s:
        return
    acc['base_params'].append(data['base_params'][s:e])
    acc['last_candles'].append(data['last_candles'][s:e])
    for tf in timeframes:
        acc['X'][tf].append(data[f'X_{tf}'][s:e])
    for t in targets:
        acc['y'][t].append(data[f'y_{t}'][s:e])


def _finalize_accumulator(acc, data, timeframes, targets) -> dict:
    out = {
        'base_params': (np.concatenate(acc['base_params'], axis=0)
                        if acc['base_params'] else np.empty((0, 2), dtype='float32')),
        'last_candles': (np.concatenate(acc['last_candles'], axis=0)
                         if acc['last_candles']
                         else np.empty((0, data['last_candles'].shape[1]),
                                       dtype=data['last_candles'].dtype)),
    }
    out['y'] = {t: (np.concatenate(acc['y'][t], axis=0) if acc['y'][t]
                    else np.empty((0,), dtype='float32')) for t in targets}
    for tf in timeframes:
        if acc['X'][tf]:
            out[f'X_{tf}'] = np.concatenate(acc['X'][tf], axis=0)
        else:
            shape = data[f'X_{tf}'].shape
            out[f'X_{tf}'] = np.empty((0, shape[1], shape[2]), dtype='float32')
    return out


def _slice(data, s, e, timeframes, targets) -> dict:
    out = {
        'base_params': data['base_params'][s:e],
        'last_candles': data['last_candles'][s:e],
        'y': {t: data[f'y_{t}'][s:e] for t in targets},
    }
    for tf in timeframes:
        out[f'X_{tf}'] = data[f'X_{tf}'][s:e]
    return out


def add_y_prefix(split: dict) -> dict:
    """يُضيف بادئة ``y_`` لمفاتيح الأهداف إن لم تكن موجودة (في المكان).

    Keras يطابق مفاتيح ``y`` بأسماء طبقات المخرجات، وهي مُسمّاة ``y_<head>``.
    """
    y = split.get('y')
    if y and not next(iter(y)).startswith(Y_PREFIX):
        split['y'] = {f'{Y_PREFIX}{t}': arr for t, arr in y.items()}
    return split


def strip_y_prefix(y: dict) -> dict:
    """يُزيل بادئة ``y_`` — يُستخدم في التقييم لمطابقة أسماء الرؤوس."""
    return {(k[len(Y_PREFIX):] if k.startswith(Y_PREFIX) else k): v
            for k, v in y.items()}


def _take(data: Dict, mask: np.ndarray, timeframes, targets) -> dict:
    """يقتطع عيّنات مجموعة البيانات حسب قناع منطقي."""
    out = {
        'base_params': data['base_params'][mask],
        'last_candles': data['last_candles'][mask],
        'y': {t: data[f'y_{t}'][mask] for t in targets},
    }
    for tf in timeframes:
        out[f'X_{tf}'] = data[f'X_{tf}'][mask]
    return out


def _kept_counts(ordered: np.ndarray, t1, t2, gap: int):
    """أعداد العيّنات **المُبقاة** ``(train, val, test)`` لحدّين ``t1 ≤ t2`` وفجوة ``gap``.

    كلها أعداد صحيحة بالنانوثانية. ``t2`` قد يكون مصفوفة (تقييم عدة حدود معاً).
    القواعد مطابقة لأقنعة ``_split_global_time``:
    ``train: ts ≤ t1`` | ``val: t1+gap < ts ≤ t2`` | ``test: ts > t2+gap``.
    """
    n = len(ordered)
    n_tr = np.searchsorted(ordered, t1, side='right')
    a = np.searchsorted(ordered, t1 + gap, side='right')
    b = np.searchsorted(ordered, t2, side='right')
    c = np.searchsorted(ordered, t2 + gap, side='right')
    return n_tr, np.maximum(b - a, 0), n - c


def _best_val_end(ordered, uniq, t1, gap, val_share, min_n):
    """أفضل ``val_end`` لحدّ ``t1``: حصة val من (val+test) الأقرب للهدف، وكلاهما ≥ ``min_n``."""
    cand = uniq[uniq > t1 + gap]
    if cand.size == 0:
        return None
    _, n_va, n_te = _kept_counts(ordered, t1, cand, gap)
    ok = (n_va >= min_n) & (n_te >= min_n)
    if not ok.any():
        return None
    share = n_va / np.maximum(n_va + n_te, 1)
    j = int(np.argmin(np.where(ok, np.abs(share - val_share), np.inf)))
    return int(cand[j]), int(n_va[j]), int(n_te[j])


def _resolve_by_kept_share(ordered: np.ndarray, gap: int, train_pct: float,
                           val_pct: float, min_n: int):
    """يبحث عن ``(train_end, val_end)`` بحيث تقترب نِسَب **المُبقاة** من المطلوب.

    ``ordered``: طوابع العيّنات مرتّبة (int64 ns). يُرجع
    ``(t1, t2, n_train, n_val, n_test)`` أو ``None`` إن لم يوجد حدّان يُبقيان
    ``min_n`` عيّنة على الأقل في كل قسم. بحث ثنائي على ``t1`` (كل تقييم
    ``O(U log N)``)، فيبقى سريعاً حتى مع ملايين العيّنات.
    """
    uniq = np.unique(ordered)
    if uniq.size == 0:
        return None
    test_pct = max(1.0 - train_pct - val_pct, 0.0)
    val_share = val_pct / (val_pct + test_pct) if (val_pct + test_pct) > 0 else 0.5

    def ev(i):
        t1 = int(uniq[i])
        n_tr = int(np.searchsorted(ordered, t1, side='right'))
        if n_tr < min_n:
            return "low"                       # train صغير جداً: اذهب لحدّ أحدث
        r = _best_val_end(ordered, uniq, t1, gap, val_share, min_n)
        if r is None:
            return "high"                      # لا يتّسع val/test: اذهب لحدّ أقدم
        t2, n_va, n_te = r
        return (n_tr / (n_tr + n_va + n_te), t1, t2, n_tr, n_va, n_te)

    lo, hi = 0, len(uniq) - 1
    while lo < hi:                             # أقدم حدّ يُبقي train كافياً
        mid = (lo + hi) // 2
        if ev(mid) == "low":
            lo = mid + 1
        else:
            hi = mid
    i_lo = lo
    lo, hi = i_lo, len(uniq) - 1
    while lo < hi:                             # أحدث حدّ يتّسع معه val و test
        mid = (lo + hi + 1) // 2
        if ev(mid) == "high":
            hi = mid - 1
        else:
            lo = mid
    i_hi = lo
    if isinstance(ev(i_lo), str) or isinstance(ev(i_hi), str):
        return None

    lo, hi = i_lo, i_hi                        # أول حدّ تبلغ عنده حصة train الهدف
    while lo < hi:
        mid = (lo + hi) // 2
        if ev(mid)[0] >= train_pct:
            hi = mid
        else:
            lo = mid + 1
    best = ev(lo)
    if lo > i_lo:
        prev = ev(lo - 1)
        if abs(prev[0] - train_pct) < abs(best[0] - train_pct):
            best = prev
    _, t1, t2, n_tr, n_va, n_te = best
    return t1, t2, n_tr, n_va, n_te


#: طرق اشتقاق حدّ(ي) التقسيم في split_mode='global_time'.
CUTOFF_METHODS = ('kept_share', 'raw_quantile')


def compute_global_cutoff(data: Dict, pct: float) -> pd.Timestamp:
    """حدّ زمني واحد بحيث يقع بعده أقرب عدد ممكن إلى ``pct × إجمالي العيّنات``.

    "آخر N من كل العيّنات (بصرف النظر عن عملتها)" تقارب النسبة المطلوبة
    مباشرة — استيداك (percentile) على الطابع الزمني **الخام** لكل العيّنات
    مجمّعة معاً من كل الأصول، لا لكل أصل على حدة؛ وهذا جوهر منع التسرّب بين
    العملات (حدّ واحد مشترك، مطابقاً لما يفعله ``_split_global_time`` أصلاً).

    ⚠️ يستهدف نسبة العيّنات **الخام قبل** فجوة العزل، لا **المُبقاة بعدها**
    (خلافاً لـ :func:`resolve_split_dates` بطريقتها الافتراضية ``'kept_share'``،
    التي تستهدف المُبقاة فعلياً — أدقّ لكن أعقد بحثاً). الفرق بين الطريقتين
    **يتناسب مع (عيّنات فجوة العزل المفقودة ÷ حجم القسم المستهدف)** — يصغر
    كلما كبر val_pct/test_pct×الإجمالي عن خسارة العزل، ويكبر كلما صغر: حتى
    على بيانات موزَّعة **بانتظام تام** زمنياً، val_pct صغيرة (٥٪ مثلاً) مع
    فجوة عزل معتدلة قد تُنتج قسم val أصغر فعلياً من المطلوب بعشرات بالمئة —
    قِيس فعلياً (٣٣٪ نقصان على ١٠ آلاف عيّنة، ٥ أصول، فجوة ٣٣ يوماً). وعلى
    بيانات **مكدَّسة** أيضاً (عملات كثيرة حديثة الإدراج) قد يقع الحدّ في منطقة
    خفيفة العيّنات فيتفاقم الأثر أو يفشل تماماً — وهذا بالضبط ما وقع فعلياً
    على بيانات ٤٢٢ عملة حقيقية ودفع لكتابة ``'kept_share'`` أصلاً (راجع توثيق
    ``_split_global_time``). الحماية الوحيدة هنا رفض صريح
    (``_assert_split_sizes``) لا إصلاح للانحراف نفسه — استخدم ``'kept_share'``
    (الافتراضي) إن أردتَ مطابقة دقيقة للنِّسَب المطلوبة فعلياً.
    """
    ts = extract_dates(data)
    if len(ts) == 0:
        raise ValueError("لا عيّنات لحساب حدّ فاصل منها.")
    pct = float(pct)
    if not 0.0 < pct < 1.0:
        raise ValueError(f"النسبة يجب أن تقع بين 0 و1 حصرياً (وصلت {pct}).")
    q = np.quantile(np.asarray(ts.astype('int64')), 1.0 - pct)
    return pd.Timestamp(int(round(q)), tz='UTC')


#: مرادف موثَّق لـ sample_timestamps بالاسم المطلوب — نفس الدالة تماماً، بلا
#: منطق إضافي؛ لا يوجد داعٍ لنسخة ثانية من نفس الحساب.
extract_dates = sample_timestamps


def extract_test_dates(test: Dict) -> pd.DatetimeIndex:
    """طوابع زمن عيّنات قسم ``test`` — يدعم الشكلين اللذين يُرجعهما ``split_data``:
    قاموس مسطّح واحد (``keep_asset_test_separate=False``)، أو
    ``{asset: split}`` (``keep_asset_test_separate=True``، تُجمَع طوابع كل
    الأصول معاً بالترتيب الزمني)."""
    if 'last_candles' in test:
        return extract_dates(test)
    parts = [extract_dates(v) for v in test.values() if len(v.get('last_candles', []))]
    if not parts:
        return pd.DatetimeIndex([], tz='UTC')
    return parts[0].append(parts[1:]) if len(parts) > 1 else parts[0]


def resolve_split_dates(data: Dict,
                        train_pct: float,
                        val_pct: float,
                        config: Optional[dict] = None,
                        cutoff_method: Optional[str] = None,
                        ) -> Tuple[pd.Timestamp, pd.Timestamp]:
    """حدّا التقسيم الزمنيان المشتركان ``(train_end, val_end)``.

    إن حُدِّدت ``CONFIG['split_dates']`` تُستخدم كما هي؛ وإلا تُشتقّان بحسب
    ``cutoff_method`` (افتراضياً ``config['split_cutoff_method']``، وهو
    ``'kept_share'``):

    * ``'kept_share'`` — تقترب **نِسَب العيّنات المُبقاة بعد فجوتَي العزل** من
      ``train_pct``/``val_pct``. الحدّ الزمني واحد للجميع، وهذا جوهر منع
      التسرّب. إن لم يتّسع الزمن لقسمين بعد train (فجوتان + ``min_split_
      samples``) يُرفع ``ValueError`` بشرح الأسباب.
    * ``'raw_quantile'`` — حدّان مباشران عبر :func:`compute_global_cutoff`
      على نسبة العيّنات **الخام قبل** خصم فجوة العزل؛ أبسط لكن أقلّ دقّة على
      بيانات مكدَّسة زمنياً — راجع تحذير تلك الدالة. النتيجة تخضع لنفس فحص
      ``min_split_samples`` (رفض صريح، لا قسم فارغ صامت).
    """
    config = CONFIG if config is None else config
    dates = config.get('split_dates') or {}

    if dates.get('train_end') and dates.get('val_end'):
        return (pd.Timestamp(dates['train_end'], tz='UTC'),
                pd.Timestamp(dates['val_end'], tz='UTC'))

    method = cutoff_method or config.get('split_cutoff_method', 'kept_share')
    if method not in CUTOFF_METHODS:
        raise ValueError(f"split_cutoff_method غير معروفة: {method!r} — "
                         f"المتاح: {CUTOFF_METHODS}")

    if method == 'raw_quantile':
        test_pct = max(1.0 - train_pct - val_pct, 0.0)
        # حدّان مستقلّان بنفس المبدأ: train_end بحيث يقع بعده (val_pct+test_pct)
        # من كل العيّنات، وval_end بحيث يقع بعده test_pct منها فقط.
        train_end = compute_global_cutoff(data, val_pct + test_pct)
        val_end = compute_global_cutoff(data, test_pct)
        gap = embargo_duration(data, config)
        ts = sample_timestamps(data)
        sizes = {'train': int((ts <= train_end).sum()),
                'val': int(((ts > train_end + gap) & (ts <= val_end)).sum()),
                'test': int((ts > val_end + gap).sum())}
        _assert_split_sizes(sizes, config, gap, train_end, val_end)
        kept = sum(sizes.values())
        if kept and (abs(sizes['train'] / kept - train_pct) > 0.02
                     or abs(sizes['val'] / kept - val_pct) > 0.02):
            print(f"   ℹ️ النِّسَب الفعلية (raw_quantile): train={sizes['train'] / kept:.1%} | "
                  f"val={sizes['val'] / kept:.1%} | test={sizes['test'] / kept:.1%} "
                  f"(المطلوب {train_pct:.0%}/{val_pct:.0%}/{test_pct:.0%}) — فجوة العزل "
                  f"غيّرت النسب الخام؛ جرّب cutoff_method='kept_share' لمطابقة أدقّ.")
        return train_end, val_end

    # 'kept_share' (الافتراضي)
    ordered = np.sort(np.asarray(data['last_candles'])[:, TS_COL].astype('int64'))
    gap = int(embargo_duration(data, config).value)
    min_n = int(config.get('min_split_samples', 1))

    found = _resolve_by_kept_share(ordered, gap, train_pct, val_pct, min_n)
    day = 86_400 * 10 ** 9
    if found is None:
        span = (int(ordered[-1]) - int(ordered[0])) / day if len(ordered) else 0.0
        raise ValueError(
            f"❌ لا يمكن اشتقاق حدّي تقسيم يُبقيان ≥{min_n} عيّنة في train و val و test.\n"
            f"   العيّنات: {len(ordered):,} | المدى الزمني: {span:.0f} يوماً | "
            f"فجوة العزل: {gap / day:.0f} يوماً (تُفقَد مرّتين: بعد train وبعد val).\n"
            f"   الحلول: (١) حدّد CONFIG['split_dates'] يدوياً، (٢) وفّر تاريخاً أطول "
            f"أو stride أصغر، (٣) خفّض min_split_samples إن كان الحدّ مبالَغاً فيه.\n"
            f"   لا تُقلّص فجوة العزل لإنقاذ التقسيم: النوافذ المتراكبة تُسرّب الأداء.")

    t1, t2, n_tr, n_va, n_te = found
    kept = n_tr + n_va + n_te
    if abs(n_tr / kept - train_pct) > 0.02 or abs(n_va / kept - val_pct) > 0.02:
        print(f"   ℹ️ النِّسَب المُبقاة بعد العزل: train={n_tr / kept:.1%} | "
              f"val={n_va / kept:.1%} | test={n_te / kept:.1%} "
              f"(المطلوب {train_pct:.0%}/{val_pct:.0%}/{max(1 - train_pct - val_pct, 0):.0%}) — "
              f"أقرب ما يسمح به توزيع العيّنات زمنياً.")
    return pd.Timestamp(t1, tz='UTC'), pd.Timestamp(t2, tz='UTC')


def _assert_split_sizes(sizes: Dict[str, int], config: dict, gap: pd.Timedelta,
                        train_end: pd.Timestamp, val_end: pd.Timestamp) -> None:
    """يرفع خطأً صريحاً إن قلّ أي قسم عن ``min_split_samples`` (بدل قسم فارغ صامت)."""
    min_n = int(config.get('min_split_samples', 1))
    small = {k: v for k, v in sizes.items() if v < min_n}
    if small:
        raise ValueError(
            f"❌ أقسام أصغر من الحدّ الأدنى ({min_n}): {small}\n"
            f"   الحدّان: train_end={train_end:%Y-%m-%d} | val_end={val_end:%Y-%m-%d} | "
            f"عزل={gap}.\n"
            f"   إن كانا من CONFIG['split_dates'] فباعِد بينهما بما يزيد على فجوة العزل؛ "
            f"وإلا اترك split_dates=None ليُشتقّا تلقائياً.")


def _split_global_time(data, train_pct, val_pct, keep_asset_test_separate,
                       timeframes, targets, config):
    """تقسيم بحدود زمنية مطلقة مشتركة بين كل العملات."""
    ts = sample_timestamps(data)
    train_end, val_end = resolve_split_dates(data, train_pct, val_pct, config)
    gap = embargo_duration(data, config)

    train_mask = ts <= train_end
    val_mask = (ts > train_end + gap) & (ts <= val_end)
    test_mask = ts > val_end + gap

    train = _take(data, train_mask, timeframes, targets)
    val = _take(data, val_mask, timeframes, targets)

    print(f"📊 تقسيم زمني مشترك (عزل={gap}):")
    print(f"   train ≤ {train_end:%Y-%m-%d}          : {int(train_mask.sum()):,}")
    print(f"   val   ≤ {val_end:%Y-%m-%d}          : {int(val_mask.sum()):,}")
    print(f"   test  >  {(val_end + gap):%Y-%m-%d}          : {int(test_mask.sum()):,}")

    _assert_split_sizes({'train': int(train_mask.sum()), 'val': int(val_mask.sum()),
                         'test': int(test_mask.sum())}, config, gap, train_end, val_end)

    if not keep_asset_test_separate:
        test = _take(data, test_mask, timeframes, targets)
        return train, val, test

    # test مفصول لكل عملة — نتقاطع قناع الزمن مع نطاق كل عملة
    test: Dict[str, dict] = {}
    skipped: List[str] = []
    for b in data.get('asset_bounds') or []:
        asset_mask = np.zeros(len(ts), dtype=bool)
        asset_mask[b['start']:b['end']] = True
        combined = asset_mask & test_mask
        name = b.get('name', b.get('asset', f"asset_{b['start']}_{b['end']}"))
        if combined.any():
            test[name] = _take(data, combined, timeframes, targets)
        else:
            skipped.append(name)

    print(f"   test مفصول عبر {len(test)} عملة")
    if skipped:
        # عملات تاريخها ينتهي قبل حدّ الاختبار — سلوك صحيح، لكن يستحق التنبيه
        print(f"   ℹ️ {len(skipped)} عملة بلا عيّنات اختبار (تاريخها ينتهي قبل "
              f"{(val_end + gap):%Y-%m-%d}): {skipped[:6]}"
              f"{'...' if len(skipped) > 6 else ''}")
    return train, val, test


def split_data(
    data: Dict,
    train_pct: Optional[float] = None,
    val_pct: Optional[float] = None,
    keep_asset_test_separate: Optional[bool] = None,
    prefix_y: bool = True,
    mode: Optional[str] = None,
    config: Optional[dict] = None,
) -> Tuple[dict, dict, dict]:
    """تقسيم ``train / val / test`` زمنياً مع فجوة عزل.

    Args:
        mode: ``'global_time'`` (افتراضي) حدود زمنية مشتركة تمنع التسرّب بين
            العملات مختلفة الأطوال؛ ``'per_asset'`` السلوك القديم (نسب لكل عملة).
        keep_asset_test_separate: True يُرجع ``test`` كقاموس ``{asset: split}``
            (يسمح بتقييم كل عملة على حدة)، False يدمجها في قسم واحد.
        prefix_y: True يُضيف بادئة ``y_`` لمفاتيح الأهداف لتطابق أسماء مخرجات النموذج.
    """
    config = CONFIG if config is None else config
    train_pct = config['train_pct'] if train_pct is None else train_pct
    val_pct = config['val_pct'] if val_pct is None else val_pct
    if keep_asset_test_separate is None:
        keep_asset_test_separate = config['keep_asset_test_separate']
    mode = mode or config.get('split_mode', 'global_time')
    if mode not in SPLIT_MODES:
        raise ValueError(f"split_mode غير معروف: {mode!r} — المتاح: {SPLIT_MODES}")

    if mode == 'global_time':
        train, val, test = _split_global_time(
            data, train_pct, val_pct, keep_asset_test_separate,
            data['timeframes'], data['targets'], config)
        if prefix_y:
            add_y_prefix(train)
            add_y_prefix(val)
            if isinstance(test, dict) and 'y' in test:
                add_y_prefix(test)
            else:
                for asset_split in test.values():
                    add_y_prefix(asset_split)
        return train, val, test

    timeframes: List[str] = data['timeframes']
    targets: List[str] = data['targets']
    bounds = data.get('asset_bounds')
    embargo_samples = embargo_size(data, config)

    def _bounds(s, e):
        n = e - s
        t_end = s + int(n * train_pct)
        v_start = min(t_end + embargo_samples, e)
        v_end = min(v_start + int(n * val_pct), e)
        te_start = min(v_end + embargo_samples, e)
        return (s, t_end), (v_start, v_end), (te_start, e)

    if not bounds:
        n = len(data['base_params'])
        (s_tr, e_tr), (s_va, e_va), (s_te, e_te) = _bounds(0, n)
        train = _slice(data, s_tr, e_tr, timeframes, targets)
        val = _slice(data, s_va, e_va, timeframes, targets)
        test = _slice(data, s_te, e_te, timeframes, targets)
        print(f"📊 تقسيم (بدون حدود عملات) | عزل={embargo_samples} عيّنة: "
              f"Train={e_tr - s_tr:,} | Val={e_va - s_va:,} | Test={e_te - s_te:,}")
    else:
        train_acc = _new_accumulator(timeframes, targets)
        val_acc = _new_accumulator(timeframes, targets)
        test_acc = _new_accumulator(timeframes, targets)
        test_per_asset: Dict[str, dict] = {}
        asset_stats: List[str] = []

        for b in bounds:
            s, e = b['start'], b['end']
            asset_name = b.get('name', b.get('asset', f'asset_{s}_{e}'))
            if e - s <= 0:
                continue
            (s_tr, e_tr), (s_va, e_va), (s_te, e_te) = _bounds(s, e)
            _append_range(data, s_tr, e_tr, train_acc, timeframes, targets)
            _append_range(data, s_va, e_va, val_acc, timeframes, targets)

            if keep_asset_test_separate:
                if e_te > s_te:
                    asset_split = _slice(data, s_te, e_te, timeframes, targets)
                    test_per_asset[asset_name] = asset_split
                    asset_stats.append(f"{asset_name}: {len(asset_split['base_params']):,}")
            else:
                _append_range(data, s_te, e_te, test_acc, timeframes, targets)

        train = _finalize_accumulator(train_acc, data, timeframes, targets)
        val = _finalize_accumulator(val_acc, data, timeframes, targets)

        if keep_asset_test_separate:
            test = test_per_asset
            n_te = sum(len(v['base_params']) for v in test.values())
            print(f"📊 تقسيم (عزل={embargo_samples} عيّنة، test منفصل لكل عملة):")
            print(f"   Train: {len(train['base_params']):,}")
            print(f"   Val:   {len(val['base_params']):,}")
            print(f"   Test:  {n_te:,} عبر {len(test)} عملة")
            if asset_stats:
                print(f"   التفاصيل: {', '.join(asset_stats[:5])}"
                      + ("..." if len(asset_stats) > 5 else ""))
        else:
            test = _finalize_accumulator(test_acc, data, timeframes, targets)
            print(f"📊 تقسيم (عزل={embargo_samples} عيّنة): "
                  f"Train={len(train['base_params']):,} | "
                  f"Val={len(val['base_params']):,} | "
                  f"Test={len(test['base_params']):,}")

    if prefix_y:
        add_y_prefix(train)
        add_y_prefix(val)
        if isinstance(test, dict) and 'y' in test:
            add_y_prefix(test)
        else:
            for asset_split in test.values():
                add_y_prefix(asset_split)

    return train, val, test


def build_leak_free_split(data: Dict,
                          train_pct: Optional[float] = None,
                          val_pct: Optional[float] = None,
                          keep_asset_test_separate: Optional[bool] = None,
                          config: Optional[dict] = None) -> Tuple[dict, dict, dict]:
    """يُعيد بناء ``train/val/test`` بحدّ فاصل **واحد مشترك** بين كل العملات،
    باستخدام ``compute_global_cutoff``: "آخر N عيّنة" من كل العيّنات (أي
    عملة) تقارب النسبة المطلوبة مباشرة — لا نسبة كل عملة على حدة.

    غلاف رقيق حول ``split_data(mode='global_time', split_cutoff_method=
    'raw_quantile')`` — لا منطق تقطيع مستقلّ، فيبقى هناك تنفيذ واحد فقط
    للتقسيم بحدود مشتركة (``_split_global_time``)، لا نسختان قد تنحرف إحداهما
    عن الأخرى بمرور الوقت.

    ⚠️ **هذه ليست إصلاحاً إضافياً لتسرّب قائم بين العملات** — ``split_data``
    بوضعه الافتراضي (``mode='global_time'``، ``split_cutoff_method=
    'kept_share'``) يمنع هذا التسرّب أصلاً بحدّ زمني واحد مشترك؛ الفرق هنا هو
    طريقة **اختيار** ذلك الحدّ فقط (نسبة خام من كل العيّنات، لا نسبة مُبقاة
    بعد العزل) — راجع تحذير :func:`compute_global_cutoff` قبل اعتمادها
    افتراضاً على بيانات مكدَّسة زمنياً (عملات كثيرة حديثة الإدراج).
    """
    config = dict(CONFIG if config is None else config)
    config['split_cutoff_method'] = 'raw_quantile'
    return split_data(data, train_pct=train_pct, val_pct=val_pct,
                      keep_asset_test_separate=keep_asset_test_separate,
                      mode='global_time', config=config)


# ══════════════════════════════════════════════════════════════════════════
# إعادة تدريب دورية (walk-forward) — عدّة نوافذ متحرّكة بدل حدّ ثابت واحد
# ══════════════════════════════════════════════════════════════════════════
#: علامة "لم يُمرَّر" لتمييزها عن None (قيمة صحيحة في train_span/initial_train_span).
_UNSET = object()


def rolling_split_schedule(data: Dict, test_span=None, val_span=None,
                           train_span=_UNSET, initial_train_span=_UNSET,
                           step=None, max_windows: Optional[int] = None,
                           config: Optional[dict] = None) -> List[Dict[str, pd.Timestamp]]:
    """يبني جدول حدود لنوافذ تدريب/تحقق/اختبار متحرّكة (walk-forward) —
    لإعادة تدريب دورية، بدل حدّ واحد ثابت من :func:`resolve_split_dates`.

    كل نافذة: ``train`` حتى ``train_end``، ثم ``val`` حتى ``val_end`` (بعد
    فجوة عزل)، ثم ``test`` حتى ``test_end`` (بعد فجوة عزل أخرى) — نفس منطق
    العزل في ``_split_global_time`` بالضبط، مُطبَّق داخل كل نافذة على حدة.

    Args:
        test_span/val_span: مدد زمنية (``pd.Timedelta`` أو نص مثل ``'30D'``)
            لكل من الاختبار والتحقق في كل نافذة — بلا افتراض عام، حدّدهما أو
            اضبطهما في ``CONFIG['rolling_retrain']``.
        train_span: مدة تدريب ثابتة (نافذة **منزلقة**) — أو ``None`` صراحةً
            لنافذة **متمدّدة** (تبدأ من أقدم عيّنة وتكبر مع كل إعادة تدريب،
            الأشيع لأن مزيداً من التاريخ عادة لا يضرّ). غير المُمرَّر
            (``_UNSET``) يقرأ من ``CONFIG['rolling_retrain']['train_span']``.
        initial_train_span: حجم **أول** نافذة تدريب — إلزامي عند نافذة
            متمدّدة (``train_span=None``) لأنه لا حجم افتراضي صحيح عالمياً؛
            يُرفع ``ValueError`` صريح إن غاب. يُتجاهَل عند نافذة منزلقة
            (``train_span`` يحكم كل النوافذ فيها بما فيها الأولى).
        step: مقدار انزلاق كل نافذة عن سابقتها. افتراضياً = ``test_span``
            (نوافذ اختبار متتالية غير متداخلة — كل تدريب يُقيَّم مرّة واحدة).
        max_windows: أقصى عدد نوافذ (``None`` = حتى نفاد البيانات).

    Returns:
        قائمة قواميس ``{'train_start','train_end','val_end','test_end'}``
        (``train_start`` أقدم طابع في البيانات دائماً عند نافذة متمدّدة).
        مرّرها إلى :func:`rolling_splits` لتقطيع ``dataset`` فعلياً، أو
        استخدمها مباشرة لمعاينة الجدول قبل أي تقطيع.
    """
    config = CONFIG if config is None else config
    rr = config.get('rolling_retrain') or {}
    test_span = test_span if test_span is not None else rr.get('test_span')
    val_span = val_span if val_span is not None else rr.get('val_span')
    if train_span is _UNSET:
        train_span = rr.get('train_span')
    if initial_train_span is _UNSET:
        initial_train_span = rr.get('initial_train_span')
    step = step if step is not None else rr.get('step')

    if test_span is None or val_span is None:
        raise ValueError(
            "حدّد test_span وval_span، أو اضبطهما في CONFIG['rolling_retrain'].")
    test_span = pd.Timedelta(test_span)
    val_span = pd.Timedelta(val_span)
    step = pd.Timedelta(step) if step is not None else test_span

    sliding = train_span is not None
    if sliding:
        train_span = pd.Timedelta(train_span)
        first_train_len = train_span
    else:
        if initial_train_span is None:
            raise ValueError(
                "نافذة تدريب متمدّدة (train_span=None) تتطلّب initial_train_span "
                "صراحةً — حجم أول نافذة تدريب قبل أن تبدأ بالتمدّد؛ لا قيمة "
                "افتراضية صحيحة عالمياً. حدّدها أو اضبطها في CONFIG['rolling_retrain'].")
        initial_train_span = pd.Timedelta(initial_train_span)
        first_train_len = initial_train_span

    ts = sample_timestamps(data)
    if len(ts) == 0:
        raise ValueError("dataset فارغ — لا عيّنات لبناء جدول نوافذ منه.")
    ts_min, ts_max = ts.min(), ts.max()
    gap = embargo_duration(data, config)

    schedule: List[Dict[str, pd.Timestamp]] = []
    train_end = ts_min + first_train_len
    while True:
        val_end = train_end + gap + val_span
        test_end = val_end + gap + test_span
        if test_end > ts_max:
            break
        train_start = (train_end - train_span) if sliding else ts_min
        schedule.append({'train_start': train_start, 'train_end': train_end,
                         'val_end': val_end, 'test_end': test_end})
        if max_windows and len(schedule) >= max_windows:
            break
        train_end = train_end + step

    if not schedule:
        needed = first_train_len + 2 * gap + val_span + test_span
        raise ValueError(
            f"❌ لا تتّسع البيانات لنافذة واحدة حتى: المدى الزمني المتاح "
            f"{(ts_max - ts_min).days:,} يوماً، والمطلوب لأول نافذة "
            f"{needed.days:,} يوماً (تدريب+عزلان+تحقق+اختبار). صغّر test_span/"
            f"val_span/initial_train_span أو وفّر تاريخاً أطول.")
    return schedule


def rolling_splits(data: Dict, test_span=None, val_span=None,
                   train_span=_UNSET, initial_train_span=_UNSET, step=None,
                   max_windows: Optional[int] = None,
                   keep_asset_test_separate: Optional[bool] = None,
                   prefix_y: bool = True, config: Optional[dict] = None,
                   verbose: bool = True) -> List[Tuple[dict, dict, dict]]:
    """يبني عدّة أزواج ``(train, val, test)`` بحدود زمنية متحرّكة — لإعادة
    تدريب دورية walk-forward، بدل حدّ ثابت واحد من :func:`split_data`.

    كل نافذة مُطهَّرة (purged+embargoed) بنفس منطق ``_split_global_time``
    تماماً — فجوة عزل بين كل قسمين، فلا تسرّب داخل أي نافذة على حدة.

    ⚠️ النوافذ **تتشارك بيانات تدريب** حتماً (تدريب كل نافذة يمتدّ من سابقتها
    أو يتقدّم عليها) — هذا هو المقصود بإعادة التدريب الدوري (كل نافذة =
    نموذج جديد يُدرَّب على تاريخ أحدث)، وليس تسرّباً: اختبار كل نافذة لا
    يتداخل زمنياً مع تدريبها هي فتبقى نتيجتها نظيفة على حدة.

    الوسائط كما في :func:`rolling_split_schedule`. يُرجع قائمة
    ``(train, val, test)`` بنفس شكل مخرجات ``split_data`` (``test`` قاموس
    ``{asset: split}`` إن ``keep_asset_test_separate=True``).
    """
    config = CONFIG if config is None else config
    if keep_asset_test_separate is None:
        keep_asset_test_separate = config['keep_asset_test_separate']

    schedule = rolling_split_schedule(data, test_span, val_span, train_span,
                                      initial_train_span, step, max_windows, config)
    ts = sample_timestamps(data)
    gap = embargo_duration(data, config)
    timeframes, targets = data['timeframes'], data['targets']

    out: List[Tuple[dict, dict, dict]] = []
    for i, w in enumerate(schedule, 1):
        train_mask = np.asarray((ts > w['train_start']) & (ts <= w['train_end']))
        val_mask = np.asarray((ts > w['train_end'] + gap) & (ts <= w['val_end']))
        test_mask = np.asarray((ts > w['val_end'] + gap) & (ts <= w['test_end']))

        sizes = {'train': int(train_mask.sum()), 'val': int(val_mask.sum()),
                'test': int(test_mask.sum())}
        _assert_split_sizes(sizes, config, gap, w['train_end'], w['val_end'])

        train = _take(data, train_mask, timeframes, targets)
        val = _take(data, val_mask, timeframes, targets)
        if keep_asset_test_separate:
            test: Dict[str, dict] = {}
            for b in data.get('asset_bounds') or []:
                asset_mask = np.zeros(len(ts), dtype=bool)
                asset_mask[b['start']:b['end']] = True
                combined = asset_mask & test_mask
                if combined.any():
                    name = b.get('name', b.get('asset', f"asset_{b['start']}_{b['end']}"))
                    test[name] = _take(data, combined, timeframes, targets)
            n_te = sum(len(v['base_params']) for v in test.values())
        else:
            test = _take(data, test_mask, timeframes, targets)
            n_te = sizes['test']

        if prefix_y:
            add_y_prefix(train)
            add_y_prefix(val)
            if isinstance(test, dict) and 'y' in test:
                add_y_prefix(test)
            else:
                for asset_split in test.values():
                    add_y_prefix(asset_split)

        if verbose:
            print(f"🔁 نافذة {i}/{len(schedule)}: "
                  f"train≤{w['train_end']:%Y-%m-%d} ({sizes['train']:,}) | "
                  f"val≤{w['val_end']:%Y-%m-%d} ({sizes['val']:,}) | "
                  f"test≤{w['test_end']:%Y-%m-%d} ({n_te:,})")
        out.append((train, val, test))
    return out


## 17) عميل Binance وجلب شموع حيّة

ثلاث دوال — كما وردت — لتوفير مصدر بيانات حيّ:

* `_create_client` / `client` — عميل Binance (يُنشأ **مرة واحدة** عند تشغيل
  الخلية، بإعادة محاولة عند انقطاع الشبكة المؤقت).
* `get_normalized_futures_symbols` — سرد رموز العقود الآجلة الخطّية النشطة
  المنتهية بـ USDT (عبر `ccxt`).
* `fetch_data` — جلب شموع OHLCV لرمز وفريم معيّنين (عبر `python-binance`).

القسم التالي (18) يستخدم `fetch_data` كمصدر بديل عن `load_asset` (Drive)
داخل **نفس** `build_dataset_from_loader`/`prepare_single_asset` تماماً —
بلا أي نسخة موازية من منطق التجهيز.


In [49]:
# @title
"""
عميل Binance وجلب الشموع الحيّة.

**مفتاحا API:** اتركهما فارغتين (``""``) لبيانات السوق العامة فقط (شموع
OHLCV) — لا تحتاج مصادقة. عيّنهما فقط إن احتجت نقاط نهاية خاصة بالحساب
(الرصيد، الصفقات...) لاحقاً. لا تكتبهما هنا مباشرة في دفتر تُشاركه — استخدم
متغيرات بيئة أو أسرار Colab (``google.colab.userdata.get(...)``).
"""

# 🔧 عدّل هذا (أو اتركه فارغاً لبيانات السوق العامة فقط):
API_KEY = ""
API_SECRET = ""

#: الحدّ الأقصى لعدد الشموع في طلب futures_klines واحد على Binance.
BINANCE_MAX_LIMIT = 1500


import threading
from collections import deque


class RateLimiter:
    """حدّ معدّل بنافذة منزلقة، آمن للخيوط: لا يتجاوز ``max_calls`` طلباً في أي
    فترة طولها ``period`` ثانية، مهما بلغ عدد الخيوط.

    ``acquire()`` **يحجب** الخيط حتى يُتاح له طلب ثم يسجّله (لا يرفض ولا يُسقط).
    النافذة المنزلقة بدل «عدّاد يُصفَّر كل دقيقة»: العدّاد الدوري يسمح بضعف الحدّ
    عند حدّ الدقيقتين (٢٠٠٠ في آخر ثانية + ٢٠٠٠ في أول ثانية من التالية).
    ``clock``/``sleep`` قابلان للحقن (للاختبار بلا انتظار حقيقي).
    """

    def __init__(self, max_calls: int, period: float = 60.0,
                 clock=time.monotonic, sleep=time.sleep):
        if int(max_calls) <= 0 or float(period) <= 0:
            raise ValueError("max_calls و period يجب أن يكونا موجبين.")
        self.max_calls, self.period = int(max_calls), float(period)
        self._clock, self._sleep = clock, sleep
        self._calls: deque = deque()
        self._lock = threading.Lock()
        self.total_waited = 0.0          # مجموع ثواني الانتظار (للمراقبة)

    def acquire(self) -> float:
        """يحجب إن لزم ثم يحجز طلباً. يُرجع ثواني الانتظار في هذا الاستدعاء."""
        waited = 0.0
        while True:
            with self._lock:
                now = self._clock()
                while self._calls and now - self._calls[0] >= self.period:
                    self._calls.popleft()
                if len(self._calls) < self.max_calls:
                    self._calls.append(now)
                    self.total_waited += waited
                    return waited
                wait = self.period - (now - self._calls[0])
            self._sleep(wait)            # خارج القفل: لا نحجب بقية الخيوط أثناء النوم
            waited += wait


_REQUEST_LIMITERS: Dict[int, RateLimiter] = {}
_REQUEST_LIMITERS_LOCK = threading.Lock()


def get_request_limiter(config: Optional[dict] = None) -> Optional[RateLimiter]:
    """محدِّد الطلبات المشترك بين كل الخيوط، من ``CONFIG['live_max_requests_per_minute']``.

    نفس القيمة ⇒ نفس المحدِّد (ليُحصى كل شيء في نافذة واحدة). ``0``/``None`` ⇒
    بلا حدّ. تغيير القيمة أثناء التشغيل يبدأ محدِّداً جديداً بنافذة فارغة.
    """
    config = CONFIG if config is None else config
    n = config.get("live_max_requests_per_minute", 2000)
    if not n:
        return None
    n = int(n)
    with _REQUEST_LIMITERS_LOCK:
        limiter = _REQUEST_LIMITERS.get(n)
        if limiter is None:
            limiter = _REQUEST_LIMITERS[n] = RateLimiter(n, 60.0)
        return limiter

#: هامش إحماء عام (بالشموع) تحتاجه أطول نافذة تدحرج في الميزات المخصّصة
#: (custom.py: range_windows حتى 50، vol_windows حتى 48) والمؤشرات الفنية —
#: يُستخدَم في build_live_dataset عند حساب limit تلقائياً، فلا تُستنزَف كل
#: الصفوف بعد dropna() على الفريمات الأعلى ذات العدد الأقل من الشموع.
LIVE_FEATURE_WARMUP = 60


def _create_client(retries: int = 5, delay: float = 5.0):
    """يُنشئ عميل Binance مع إعادة محاولة عند انقطاع الشبكة المؤقت.

    ``Client(...)`` يستدعي ping داخليًا عند الإنشاء، فأي انقطاع DNS/شبكة
    عابر (شائع في بوت يعمل ساعات طويلة) كان يُسقِط البرنامج بالكامل فورًا.
    """
    from binance.client import Client

    last_err: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            return Client(API_KEY, API_SECRET)
        except Exception as e:                             # noqa: BLE001
            last_err = e
            print(f"⚠️ فشل الاتصال بـ Binance (محاولة {attempt}/{retries}): {e}")
            if attempt < retries:
                time.sleep(delay)
    raise RuntimeError(
        f"لا يوجد اتصال بالإنترنت أو فشل تهيئة العميل بعد {retries} محاولات: {last_err}"
    ) from last_err


#: عميل Binance الحيّ — يُنشأ مرة واحدة عند تشغيل هذه الخلية. يتطلب اتصال شبكة
#: فعلياً؛ أعد تشغيل الخلية لإعادة إنشائه إن انقطع الاتصال طويلاً.
# client = _create_client()


def get_normalized_futures_symbols() -> List[str]:
    """أزواج عقود Binance الآجلة الخطّية النشطة المنتهية بـ USDT."""
    import ccxt

    markets = ccxt.binance({"options": {"defaultType": "future"}}).load_markets()
    return sorted(
        s.replace("/", "").split(":")[0] for s in markets
        if markets[s]["linear"] and markets[s]["active"] and s.endswith("USDT")
    )


_KLINE_COLUMNS = [
    "timestamp", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore",
]
_OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]


def _klines_to_frame(klines) -> pd.DataFrame:
    """يحوّل ردّ ``futures_klines`` الخام إلى DataFrame (timestamp UTC + OHLCV رقمية)."""
    df = pd.DataFrame(klines, columns=_KLINE_COLUMNS)
    df = df[["timestamp"] + _OHLCV_COLUMNS]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    for col in _OHLCV_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=_OHLCV_COLUMNS).reset_index(drop=True)


def fetch_data(symbol: str, interval: str, limit: int,
               retries: int = 3, delay: float = 2.0,
               page_delay: float = 0.0,
               rate_limiter: Optional[Any] = None) -> Optional[pd.DataFrame]:
    """جلب شموع OHLCV من Binance Futures، مفهرسة زمنيًا (UTC) ومرتّبة.

    إن زاد ``limit`` على ``BINANCE_MAX_LIMIT`` (الحدّ الأقصى للطلب الواحد) يُقسَّم
    الجلب إلى عدة طلبات **من الأحدث إلى الأقدم** (كل طلب يبدأ حيث انتهى السابق عبر
    ``endTime``)، ثم تُدمج الصفحات وتُرتَّب وتُزال تكراراتها قبل التسليم — فيصل
    للمستدعي ``DataFrame`` واحد مرتّب بأحدث ``limit`` شمعة، كأن الطلب كان واحداً.

    * إن كان التاريخ المتاح أقصر من ``limit`` تُرجَع الشموع المتوفرة فقط (لا خطأ).
    * إن فشلت أي صفحة بعد ``retries`` محاولات تُرجَع ``None`` للطلب كله: نتيجة
      مبتورة من الأقدم تبدو سليمة وتمرّ بصمت إلى التدريب/الاستدلال، والفشل الصريح
      أفضل منها (المستدعي يتعامل أصلاً مع ``None``).
    * ``page_delay``: ثوانٍ انتظار بين الصفحات (لتخفيف ضغط حدّ الوزن على Binance).
    * **حدّ المعدّل:** كل طلب (صفحة أو إعادة محاولة) يمرّ عبر محدِّد مشترك بين الخيوط
      قيمته ``CONFIG['live_max_requests_per_minute']`` (2000 افتراضياً) — يحجب الخيط
      حتى يتوفّر طلب بدل رفضه. ``rate_limiter`` يتجاوزه لهذا الاستدعاء (للاختبار).
    """
    from binance.exceptions import BinanceAPIException

    binance_interval = "1d" if interval == "1D" else interval
    limit = int(limit)
    if limit <= 0:
        raise ValueError(f"limit يجب أن يكون موجباً (وصل {limit}).")
    limiter = rate_limiter if rate_limiter is not None else get_request_limiter()

    def _fetch_page(page_limit: int, end_time: Optional[int], page_no: int):
        """صفحة واحدة بإعادة محاولة → ``(عدد الشموع الخام، أول open_time، DataFrame)`` أو None."""
        params = dict(symbol=symbol, interval=binance_interval, limit=page_limit)
        if end_time is not None:
            params["endTime"] = end_time
        tag = f"{symbol}/{interval}" + (f" ص{page_no}" if page_no > 1 else "")
        for attempt in range(retries):
            try:
                if limiter is not None:
                    limiter.acquire()
                klines = client.futures_klines(**params)
                first_ms = int(klines[0][0]) if len(klines) else None
                return len(klines), first_ms, _klines_to_frame(klines)
            except BinanceAPIException as e:
                print(f"⚠️ [{tag}] خطأ API (محاولة {attempt + 1}): {e}")
            except Exception as e:                         # noqa: BLE001
                print(f"⚠️ [{tag}] خطأ غير متوقع (محاولة {attempt + 1}): {e}")
            if attempt < retries - 1:
                time.sleep(delay)
        print(f"❌ [{tag}] فشلت كل محاولات الجلب")
        return None

    frames: List[pd.DataFrame] = []      # الأحدث أولاً (ترتيب الجلب)
    remaining, end_time, page_no = limit, None, 0
    while remaining > 0:
        page_no += 1
        page_limit = min(remaining, BINANCE_MAX_LIMIT)
        page = _fetch_page(page_limit, end_time, page_no)
        if page is None:
            return None
        raw_n, first_ms, frame = page
        if raw_n == 0:
            break                        # لا تاريخ أقدم
        frames.append(frame)
        remaining -= raw_n
        if raw_n < page_limit:
            break                        # الصفحة ناقصة ⇒ استُنفد التاريخ المتاح
        next_end = first_ms - 1
        if end_time is not None and next_end >= end_time:
            break                        # شبكة أمان: لا تقدّم ⇒ لا حلقة لا نهائية
        end_time = next_end
        if remaining > 0 and page_delay:
            time.sleep(page_delay)

    if not frames:
        return _klines_to_frame([])
    df = pd.concat(frames[::-1], ignore_index=True)      # الأقدم أولاً
    df = (df.drop_duplicates("timestamp", keep="last")
            .sort_values("timestamp").reset_index(drop=True))
    return df.tail(limit).reset_index(drop=True)[["timestamp"] + _OHLCV_COLUMNS]


# ══════════════════════════════════════════════════════════════════════════
# معدّل التمويل والفائدة المفتوحة (عقود آجلة) — للاستخدام لاحقاً
# ══════════════════════════════════════════════════════════════════════════
#: الحدّ الأقصى لعدد السجلات في طلب futures_funding_rate واحد.
FUNDING_MAX_LIMIT = 1000
#: الحدّ الأقصى لعدد السجلات في طلب futures_open_interest_hist واحد.
OPEN_INTEREST_MAX_LIMIT = 500
#: ⚠️ Binance نفسها لا تخدم openInterestHist أبعد من هذه المدة، مهما طُلب أو
#: قُسِّم الطلب — قيد من طرف المنصة، لا هذا الكود. معدّل التمويل لا يحمل هذا
#: القيد (تاريخه الكامل متاح عبر futures_funding_rate كأي شمعة).
OPEN_INTEREST_MAX_LOOKBACK = pd.Timedelta(days=30)


def fetch_funding_rate(symbol: str, limit: int, retries: int = 3, delay: float = 2.0,
                       page_delay: float = 0.0,
                       rate_limiter: Optional[Any] = None) -> Optional[pd.DataFrame]:
    """جلب تاريخ معدّل التمويل (funding rate) لعقد آجل — تاريخه الكامل متاح
    (لا حدّ ٣٠ يوماً، خلافاً للفائدة المفتوحة أدناه)، فتقسيم الطلبات الكبيرة
    يتبع **نفس** منطق :func:`fetch_data` (تراجع بـ endTime، إعادة محاولة،
    حدّ معدّل مشترك) — منسوخ لا مُشترَك معه عمداً، تفادياً لأي مخاطرة بكسر
    اختباراته القائمة عبر إعادة هيكلة مشتركة.

    يُرجع ``DataFrame`` بعمودي ``timestamp`` (UTC) و``funding_rate``، مرتّباً
    تصاعدياً بأحدث ``limit`` سجل. أحداث التمويل كل ٨ ساعات (٣ في اليوم)، فـ
    ``limit=1095`` يغطي سنة تقريباً.
    """
    from binance.exceptions import BinanceAPIException

    limit = int(limit)
    if limit <= 0:
        raise ValueError(f"limit يجب أن يكون موجباً (وصل {limit}).")
    limiter = rate_limiter if rate_limiter is not None else get_request_limiter()

    def _fetch_page(page_limit: int, end_time: Optional[int], page_no: int):
        params = dict(symbol=symbol, limit=page_limit)
        if end_time is not None:
            params["endTime"] = end_time
        tag = f"{symbol}/funding" + (f" ص{page_no}" if page_no > 1 else "")
        for attempt in range(retries):
            try:
                if limiter is not None:
                    limiter.acquire()
                rows = client.futures_funding_rate(**params)
                first_ms = int(rows[0]["fundingTime"]) if len(rows) else None
                return len(rows), first_ms, rows
            except BinanceAPIException as e:
                print(f"⚠️ [{tag}] خطأ API (محاولة {attempt + 1}): {e}")
            except Exception as e:                             # noqa: BLE001
                print(f"⚠️ [{tag}] خطأ غير متوقع (محاولة {attempt + 1}): {e}")
            if attempt < retries - 1:
                time.sleep(delay)
        print(f"❌ [{tag}] فشلت كل محاولات الجلب")
        return None

    frames: List[list] = []
    remaining, end_time, page_no = limit, None, 0
    while remaining > 0:
        page_no += 1
        page_limit = min(remaining, FUNDING_MAX_LIMIT)
        page = _fetch_page(page_limit, end_time, page_no)
        if page is None:
            return None
        raw_n, first_ms, rows = page
        if raw_n == 0:
            break
        frames.append(rows)
        remaining -= raw_n
        if raw_n < page_limit:
            break
        next_end = first_ms - 1
        if end_time is not None and next_end >= end_time:
            break
        end_time = next_end
        if remaining > 0 and page_delay:
            time.sleep(page_delay)

    if not frames:
        return pd.DataFrame(columns=["timestamp", "funding_rate"])
    rows = [r for page in reversed(frames) for r in page]
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["fundingTime"].astype("int64"), unit="ms", utc=True)
    df["funding_rate"] = pd.to_numeric(df["fundingRate"], errors="coerce")
    df = (df[["timestamp", "funding_rate"]].dropna(subset=["funding_rate"])
          .drop_duplicates("timestamp", keep="last")
          .sort_values("timestamp").reset_index(drop=True))
    return df.tail(limit).reset_index(drop=True)


def fetch_open_interest_hist(symbol: str, limit: int, period: str = "1h",
                             retries: int = 3, delay: float = 2.0,
                             rate_limiter: Optional[Any] = None) -> Optional[pd.DataFrame]:
    """جلب تاريخ الفائدة المفتوحة (open interest).

    ⚠️ Binance نفسها لا تخدم أبعد من :data:`OPEN_INTEREST_MAX_LOOKBACK`
    (٣٠ يوماً) عبر هذا المسار، مهما كان ``limit`` أو عدد الصفحات — قيد من
    طرف المنصة لا هذا الكود، فلا فائدة من طلب تاريخ أبعد. لأرشيف أطول: اجمعها
    دورياً (يومياً مثلاً) بـ :func:`save_funding_open_interest`؛ كل تشغيل
    يُضيف آخر ٣٠ يوماً المتاحة فيتراكم الأرشيف مع الزمن.

    ``period``: دقة العيّنات (\"5m\".. \"1d\"). عند ``period=\"1h\"`` تحتاج ٣٠
    يوماً صفحتين على الأكثر (٧٢٠ نقطة > الحدّ الأقصى ٥٠٠ للطلب الواحد)،
    تُجلَبان تلقائياً بنفس منطق :func:`fetch_funding_rate`.
    """
    from binance.exceptions import BinanceAPIException

    limit = int(limit)
    if limit <= 0:
        raise ValueError(f"limit يجب أن يكون موجباً (وصل {limit}).")
    limiter = rate_limiter if rate_limiter is not None else get_request_limiter()

    def _fetch_page(page_limit: int, end_time: Optional[int], page_no: int):
        params = dict(symbol=symbol, period=period, limit=page_limit)
        if end_time is not None:
            params["endTime"] = end_time
        tag = f"{symbol}/OI" + (f" ص{page_no}" if page_no > 1 else "")
        for attempt in range(retries):
            try:
                if limiter is not None:
                    limiter.acquire()
                rows = client.futures_open_interest_hist(**params)
                first_ms = int(rows[0]["timestamp"]) if len(rows) else None
                return len(rows), first_ms, rows
            except BinanceAPIException as e:
                print(f"⚠️ [{tag}] خطأ API (محاولة {attempt + 1}): {e}")
            except Exception as e:                             # noqa: BLE001
                print(f"⚠️ [{tag}] خطأ غير متوقع (محاولة {attempt + 1}): {e}")
            if attempt < retries - 1:
                time.sleep(delay)
        print(f"❌ [{tag}] فشلت كل محاولات الجلب")
        return None

    frames: List[list] = []
    remaining, end_time, page_no = limit, None, 0
    while remaining > 0:
        page_no += 1
        page_limit = min(remaining, OPEN_INTEREST_MAX_LIMIT)
        page = _fetch_page(page_limit, end_time, page_no)
        if page is None:
            return None
        raw_n, first_ms, rows = page
        if raw_n == 0:
            break
        frames.append(rows)
        remaining -= raw_n
        if raw_n < page_limit:
            break
        next_end = first_ms - 1
        if end_time is not None and next_end >= end_time:
            break
        end_time = next_end

    if not frames:
        return pd.DataFrame(columns=["timestamp", "open_interest"])
    rows = [r for page in reversed(frames) for r in page]
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"].astype("int64"), unit="ms", utc=True)
    df["open_interest"] = pd.to_numeric(df["sumOpenInterest"], errors="coerce")
    df = (df[["timestamp", "open_interest"]].dropna(subset=["open_interest"])
          .drop_duplicates("timestamp", keep="last")
          .sort_values("timestamp").reset_index(drop=True))
    return df.tail(limit).reset_index(drop=True)


def _funding_oi_drive_path(root: Path, kind: str, symbol: str, config: dict) -> Path:
    sub = (config.get(kind) or {}).get('drive_dir', kind)
    d = root / sub
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{symbol}.csv"


def save_funding_open_interest(symbols: List[str], funding_limit: int = 3000,
                               oi_limit: int = 720, oi_period: str = "1h",
                               config: Optional[dict] = None,
                               verbose: bool = True) -> None:
    """يجلب معدّل التمويل والفائدة المفتوحة لكل رمز ويُلحقها بأرشيف CSV على
    Drive (مسار كلٍّ منهما في ``CONFIG['funding_rate']['drive_dir']`` /
    ``['open_interest']['drive_dir']``) — **للاستخدام لاحقاً**، غير مدموجة
    تلقائياً في ``add_features`` بعد.

    كل تشغيل: يقرأ الأرشيف الحالي إن وُجد، يجلب الأحدث، يدمج بلا تكرار على
    ``timestamp``، ويحفظ. شغّلها **دورياً** (يدوياً، أو خلية Colab مجدولة) —
    هذا هو الحلّ الوحيد لتجاوز قيد الـ٣٠ يوماً على الفائدة المفتوحة من
    Binance نفسها: كل تشغيل يُضيف نافذته المتاحة، فيتراكم أرشيف أطول من
    السجلّ الفعلي لتشغيلاتك، لا من طلب واحد بأثر رجعي.
    """
    config = CONFIG if config is None else config
    root = mount_drive(config=config)
    fr_cfg, oi_cfg = config.get('funding_rate') or {}, config.get('open_interest') or {}

    for sym in symbols:
        if fr_cfg.get('enabled', True):
            new = fetch_funding_rate(sym, funding_limit)
            _merge_and_save_archive(root, 'funding_rate', sym, new, config, verbose)
        if oi_cfg.get('enabled', True):
            new = fetch_open_interest_hist(sym, oi_limit, period=oi_cfg.get('period', oi_period))
            _merge_and_save_archive(root, 'open_interest', sym, new, config, verbose)


def _merge_and_save_archive(root: Path, kind: str, symbol: str,
                            new_df: Optional[pd.DataFrame], config: dict,
                            verbose: bool) -> None:
    if new_df is None:
        print(f"   ❌ [{symbol}/{kind}] فشل الجلب — لم يُحفَظ شيء لهذا الرمز.")
        return
    path = _funding_oi_drive_path(root, kind, symbol, config)
    if path.exists():
        old = pd.read_csv(path, parse_dates=['timestamp'])
        merged = pd.concat([old, new_df], ignore_index=True)
    else:
        merged = new_df
    merged = (merged.drop_duplicates('timestamp', keep='last')
             .sort_values('timestamp').reset_index(drop=True))
    merged.to_csv(path, index=False)
    if verbose:
        print(f"   💾 [{symbol}/{kind}] +{len(new_df):,} سجل جديد | "
              f"{len(merged):,} إجمالي محفوظ ({path.name})")


def load_funding_open_interest(symbol: str, kind: str = 'funding_rate',
                               config: Optional[dict] = None) -> Optional[pd.DataFrame]:
    """يقرأ الأرشيف المحفوظ (``kind`` = ``'funding_rate'`` أو ``'open_interest'``)
    لرمز من Drive. ``None`` إن لم يُجمَع شيء بعد له (شغّل
    :func:`save_funding_open_interest` أولاً)."""
    config = CONFIG if config is None else config
    root = mount_drive(config=config)
    path = _funding_oi_drive_path(root, kind, symbol, config)
    if not path.exists():
        return None
    return pd.read_csv(path, parse_dates=['timestamp'])


## 17-ب) دمج معدّل التمويل والفائدة المفتوحة كميزات

بروح `add_market_context` (المحاذاة بـ `reindex(..., method='ffill')`)، لكن كل أصل يحمل **أرشيفه الخاص** (`load_funding_open_interest`) لا مرجعاً مشتركاً واحداً. تُدمَج فقط إن `CONFIG['funding_rate']['as_feature']` أو `CONFIG['open_interest']['as_feature']` مفعَّلة (الافتراض الآن: كلاهما True) — الجلب/الأرشفة (`enabled`) والإدماج في المدخلات (`as_feature`) مفتاحان منفصلان عمداً، فيمكن أرشفة بيانات دون دمجها فوراً.


In [50]:
# @title
"""
دمج معدّل التمويل والفائدة المفتوحة كميزات — المرحلة ١ من خطة اكتشاف الإشارة.

بخلاف السياق السوقي (add_market_context، مرجع واحد مشترك بين كل الأصول)،
هنا كل أصل يحمل أرشيفه الخاص المحفوظ عبر save_funding_open_interest. القيمة
الخام غير آمنة كميزة مباشرة على أكثر من محور:

  * open_interest مستوى مطلق يختلف بأوامر عشرية بين الأصول (BTC مقابل عملة
    صغيرة) — بالضبط نفس مشكلة OHLCV الخام التي حُلّت بـ RET_/RANGE_ في
    custom.py؛ الحلّ هنا مماثل: تغيّر نسبي (لوغاريتمي) لا مستوى.
  * الأرشيف يبدأ من أول تشغيل فعلي لـsave_funding_open_interest — فترات
    أقدم (شائعة على بيانات تاريخية بمدى سنوات) بلا أي عيّنة محفوظة أصلاً.
    تركها NaN ثم dropna() في add_features كان سيُسقط كل ذلك التاريخ بصمت؛
    ملء صفر بلا تمييز كان سيخلط "لا تغيّر حقيقي" بـ"لا بيانات إطلاقاً" على
    النموذج. الحل: قيمة محايدة (صفر) + عمود علم توفّر صريح
    (FUND_available/OI_available) يميّز الحالتين.
"""


def _funding_rate_frame(raw: Optional[pd.DataFrame],
                        config: Optional[dict] = None) -> Optional[pd.DataFrame]:
    """يحوّل أرشيف funding_rate الخام (عمودا timestamp، funding_rate) إلى
    إطار ميزات على فترات التمويل الأصلية (كل ٨ ساعات عادة) — غير مُحاذى بعد
    على فريم أي أصل (تتولّاه add_funding_oi_features). ``None`` إن كان
    ``raw`` فارغاً أو ``None`` (لا أرشيف محفوظ بعد لهذا الرمز).

    ``FUND_rate_z``: درجة معيارية متدحرجة (``zscore_window`` شمعة تمويل) —
    تُشغّل مباشرة فرضية "funding مرتفع جداً = تموضع مفرط" من كتالوج
    الفرضيات (قسم ٢ في خطة المشروع)، بمرجع كل عملة على تاريخها هي، لا حدّاً
    مطلقاً يتجاهل اختلاف مستويات funding الطبيعية بين الأصول.
    """
    if raw is None or raw.empty:
        return None
    config = CONFIG if config is None else config
    window = int((config.get('funding_rate') or {}).get('zscore_window', 90))
    s = raw.set_index('timestamp')['funding_rate'].astype('float64').sort_index()
    min_p = max(5, window // 3)
    mu = s.rolling(window, min_periods=min_p).mean()
    sd = s.rolling(window, min_periods=min_p).std()
    z = ((s - mu) / sd.replace(0, np.nan)).fillna(0.0)     # إحماء الرولينغ: محايد لا NaN
    return pd.DataFrame({'FUND_rate': s, 'FUND_rate_z': z})


def _oi_frame(raw: Optional[pd.DataFrame],
             config: Optional[dict] = None) -> Optional[pd.DataFrame]:
    """يحوّل أرشيف open_interest الخام إلى تغيّر نسبي لوغاريتمي — لا مستوى
    مطلق (يختلف بأوامر عشرية بين الأصول). أول نقطة في الأرشيف بلا "سابقة"
    فيُنتج diff() لها NaN، تُترَك لتُملأ محايدة صفر لاحقاً في المُحاذاة —
    حدّ إحماء طبيعي، لا خطأ."""
    if raw is None or raw.empty:
        return None
    config = CONFIG if config is None else config
    period = int((config.get('open_interest') or {}).get('change_period', 1))
    s = raw.set_index('timestamp')['open_interest'].astype('float64').sort_index()
    chg = np.log(s.clip(lower=EPS)).diff(period)
    return pd.DataFrame({f'OI_chg_{period}': chg})


def funding_oi_feature_columns(config: Optional[dict] = None) -> List[str]:
    """أسماء أعمدة معدّل التمويل/الفائدة المفتوحة — قائمة ثابتة من الإعدادات
    وحدها (بلا بيانات فعلية)، لتُدرَج في feature_order تلقائياً — بنفس دور
    market_context_columns لأعمدة MKT_."""
    config = CONFIG if config is None else config
    cols: List[str] = []
    fr = config.get('funding_rate') or {}
    if fr.get('enabled', False) and fr.get('as_feature', False):
        cols += ['FUND_rate', 'FUND_rate_z', 'FUND_available']
    oi = config.get('open_interest') or {}
    if oi.get('enabled', False) and oi.get('as_feature', False):
        period = int(oi.get('change_period', 1))
        cols += [f'OI_chg_{period}', 'OI_available']
    return cols


def add_funding_oi_features(dfs: Dict[str, pd.DataFrame], symbol: str,
                            config: Optional[dict] = None) -> Dict[str, pd.DataFrame]:
    """يُلحق أعمدة معدّل التمويل/الفائدة المفتوحة (المُفعَّلتان فقط) إلى
    ``dfs`` (لكل فريم) — محاذاة بـ ``reindex(..., method='ffill')`` من
    أرشيف Drive الخاص بـ ``symbol`` (خلاف add_market_context: كل أصل هنا
    يحمل أرشيفه الخاص، لا مرجعاً مشتركاً). لا شيء مُفعَّل ⇒ ``dfs`` تُعاد
    كما هي بلا نسخ إضافي.
    """
    config = CONFIG if config is None else config
    fr = config.get('funding_rate') or {}
    oi = config.get('open_interest') or {}
    use_fr = bool(fr.get('enabled', False) and fr.get('as_feature', False))
    use_oi = bool(oi.get('enabled', False) and oi.get('as_feature', False))
    if not use_fr and not use_oi:
        return dfs

    fr_frame = (_funding_rate_frame(load_funding_open_interest(symbol, 'funding_rate', config), config)
               if use_fr else None)
    oi_period = int(oi.get('change_period', 1))
    oi_col = f'OI_chg_{oi_period}'
    oi_frame = (_oi_frame(load_funding_open_interest(symbol, 'open_interest', config), config)
               if use_oi else None)

    out: Dict[str, pd.DataFrame] = {}
    for tf, df in dfs.items():
        d = df.copy()
        if use_fr:
            if fr_frame is not None:
                aligned = fr_frame.reindex(d.index, method='ffill')
                d['FUND_available'] = aligned['FUND_rate'].notna().astype('float64')
                d['FUND_rate'] = aligned['FUND_rate'].fillna(0.0)
                d['FUND_rate_z'] = aligned['FUND_rate_z'].fillna(0.0)
            else:
                d['FUND_rate'] = 0.0
                d['FUND_rate_z'] = 0.0
                d['FUND_available'] = 0.0
        if use_oi:
            if oi_frame is not None:
                aligned = oi_frame.reindex(d.index, method='ffill')
                d['OI_available'] = aligned[oi_col].notna().astype('float64')
                d[oi_col] = aligned[oi_col].fillna(0.0)
            else:
                d[oi_col] = 0.0
                d['OI_available'] = 0.0
        out[tf] = d
    return out


## 18) بناء بيانات حيّة — بنفس دالة `prepare_single_asset` بلا أي تكرار

هذا القسم يطبّق ما طلبتَه بالحرف: **تجهيز البيانات الحيّة مطابق تماماً**
لتجهيز بيانات التدريب — نفس `resample_fn`، ونفس `prepare_single_asset` عبر
نفس `build_dataset_from_loader` (صفر نسخ مكرّرة من منطق المحاذاة/التطبيع/الهدف).
الاختلاف الوحيد هو **مصدر الجلب**: `fetch_data` (Binance) بدل `load_asset`
(Drive) — تماماً كما طلبت.

**كيف نحصل على عيّنة بلا هدف حقيقي دون تعديل `prepare_single_asset`؟**
نُلحق بذيل البيانات المجلوبة **دفعة صورية** (candles وهمية بطول كافٍ، سعرها
= آخر إغلاق حقيقي، حجمها صفر) قبل تمريرها لخط الأنابيب، ونستدعي البناء
بـ `stride=1` بدل قيمة `CONFIG['stride']` المعتادة (لمرة هذا الاستدعاء
فقط، دون أي تعديل على `CONFIG` نفسه). هذا يمنح `prepare_single_asset` "مستقبلاً"
كافياً فلا تُسقِط آخر شمعة حقيقية بدل حسابها، فيُحسب لها **هدف صوري**
(بسعر يساوي آخر إغلاق حقيقي — قيمة تحكيمية لا معنى تنبؤياً لها) عوضاً عن
استبعادها بالكامل — تماماً كما طلبت ("بدون هدف أو بهدف افتراضي"). بعد
البناء، تُحذَف تلقائياً كل النوافذ التي **ميزاتها هي** الملوَّثة بالدفعة
الصورية (وليس فقط هدفها) عبر `drop_tail_per_asset` — فيبقى آخر صف لكل
عملة **حقيقي الميزات بالكامل** (يعتمد فقط على شموع حقيقية) بهدف صوري فقط.

**النتيجة:** لكل عملة، كل الصفوف ما عدا الأخير تملك **أهدافاً حقيقية
محسوبة فعلياً** من داخل الدفعة المجلوبة نفسها — تصلح لتقييم أداء النموذج
على الأيام الأخيرة (اختبار فوري)، أمّا الصف الأخير فهو نقطة **live**
الحقيقية (بلا هدف صالح) الجاهزة للاستدلال الفعلي. `extract_last_batch`
أدناه يستخلص هذا الصف الأخير لكل عملة من أي مجموعة بيانات مُجهَّزة.


In [51]:
# @title
def _live_pad_length(tf_order: List[str], window_sizes: Dict[str, int],
                     forecast_horizon: int, download_interval: str) -> int:
    """طول الدفعة الصورية، بوحدة شموع ``download_interval`` — **ليس** بالضرورة
    أصغر فريم في ``tf_order`` بعد الآن: تُجلَب بيانات Binance بـ
    ``download_interval`` ثم تُجمَّع (resample) صعوداً إلى كل فريم في
    ``tf_order``، فقد يكون ``download_interval`` أدقّ منه (نموذج يومي مبنيّ
    من بيانات ساعية مثلاً) — ولهذا يُحوَّل ``forecast_horizon`` (بوحدة
    tf_order[0]) وهامش تلوّث أكبر فريم كلاهما لوحدة ``download_interval``
    قبل جمعهما، لا افتراض أنهما نفس الوحدة كما كان سابقاً.
    """
    ratios = [max(1, int(np.ceil(pd.Timedelta(tf) / pd.Timedelta(download_interval))))
             for tf in tf_order]
    base_ratio = ratios[0]      # عدد شموع download_interval في شمعة واحدة من tf_order[0]
    return forecast_horizon * base_ratio + max(ratios)


def _pad_with_dummy_tail(df: pd.DataFrame, download_interval: str, n: int) -> pd.DataFrame:
    """يُلحق ``n`` شمعة صورية بذيل ``df`` (مفهرس بـ DatetimeIndex)، بخطوة
    ``download_interval`` (فريم التحميل الفعلي من Binance، لا فريم النموذج) —
    السعر = آخر إغلاق حقيقي (ثابت)، الحجم = صفر. هذا يمنح ``prepare_single_asset``
    \"مستقبلاً\" كافياً لحساب هدف (صوري) لآخر شمعة حقيقية بدل إسقاطها، **دون
    أي تعديل على تلك الدالة**.
    """
    if len(df) == 0 or n <= 0:
        return df
    last_close = float(df['close'].iloc[-1])
    step = pd.Timedelta(download_interval if download_interval != "1D" else "1d")
    future_idx = df.index[-1] + step * np.arange(1, n + 1)
    dummy = pd.DataFrame(
        {'open': last_close, 'high': last_close, 'low': last_close,
         'close': last_close, 'volume': 0.0},
        index=future_idx)
    return pd.concat([df, dummy])


def _auto_live_limit(tf_order: List[str], window_sizes: Dict[str, int],
                     forecast_horizon: int, pad_n: int, download_interval: str) -> int:
    """عدد الشموع الحقيقية المطلوب جلبها من Binance بوحدة ``download_interval``
    (لا يشمل الدفعة الصورية).

    ✅ بلا حدّ أقصى هنا (لا ``min(..., BINANCE_MAX_LIMIT)`` كسابقاً): ``fetch_data``
    تُقسّم أي ``limit`` أكبر من ``BINANCE_MAX_LIMIT`` على عدة طلبات تلقائياً —
    تقييد الطلب هنا كان يُبطل تلك الإضافة، وهو تحديداً سبب "٣٢ شمعة فقط" في
    محاذاة النوافذ سابقاً حين يكبر ``download_interval`` نسبةً لأصغر فريم في
    ``tf_order`` (نموذج يومي يحتاج مئات الساعات من الإحماء، تتجاوز ١٥٠٠
    بسهولة). القصّ الوحيد المتبقي الآن هو التاريخ الفعلي المتاح للعملة نفسها
    (تتولّاه fetch_data بإرجاع ما توفّر لا خطأ).
    """
    ratios = {tf: max(1, int(np.ceil(pd.Timedelta(tf) / pd.Timedelta(download_interval))))
             for tf in tf_order}
    biggest_span = max(
        (window_sizes[tf] + LIVE_FEATURE_WARMUP) * ratios[tf] for tf in tf_order
    )
    base_ratio = ratios[tf_order[0]]
    return (biggest_span + forecast_horizon * base_ratio) * 2 + pad_n


def _live_contamination_len(tf_order: List[str]) -> int:
    """عدد نوافذ ``tf_order[0]`` (وحدة النموذج) المُلوَّثة في الذيل — يعتمد
    فقط على نِسَب فريمات ``tf_order`` فيما بينها، بصرف النظر تماماً عن
    ``download_interval``: فريم الجلب لا يُغيّر عدد نوافذ *النموذج* المتضرّرة،
    فقط عدد شموع *التحميل* الصورية اللازمة لتوليدها (انظر _live_pad_length).
    """
    ratios = [max(1, int(np.ceil(pd.Timedelta(tf) / pd.Timedelta(tf_order[0]))))
             for tf in tf_order]
    return max(ratios)


# ══════════════════════════════════════════════════════════════════════════
# أدوات عامة: اقتطاع/استخلاص حسب نطاقات الأصول (asset_bounds)
# ══════════════════════════════════════════════════════════════════════════
def _asset_tail_mask(dataset: Dict, n: int, keep_tail: bool) -> np.ndarray:
    """قناع منطقي ``(N,)`` يُبقي (``keep_tail=True``) أو يُسقط
    (``keep_tail=False``) آخر ``n`` عيّنة من نطاق **كل أصل** في
    ``dataset['asset_bounds']`` على حدة (لا آخر n من المصفوفة كلها)."""
    total = len(dataset['base_params'])
    bounds = dataset.get('asset_bounds') or [{'start': 0, 'end': total}]
    mask = np.zeros(total, dtype=bool) if keep_tail else np.ones(total, dtype=bool)
    for b in bounds:
        s, e = b['start'], b['end']
        cut = max(s, e - n)
        mask[cut:e] = keep_tail
    return mask


def _apply_asset_mask(dataset: Dict, mask: np.ndarray) -> Dict:
    """يقتطع ``dataset`` حسب قناع منطقي عام، ويُعيد بناء ``asset_bounds``
    بإزاحات صحيحة (كل أصل يفقد عدداً مختلفاً من عيّناته حسب القناع)."""
    array_keys = ({'base_params', 'last_candles', 'is_live'}
                  | {f'X_{tf}' for tf in dataset['timeframes']}
                  | {f'y_{h}' for h in dataset['targets']})
    out: Dict = {k: v for k, v in dataset.items() if k not in array_keys | {'asset_bounds'}}

    out['base_params'] = dataset['base_params'][mask]
    out['last_candles'] = dataset['last_candles'][mask]
    if 'is_live' in dataset:
        out['is_live'] = dataset['is_live'][mask]
    for tf in dataset['timeframes']:
        out[f'X_{tf}'] = dataset[f'X_{tf}'][mask]
    for h in dataset['targets']:
        out[f'y_{h}'] = dataset[f'y_{h}'][mask]

    new_bounds, offset = [], 0
    for b in dataset.get('asset_bounds') or []:
        s, e = b['start'], b['end']
        kept = int(mask[s:e].sum())
        if kept:
            new_bounds.append({**{k: v for k, v in b.items() if k not in ('start', 'end')},
                               'start': offset, 'end': offset + kept})
            offset += kept
    out['asset_bounds'] = new_bounds
    return out


def drop_tail_per_asset(dataset: Dict, n: int) -> Dict:
    """يُسقط آخر ``n`` عيّنة من **كل أصل** على حدة — تُستخدَم داخلياً لحذف
    النوافذ المموَّهة بالدفعة الصورية (ميزاتها لا هدفها فقط) بعد بناء بيانات
    حيّة بـ ``stride=1``. عامة الاستخدام: تصلح لأي ``dataset`` فيه
    ``asset_bounds``."""
    return _apply_asset_mask(dataset, _asset_tail_mask(dataset, n, keep_tail=False))


def extract_last_batch(dataset: Dict, n: int = 1) -> Dict:
    """يستخلص آخر ``n`` عيّنة من **كل عملة** في مجموعة بيانات مُجهَّزة (ناتج
    ``build_dataset``/``build_dataset_live`` أياً كان مصدرها) — العيّنات
    الأحدث زمنياً لكل أصل. مع ``build_dataset_live`` هذه هي عيّنة **live**
    (بميزات حقيقية بالكامل وهدف صوري يجب تجاهله)؛ مع مجموعة تاريخية عادية
    هي ببساطة آخر عيّنة حقيقية لكل عملة."""
    result = _apply_asset_mask(dataset, _asset_tail_mask(dataset, n, keep_tail=True))
    print(f"📤 استُخلصت آخر {n} عيّنة لكل عملة عبر "
          f"{len(result.get('asset_bounds') or [])} أصل "
          f"({len(result['base_params']):,} عيّنة إجمالاً)")
    return result


# ══════════════════════════════════════════════════════════════════════════
# البناء الحيّ — إعادة استخدام كاملة لـ build_dataset_from_loader
# ══════════════════════════════════════════════════════════════════════════
def build_dataset_live(
    symbols: List[str],
    limit: Optional[int] = None,
    max_workers: Optional[int] = None,
    config: Optional[dict] = None,
    **kwargs,
) -> Dict:
    """يبني مجموعة بيانات حيّة من Binance Futures — عبر **نفس**
    :func:`build_dataset_from_loader` و:func:`prepare_single_asset` المستخدمتين
    في المسار التاريخي، بلا أي تغيير عليهما. الفرق الوحيد: ``load_asset_fn``
    يجلب من Binance (:func:`fetch_data`) بدل Drive، وتُستدعى المحاذاة بـ
    ``stride=1`` مع دفعة صورية في الذيل بدل قيمة ``CONFIG['stride']``
    المعتادة (لمرة هذا الاستدعاء فقط — ``CONFIG`` لا يتغيّر).

    آخر عيّنة لكل رمز في الناتج **حقيقية الميزات بالكامل** لكن بهدف صوري —
    استخلصها بـ ``extract_last_batch(dataset, n=1)``. كل ما قبلها أهدافه
    حقيقية 100% (اختبار فوري على أحدث بيانات لم يرها التدريب).

    Args:
        symbols: رموز Binance Futures، مثل ``['BTCUSDT', 'ETHUSDT']``.
        limit: عدد الشموع **الحقيقية** المطلوب جلبها لكل رمز. ``None`` يحسبها
            تلقائياً (هامش يكفي أطول نافذة + إحماء الميزات).
        download_interval: فريم Binance الفعلي المطلوب جلبه (افتراضياً
            ``CONFIG['download_interval']``، '1h') — مستقلّ عن ``tf_order``/
            فريم النموذج؛ يُجمَّع (resample) صعوداً إليه بعد الجلب.
        **kwargs: أي معامل آخر يقبله ``build_dataset_from_loader`` (مثل
            ``targets``، ``forecast_horizon``...) يمرَّر كما هو.
    """
    config = CONFIG if config is None else config
    tf_order = kwargs.get('tf_order') or config['tf_order']
    window_sizes = kwargs.get('window_sizes') or config['window_sizes']
    forecast_horizon = kwargs.get('forecast_horizon') or config['forecast_horizon']
    download_interval = kwargs.pop('download_interval', None) or config.get('download_interval', '1h')

    pad_n = _live_pad_length(tf_order, window_sizes, forecast_horizon, download_interval)
    real_limit = limit or _auto_live_limit(tf_order, window_sizes, forecast_horizon, pad_n,
                                           download_interval)

    def _live_loader(file_id, name):
        df = fetch_data(name, download_interval, real_limit)
        if df is None:
            raise RuntimeError(f"تعذّر جلب {name} من Binance")
        df = df.set_index('timestamp').sort_index()
        return _pad_with_dummy_tail(df, download_interval, pad_n)

    configs = [{'name': s} for s in symbols]
    resample_fn = make_resample_fn(config)

    print(f"📡 بناء بيانات حيّة لـ {len(symbols)} رمز "
          f"(limit={real_limit} شمعة حقيقية + {pad_n} صورية، stride=1)...")
    raw = build_dataset_from_loader(
        configs, load_asset_fn=_live_loader, resample_fn=resample_fn,
        stride=1, config=config, max_workers=max_workers, **kwargs,
    )
    # ✅ عدد نوافذ *النموذج* (وحدة tf_order[0]) الملوَّثة في الذيل — مستقلّ
    # تماماً عن download_interval وpad_n (تينك بوحدة تحميل مختلفة قد لا تساوي
    # وحدة النموذج بعد الآن): يعتمد فقط على نِسَب فريمات tf_order فيما بينها
    # (max(ratios) — انظر _live_contamination_len)، وهو ما ينجو من كسر الحلقة
    # في prepare_single_asset رغم أن مركزه (end_idx) نفسه داخل المنطقة
    # الصورية. هذه فقط ميزاتها ملوَّثة فعلياً (مركزها شمعة صورية) ويجب حذفها؛
    # العيّنة الحقيقية المطلوبة (مركزها آخر شمعة حقيقية) هي التي تسبقها مباشرة.
    n_contaminated = _live_contamination_len(tf_order)
    dataset = drop_tail_per_asset(raw, n_contaminated)

    # ✅ شبكة أمان: الدفعة الصورية شموع مسطّحة تماماً (high=low=close) بالبناء —
    # حالة تكاد تستحيل في بيانات سوق حقيقية. إن ظهرت رغم الحذف أعلاه، هذا
    # يعني أن الصيغة الحسابية لم تُطابق حالتك (تهيئة تعدّد فريمات غير معتادة)
    # فيُحذَّر صراحة بدل إرجاع عيّنة "live" ملوَّثة الميزات بصمت.
    last = dataset['last_candles']
    for b in dataset.get('asset_bounds') or []:
        if b['end'] <= b['start']:
            continue
        i = b['end'] - 1
        if last[i, 0] == last[i, 1] == last[i, 2]:
            print(f"   ⚠️ {b['name']}: العيّنة الأخيرة ما زالت تبدو صورية "
                  f"(high=low=close) — راجع forecast_horizon/tf_order/window_sizes.")

    n_assets = len(dataset.get('asset_bounds') or [])
    print(f"✅ بيانات حيّة جاهزة: {len(dataset['base_params']):,} عيّنة عبر "
          f"{n_assets} أصل (آخر عيّنة لكل أصل = live بهدف صوري، البقية اختبار فوري)")
    return dataset


تخزين وتحميل البيانات


In [ ]:
# @title
"""
حفظ/تحميل مجموعة البيانات المُجهَّزة من/إلى Google Drive.

النسخة السابقة كانت تحمل دالتين غير متوافقتين فعلياً:
  * ``save_data_to_drive`` تحفظ عبر Drive المُركَّب (``drive.mount``)، بمسار
    مُوسَّم بالوقت فقط — بلا نسخة بمسار ثابت (سطر ``save_latest`` كان
    مُعلَّقاً/معطَّلاً)، فلا توجد طريقة آلية لمعرفة أحدث ملف لاحقاً.
  * ``load_preprocessed_data_from_drive`` تُحمِّل عبر ``gdown`` + رابط
    مشاركة عام (``file_id``) — مسار مختلف تماماً عن Drive المُركَّب،
    ولا يستطيع قراءة ما تكتبه ``save_data_to_drive`` إطلاقاً.

هنا: الحفظ يكتب نسخة مؤرَّخة **ونسخة `_latest` فعلية** بنفس المسار
المُركَّب (``mount_drive()``، الدالة الموحّدة المستخدمة أصلاً لأرشيف
funding/OI في هذا الدفتر) — والتحميل يقرأ من نفس المسار المُركَّب مباشرة،
لا عبر رابط عام. ``load_preprocessed_data_from_drive`` (gdown/file_id)
أُبقيت للتوافق الخلفي فقط مع من لا يزال يستخدم ملفاً عاماً مُشارَكاً —
لا تستخدمها لقراءة ما يكتبه ``save_data_to_drive``.
"""
import gzip
import pickle
import os
from pathlib import Path
from datetime import datetime
from typing import Any, Optional


def save_data_to_drive(
    data: Any,
    project_name: Optional[str] = None,
    data_type: str = "preprocessed_data",
    filename_base: str = "preprocessing_output",
    save_latest: bool = True,
    config: Optional[dict] = None,
) -> Path:
    """يحفظ ``data`` إلى Drive المُركَّب: نسخة مؤرَّخة دائماً، ونسخة
    ``{filename_base}_latest.pkl.gz`` بمسار ثابت إن ``save_latest=True``
    (هذا ما يقرؤه ``load_data_from_drive`` افتراضياً).

    ``project_name=None`` يستخدم ``config['project_name']`` (افتراضياً
    ``CONFIG``) بدل اسم عام ثابت — بحيث يُحفَظ فعلياً تحت مشروعك، لا تحت
    ``trading_project`` بغضّ النظر عن إعداداتك.
    """
    config = CONFIG if config is None else config
    project_name = project_name or config.get("project_name", "crypto_model")

    root = mount_drive(config=config)
    base_dir = (root / project_name / data_type) if root is not None else Path(f"./{project_name}/{data_type}")
    base_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    versioned_path = base_dir / f"{filename_base}_{timestamp}.pkl.gz"
    with gzip.open(versioned_path, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    if save_latest:
        latest_path = base_dir / f"{filename_base}_latest.pkl.gz"
        with gzip.open(latest_path, "wb") as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    v_size = versioned_path.stat().st_size / (1024 * 1024)
    print(f"✅ تم الحفظ: {versioned_path.name} ({v_size:.2f} MB)"
          + (f" — ونسخة latest بنفس المجلد" if save_latest else ""))
    return versioned_path


def load_data_from_drive(
    project_name: Optional[str] = None,
    data_type: str = "preprocessed_data",
    filename_base: str = "preprocessing_output",
    filename: Optional[str] = None,
    config: Optional[dict] = None,
) -> Any:
    """يقرأ ما كتبته ``save_data_to_drive`` من Drive المُركَّب — بلا
    ``gdown`` ولا رابط عام، بنفس مسار الحفظ بالضبط.

    ``filename=None`` (الافتراضي) يقرأ ``{filename_base}_latest.pkl.gz``.
    مرّر ``filename`` صراحةً (مثلاً اسم نسخة مؤرَّخة بعينها من مخرَج
    ``save_data_to_drive`` السابق) لتحميل نسخة محدَّدة بدل الأحدث.
    """
    config = CONFIG if config is None else config
    project_name = project_name or config.get("project_name", "crypto_model")

    root = mount_drive(config=config)
    base_dir = (root / project_name / data_type) if root is not None else Path(f"./{project_name}/{data_type}")
    path = base_dir / (filename or f"{filename_base}_latest.pkl.gz")

    if not path.exists():
        raise FileNotFoundError(
            f"❌ لا يوجد ملف محفوظ في {path} — شغّل save_data_to_drive أولاً، "
            f"أو مرّر filename لاسم نسخة مؤرَّخة موجودة فعلاً في {base_dir}."
        )

    with gzip.open(path, "rb") as f:
        data = pickle.load(f)

    print(f"✅ تم التحميل من {path.name} — نوع: {type(data)}"
          + (f"، المفاتيح: {len(data)}" if isinstance(data, dict) else ""))
    return data


def load_preprocessed_data_from_drive(
    file_id: str,
    output_filename: str = 'preprocessing_output.pkl.gz',
    download_dir: str = '.',
    quiet: bool = False,
    cleanup: bool = False
) -> Any:
    """[توافق خلفي فقط] تحميل ملف عام مُشارَك عبر رابط Drive (``gdown``).

    لا تستخدمها لقراءة مخرَج ``save_data_to_drive`` — استخدم
    ``load_data_from_drive`` بدلاً منها؛ هذه الدالة تحتاج ``file_id``
    لملف مُشارَك بامتياز "Anyone with the link"، وهو أسلوب مختلف تماماً
    عن Drive المُركَّب."""
    import gdown

    output_path = os.path.join(download_dir, output_filename)
    url = f'https://drive.google.com/uc?id={file_id}'

    try:
        print(f"🔄 جاري التحميل من Google Drive...")
        gdown.download(url, output_path, quiet=quiet)

        if not os.path.exists(output_path):
            raise FileNotFoundError(f"فشل تحميل الملف")

        with gzip.open(output_path, 'rb') as f:
            data = pickle.load(f)

        print(f"✅ تم التحميل! نوع: {type(data)}")
        if isinstance(data, dict):
            print(f"🔑 المفاتيح: {len(data)}")

        return data

    finally:
        if cleanup and os.path.exists(output_path):
            os.remove(output_path)


## 19) ملخص الدوال المتاحة

كل الأسماء التالية معرَّفة الآن مباشرة في هذا الدفتر (بلا أي `import` من
حزمة خارجية بخلاف `numpy`/`pandas`/`pandas_ta_classic`/`python-binance`/`ccxt`):

**الإعدادات** — `DEFAULT_CONFIG`, `CONFIG`, `STATIC_INDICATORS_FULL`,
`COINS_BY_CATEGORY`, `DEFAULT_ENABLED_CATEGORIES`, `get_config`,
`update_config`, `reset_config`, `save_config`, `load_config`,
`refresh_features`, `feature_order`, `n_features`, `seq_len`, `describe`

**الرؤوس** — `HEAD_KINDS`, `OUTPUT_PREFIX`, `output_name`, `head_name`,
`get_output_names`, `get_target_heads`, `split_head_name`,
`is_regression_head`, `is_class_head`, `get_class_heads`, `get_reg_heads`,
`paired_targets`, `abstention_cfg`, `abstention_enabled`,
`uncertainty_head_enabled`, `signal_enabled`, `decision_head`,
`validate_head_config`, `head_loss_weights`

**الميزات** — `add_features`, `infer_feature_columns`, `extract_features`,
`price_indices`, `add_custom_features`, `custom_feature_names`,
`DEFAULT_CUSTOM_SETTINGS`

**التطبيع** — `FEATURE_KINDS`, `CLIP_ABS`, `classify_feature`, `classify_column`,
`describe_features`, `audit_normalization`, `normalize_column`,
`calc_scale_params`, `scale_data`, `inverse_scale`, `process_window`,
`process_windows`

**المحاذاة والنوافذ** — `align_multi_timeframes_time_based`,
`prepare_single_asset` (**نفسها تُستخدَم للتاريخي والحيّ**), `LAST_COLUMNS`,
`TARGET_COLUMNS`

**خط الأنابيب** — `build_dataset`, `build_dataset_from_loader`,
`build_dataset_from_preloaded`, `prepare_regression_multiframe_multiasset`

**التقسيم** — `split_data`, `SPLIT_MODES`, `embargo_size`, `embargo_candles`,
`embargo_duration`, `resolve_split_dates`, `sample_timestamps`,
`add_y_prefix`, `strip_y_prefix`

**المصادر التاريخية (Google Drive حصراً)** — `load_asset`, `load_asset2`
(نفس الدالة)، `load_asset_registry`, `resample_timeframes`,
`make_resample_fn`, `filter_desired_coins`, `flatten_coins_by_category`,
`ALL_GLOBAL`, `mount_drive`, `is_colab`, `exclude_coins`,
`split_excluded_coins` (استبعاد عملات بعينها — انظر `CONFIG['excluded_coins']`)

**البيانات الحيّة (Binance Futures)** — `client`, `_create_client`,
`get_normalized_futures_symbols`, `fetch_data`, `API_KEY`, `API_SECRET`,
`BINANCE_MAX_LIMIT`, `LIVE_FEATURE_WARMUP`, `build_dataset_live`
(تستدعي `build_dataset_from_loader` نفسها — بلا نسخة موازية)،
`extract_last_batch`, `drop_tail_per_asset`

**التوازي** — `load_assets`, `imap_ordered`, `default_workers`

**التشخيص** — `diagnose_feature_availability`, `diagnose_data_vs_configs`,
`check_missing_values`, `summarize_dataset`

**نقاط الاستئناف** — `clear_checkpoint` (checkpoint_dir في build_dataset_from_loader)

**السياق السوقي** — `market_context_columns`, `build_market_context`, `add_market_context`

**إعادة تدريب دورية** — `rolling_split_schedule`, `rolling_splits`

**معدّل التمويل/الفائدة المفتوحة** — `fetch_funding_rate`, `fetch_open_interest_hist`, `save_funding_open_interest`, `load_funding_open_interest`

**حفظ/تحميل مجموعة البيانات المُجهَّزة (Drive المُركَّب)** — `save_data_to_drive` (نسخة مؤرَّخة + `_latest`)، `load_data_from_drive` (القراءة المقابلة، بلا رابط عام). `load_preprocessed_data_from_drive` توافق خلفي فقط (gdown/file_id لملف مُشارَك — لا يقرأ مخرَج `save_data_to_drive`).

**التقسيم** — `compute_global_cutoff`, `extract_test_dates`, `build_leak_free_split` (حدّ فاصل بديل بنسبة عيّنات خام — انظر `split_cutoff_method`)

**أهداف الانحدار** — `invert_reg_predictions` (عكس `reg_target_mode`), `cross_sectional_normalize`, `invert_cross_sectional` (تطبيع مقطعي اختياري عبر الأصول)

**الميزات** — `available_features`, `exclude_features` (استبعاد ميزات بالاسم/النمط من المدخلات)

**الجلب الحيّ** — `fetch_data` (تقسيم على عدة طلبات)، `RateLimiter`, `get_request_limiter` (حدّ `live_max_requests_per_minute`، 2000 افتراضياً)

**الاختبارات الذاتية** — `run_pipeline_selftests` (القسم التالي؛ بلا Drive ولا شبكة)


## 19-ب) اختبارات ذاتية

تتحقّق من تعديلات هذه النسخة (استبعاد العملات، ميزات الساعة، التقسيم) على بيانات
تركيبية صغيرة — **بلا Drive ولا شبكة**، فتعمل قبل أي تحميل حقيقي. شغّلها بعد أي
تعديل على الدوال أعلاه؛ الفشل يرفع `AssertionError` باسم الاختبار الفاشل.


In [53]:
# @title
"""
اختبارات ذاتية لتعديلات هذا الدفتر — بلا Drive ولا شبكة (بيانات تركيبية صغيرة).

كل اختبار يُثبت سلوكاً محدَّداً كان مكسوراً أو غير موجود:
  * استبعاد العملات (``exclude_coins`` / ``CONFIG['excluded_coins']``).
  * غياب ``TIME_hour_*`` على فريم يومي (ثابتتان بلا معلومة) وبقاؤهما داخل اليوم.
  * ``fetch_data``: تقسيم الجلب على عدة طلبات ودمجها (عميل Binance وهمي).
  * حدّ تقسيم بديل (``compute_global_cutoff``/``build_leak_free_split``) ونسبته الخام.
  * فصل فريم التحميل الحيّ (download_interval) عن فريم النموذج.
  * ذروة رام أقلّ في align_multi_timeframes_time_based/process_windows، وسقف
    خيوط احترازي بالرام، ونقاط استئناف لكل أصل (checkpoint/resume).
  * سياق سوقي عابر للأصول (add_market_context) ومحاذاته وتصفيره للمرجع.
  * إعادة تدريب دورية (rolling_split_schedule/rolling_splits) بلا تسرّب بين النوافذ.
  * معدّل التمويل والفائدة المفتوحة: الجلب المُقسَّم وتراكم أرشيف Drive.
  * هدف الانحدار كعائد مباشر (``reg_target_mode``) وعكسه، والتطبيع المقطعي
    (``cross_sectional_normalize``) وعكسه — أفكار من Qlib.
  * ``exclude_features`` وحدّ الطلبات (``RateLimiter``) وتمريره في ``fetch_data``.
  * ``split_data``: val و test غير فارغين على توزيع مكدَّس زمنياً، مع احترام
    فجوة العزل، وخطأ صريح (لا قسم فارغ صامت) حين يستحيل التقسيم.

شغّلها بعد أي تعديل على الدوال أعلاه: ``run_pipeline_selftests()``.
"""
import contextlib
import io
import threading


def run_pipeline_selftests(verbose: bool = True) -> bool:
    """يشغّل كل الاختبارات؛ يرفع ``AssertionError`` بملخّص الفاشل إن فشل أي منها."""

    def _cfg(**over) -> dict:
        cfg = deepcopy(CONFIG)
        cfg.update(tf_order=['1D'], base_tf='1D', window_sizes={'1D': 32}, stride=1,
                   feature_order=None, split_dates=None, excluded_coins=[],
                   min_split_samples=64, split_mode='global_time',
                   train_pct=0.90, val_pct=0.05, keep_asset_test_separate=False,
                   forecast_horizon=1, embargo_candles=None,
                   market_context={'enabled': False}, download_interval='1h',
                   funding_rate={'enabled': False}, open_interest={'enabled': False})
        cfg.update(over)
        return cfg

    def _ohlcv(days: int, seed: int = 0) -> pd.DataFrame:
        rng = np.random.default_rng(seed)
        n = days * 24
        idx = pd.date_range('2025-01-01', periods=n, freq='h', tz='UTC')
        close = 100 * np.exp(np.cumsum(rng.normal(0, 0.004, n)))
        open_ = np.r_[close[0], close[:-1]]
        return pd.DataFrame({
            'open': open_,
            'high': np.maximum(open_, close) * (1 + rng.random(n) * 0.003),
            'low': np.minimum(open_, close) * (1 - rng.random(n) * 0.003),
            'close': close, 'volume': rng.random(n) * 1000 + 50}, index=idx)

    def _fake_dataset(lengths, stride: int = 16) -> dict:
        """مجموعة بيانات هيكلية بأطوال عيّنات متفاوتة تنتهي كلها في نفس اليوم —
        يحاكي سجلاً فيه قليل من العملات القديمة وكثير من الحديثة (تكدّس زمني)."""
        day = 86_400 * 10 ** 9
        end = pd.Timestamp('2026-09-15', tz='UTC').value
        ts, bounds, off = [], [], 0
        for k, w in enumerate(lengths):
            ts.append(end - (w - 1 - np.arange(w)) * stride * day)
            bounds.append({'name': f'A{k}', 'start': off, 'end': off + w})
            off += w
        ts = np.concatenate(ts).astype('float64')
        N = len(ts)
        last = np.zeros((N, len(LAST_COLUMNS)), dtype='float64')
        last[:, TS_COL] = ts
        return {'base_params': np.zeros((N, 2), 'float32'), 'last_candles': last,
                'X_1D': np.zeros((N, 2, 1), 'float32'),
                'y_close_class': np.ones(N, 'float32'),
                'timeframes': ['1D'], 'targets': ['close_class'], 'base_tf': '1D',
                'window_sizes': {'1D': 32}, 'forecast_horizon': 1, 'stride': stride,
                'asset_bounds': bounds}

    def _quiet(fn, *a, **k):
        with contextlib.redirect_stdout(io.StringIO()):
            return fn(*a, **k)

    # ── استبعاد العملات ────────────────────────────────────────────────────
    def t_exclude_matching():
        configs = [{'name': 'BTCUSDT'}, {'name': 'TSLAUSDT'},
                   {'name': ' ethusdt '}, {'name': 'XAUUSDT'}]
        kept = exclude_coins(configs, ['tslausdt', ' XAUUSDT ', 'GHOSTUSDT'], verbose=False)
        assert [c['name'] for c in kept] == ['BTCUSDT', ' ethusdt '], kept
        assert len(configs) == 4, "configs الأصلية عُدِّلت"
        _, removed, missing = split_excluded_coins(
            configs, ['tslausdt', 'XAUUSDT', 'GHOSTUSDT'])
        assert removed == ['TSLAUSDT', 'XAUUSDT'], removed
        assert missing == ['GHOSTUSDT'], missing           # إملاء خاطئ لا يمرّ بصمت

    def t_exclude_edge_cases():
        configs = [{'name': 'BTCUSDT'}, {'name': 'BTCDOMUSDT'}]
        assert len(exclude_coins(configs, [], verbose=False)) == 2
        # نصّ واحد = اسم واحد (لا يُفكَّك إلى أحرف)
        assert [c['name'] for c in exclude_coins(configs, 'BTCUSDT', verbose=False)] == ['BTCDOMUSDT']
        # الجذر لا يطابق: 'BTC' لا تستبعد BTCUSDT ولا BTCDOMUSDT
        assert len(exclude_coins(configs, ['BTC'], verbose=False)) == 2
        # عناصر نصّية بدل قواميس
        assert exclude_coins(['A', 'B'], ['a'], verbose=False) == ['B']
        # None → CONFIG['excluded_coins']؛ قائمة صريحة فارغة تتجاوزه
        cfg = {'excluded_coins': ['BTCUSDT']}
        assert [c['name'] for c in exclude_coins(configs, None, config=cfg, verbose=False)] == ['BTCDOMUSDT']
        assert len(exclude_coins(configs, [], config=cfg, verbose=False)) == 2

    def t_exclude_in_pipeline():
        cfg = _cfg()
        names = ['AAAUSDT', 'BBBUSDT', 'CCCUSDT']
        data = {n: _quiet(resample_timeframes, _ohlcv(200, seed=i), ['1D'], config=cfg)
                for i, n in enumerate(names)}
        ds = _quiet(build_dataset_from_preloaded, [{'name': n} for n in names], data,
                    config=cfg, max_workers=1, exclude=['bbbusdt'])
        got = [b['name'] for b in ds['asset_bounds']]
        assert got == ['AAAUSDT', 'CCCUSDT'], got
        assert ds['skipped_assets'].get('excluded') == ['BBBUSDT'], ds['skipped_assets']
        # وعبر CONFIG بلا وسيط
        cfg2 = _cfg(excluded_coins=['AAAUSDT'])
        ds2 = _quiet(build_dataset_from_preloaded, [{'name': n} for n in names], data,
                     config=cfg2, max_workers=1)
        assert [b['name'] for b in ds2['asset_bounds']] == ['BBBUSDT', 'CCCUSDT']

    # ── ميزات الساعة ───────────────────────────────────────────────────────
    def t_hour_features_daily():
        cfg = _cfg()
        names = custom_feature_names(config=cfg)
        assert 'TIME_hour_sin' not in names and 'TIME_hour_cos' not in names, names
        assert 'TIME_dow_sin' in names and 'TIME_dow_cos' in names
        daily = _ohlcv(120).resample('1D').agg(OHLCV_AGG)
        out = add_custom_features(daily, config=cfg)
        assert not any(c.startswith('TIME_hour') for c in out.columns), list(out.columns)
        assert set(names) <= set(out.columns), set(names) - set(out.columns)
        feats = _quiet(infer_feature_columns, cfg)
        assert not any(f.startswith('TIME_hour') for f in feats), feats

    def t_hour_features_intraday():
        cfg = _cfg(tf_order=['1h'], base_tf='1h', window_sizes={'1h': 32})
        names = custom_feature_names(config=cfg)
        assert {'TIME_hour_sin', 'TIME_hour_cos'} <= set(names), names
        out = add_custom_features(_ohlcv(10), config=cfg)
        assert set(names) <= set(out.columns)
        assert out['TIME_hour_sin'].nunique() > 1          # فعلاً تتغيّر داخل اليوم
        # فريم مختلط (داخل اليوم + يومي): تبقى لأن أصغر فريم داخل اليوم
        mixed = _cfg(tf_order=['4h', '1D'], base_tf='4h', window_sizes={'4h': 32, '1D': 32})
        assert 'TIME_hour_sin' in custom_feature_names(config=mixed)

    # ── التقسيم ────────────────────────────────────────────────────────────
    #: أطوال عيّنات تحاكي سجلاً حقيقياً: قليل قديم وكثير حديث، كلها تنتهي معاً.
    CROWDED = [140] * 5 + [120] * 20 + [45] * 80 + [18] * 150 + [5] * 120

    def t_split_crowded_timeline():
        cfg = _cfg()
        ds = _fake_dataset(CROWDED)
        train, val, test = _quiet(split_data, ds, config=cfg)
        n_tr, n_va, n_te = (len(s['base_params']) for s in (train, val, test))
        assert min(n_tr, n_va, n_te) >= 64, (n_tr, n_va, n_te)     # كان val=test=0
        kept = n_tr + n_va + n_te
        assert 0.80 <= n_tr / kept <= 0.95, n_tr / kept
        gap = embargo_duration(ds, cfg)
        t_tr, t_va, t_te = (sample_timestamps(s) for s in (train, val, test))
        assert t_va.min() - t_tr.max() >= gap, (t_va.min() - t_tr.max(), gap)
        assert t_te.min() - t_va.max() >= gap, (t_te.min() - t_va.max(), gap)

    def t_split_uniform_matches_requested():
        # عيّنات موزَّعة بانتظام: النسب المُبقاة تقارب 90/5/5 (بعد اقتطاع العزل)
        cfg = _cfg()
        ds = _fake_dataset([3000] * 3, stride=1)
        train, val, test = _quiet(split_data, ds, config=cfg)
        kept = sum(len(s['base_params']) for s in (train, val, test))
        assert abs(len(train['base_params']) / kept - 0.90) < 0.03
        assert abs(len(val['base_params']) / kept - 0.05) < 0.03

    def t_split_raises_when_impossible():
        cfg = _cfg()
        ds = _fake_dataset([10] * 5)                       # 50 عيّنة فقط < الحدّ الأدنى
        try:
            _quiet(split_data, ds, config=cfg)
        except ValueError as exc:
            assert 'split_dates' in str(exc)
        else:
            raise AssertionError("كان يجب أن يرفع ValueError بدل قسم فارغ")

    def t_split_explicit_dates_guarded():
        cfg = _cfg(split_dates={'train_end': '2025-06-01', 'val_end': '2030-01-01'})
        ds = _fake_dataset(CROWDED)                         # test سيخرج فارغاً
        try:
            _quiet(split_data, ds, config=cfg)
        except ValueError as exc:
            assert 'test' in str(exc)
        else:
            raise AssertionError("تواريخ صريحة تُنتج test فارغاً يجب أن ترفع ValueError")

    # ── fetch_data: الجلب المُقسَّم على عدة طلبات ───────────────────────────
    class _FakeFuturesClient:
        """عميل وهمي يحاكي futures_klines: حدّ 1500، endTime، وتاريخ محدود الطول."""
        H = 3_600_000

        def __init__(self, n_total, end_ms=1_800_000_000_000, fail=None):
            self.n, self.end_ms, self.calls = n_total, end_ms, []
            self.ok = 0                     # عدد الصفحات التي نجحت حتى الآن
            self._fails_left = dict(fail or {})   # {رقم الصفحة: مرات الفشل قبل النجاح (99 = دائم)}

        def ts(self, i):                    # طابع الشمعة i (0 = الأقدم)
            return self.end_ms - (self.n - 1 - i) * self.H

        def futures_klines(self, symbol, interval, limit, endTime=None):
            self.calls.append({'limit': limit, 'endTime': endTime, 'interval': interval})
            page = self.ok + 1
            assert limit <= BINANCE_MAX_LIMIT, f"طلب {limit} > الحدّ الأقصى"
            if self._fails_left.get(page, 0) > 0:
                self._fails_left[page] -= 1
                raise RuntimeError("simulated network error")
            self.ok += 1
            idx = [i for i in range(self.n) if endTime is None or self.ts(i) <= endTime]
            return [[self.ts(i), "1", "2", "0.5", "1.5", "10", self.ts(i) + self.H - 1,
                     "0", 0, "0", "0", "0"] for i in idx[-limit:]]

    def _with_client(fake, fn):
        old_client, old_sleep = globals().get('client'), time.sleep
        globals()['client'], time.sleep = fake, (lambda *_: None)
        try:
            return _quiet(fn)
        finally:
            globals()['client'], time.sleep = old_client, old_sleep

    def t_fetch_single_page_unchanged():
        fake = _FakeFuturesClient(5000)
        df = _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 300))
        assert len(fake.calls) == 1 and fake.calls[0]['endTime'] is None, fake.calls
        assert list(df.columns) == ['timestamp', 'open', 'high', 'low', 'close', 'volume']
        assert len(df) == 300 and str(df['timestamp'].dt.tz) == 'UTC'

    def t_fetch_paginates_and_merges():
        fake = _FakeFuturesClient(10_000)
        df = _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 4000))
        assert [c['limit'] for c in fake.calls] == [1500, 1500, 1000], fake.calls
        assert len(df) == 4000, len(df)
        assert df['timestamp'].is_monotonic_increasing and not df['timestamp'].duplicated().any()
        step = df['timestamp'].diff().dropna().unique()
        assert len(step) == 1 and step[0] == pd.Timedelta(hours=1), step   # بلا ثقوب ولا تداخل
        assert df['timestamp'].iloc[-1].value // 10 ** 6 == fake.ts(9_999)   # آخر شمعة = الأحدث
        assert df['timestamp'].iloc[0].value // 10 ** 6 == fake.ts(6_000)    # وأقدمها ٤٠٠٠ شمعة للخلف

    def t_fetch_exact_multiple_of_max():
        fake = _FakeFuturesClient(10_000)
        df = _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 3000))
        assert [c['limit'] for c in fake.calls] == [1500, 1500] and len(df) == 3000

    def t_fetch_history_shorter_than_limit():
        fake = _FakeFuturesClient(2000)              # عملة حديثة: 2000 شمعة فقط
        df = _with_client(fake, lambda: fetch_data('NEWUSDT', '1h', 5000))
        assert len(df) == 2000 and len(fake.calls) == 2, (len(df), fake.calls)

    def t_fetch_retries_transient_page_error():
        fake = _FakeFuturesClient(10_000, fail={2: 1})   # فشل عابر واحد في الصفحة الثانية
        df = _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 3000, retries=3))
        assert df is not None and len(df) == 3000 and len(fake.calls) == 3, len(fake.calls)

    def t_fetch_returns_none_on_persistent_page_failure():
        fake = _FakeFuturesClient(10_000, fail={2: 99})
        df = _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 3000, retries=3))
        assert df is None                                # لا نتيجة مبتورة صامتة

    def t_fetch_rejects_nonpositive_limit():
        try:
            _with_client(_FakeFuturesClient(10), lambda: fetch_data('BTCUSDT', '1h', 0))
        except ValueError:
            return
        raise AssertionError("limit=0 يجب أن يرفع ValueError")

    # ── استبعاد الميزات ────────────────────────────────────────────────────
    def t_exclude_features_names_and_patterns():
        cfg = _cfg()
        base = _quiet(feature_order, cfg)
        assert 'RSI_14' in base and any(f.startswith('BB') for f in base), base
        added = _quiet(exclude_features, ['rsi_14', 'BB*', 'GHOST_1'], config=cfg)
        assert 'RSI_14' in added and all(a == 'RSI_14' or a.startswith('BB') for a in added), added
        after = cfg['feature_order']
        assert 'RSI_14' not in after and not any(f.startswith('BB') for f in after)
        assert len(after) == len(base) - len(added)           # feature_order حُدِّثت
        assert 'GHOST_1' not in cfg['exclude_from_features']  # الاسم المجهول لا يُضاف بصمت

    def t_exclude_features_idempotent_and_guard():
        cfg = _cfg()
        _quiet(exclude_features, ['EMA_9'], config=cfg)
        n = len(cfg['exclude_from_features'])
        assert _quiet(exclude_features, ['ema_9'], config=cfg) == []       # لا تكرار
        assert len(cfg['exclude_from_features']) == n
        try:
            _quiet(exclude_features, ['*'], config=cfg)                     # سيُفرغ كل شيء
        except ValueError:
            assert len(cfg['exclude_from_features']) == n                  # بلا تعديل
        else:
            raise AssertionError("استبعاد كل الميزات يجب أن يرفع ValueError")

    def t_exclude_features_reaches_dataset():
        cfg = _cfg()
        _quiet(exclude_features, ['EMA_26', 'STOCH*'], config=cfg)
        names = ['AAAUSDT', 'BBBUSDT']
        data = {n: _quiet(resample_timeframes, _ohlcv(200, seed=i), ['1D'], config=cfg)
                for i, n in enumerate(names)}
        ds = _quiet(build_dataset_from_preloaded, [{'name': n} for n in names], data,
                    config=cfg, max_workers=1)
        assert 'EMA_26' not in ds['feature_order']
        assert not any(f.startswith('STOCH') for f in ds['feature_order'])
        assert ds['X_1D'].shape[-1] == len(cfg['feature_order']) == len(ds['feature_order'])

    # ── محدِّد معدّل الطلبات ───────────────────────────────────────────────
    def t_ratelimiter_sliding_window():
        clock, slept = [0.0], []

        def _sleep(s):
            slept.append(s)
            clock[0] += s

        rl = RateLimiter(3, 10.0, clock=lambda: clock[0], sleep=_sleep)
        assert [rl.acquire() for _ in range(3)] == [0.0, 0.0, 0.0] and not slept
        assert abs(rl.acquire() - 10.0) < 1e-9                  # الرابع ينتظر خروج أقدمها
        for t in (12.0, 14.0):
            clock[0] = t
            assert rl.acquire() == 0.0
        clock[0] = 15.0                                         # نافذة [10,12,14] ممتلئة
        assert abs(rl.acquire() - 5.0) < 1e-9 and clock[0] == 20.0

    def t_ratelimiter_threadsafe_never_exceeds():
        rl, stamps, lock = RateLimiter(8, 0.4), [], threading.Lock()

        def worker():
            for _ in range(8):
                rl.acquire()
                with lock:
                    stamps.append(time.monotonic())

        threads = [threading.Thread(target=worker) for _ in range(5)]   # 40 طلباً
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        stamps.sort()
        assert len(stamps) == 40
        # أي 9 طلبات متتالية لا تقع داخل فترة أقصر من النافذة (هامش لتأخّر تسجيل الطابع)
        worst = min(stamps[i + 8] - stamps[i] for i in range(len(stamps) - 8))
        assert worst >= 0.4 - 0.1, worst
        assert stamps[-1] - stamps[0] >= 4 * 0.4 - 0.15          # 5 دفعات ⇒ ≥ 4 نوافذ

    def t_default_request_limit_is_2000_per_minute():
        assert DEFAULT_CONFIG['live_max_requests_per_minute'] == 2000
        cfg = {'live_max_requests_per_minute': 2000}
        a = get_request_limiter(cfg)
        assert a.max_calls == 2000 and a.period == 60.0
        assert get_request_limiter(cfg) is a                    # نافذة واحدة مشتركة
        assert get_request_limiter({'live_max_requests_per_minute': 500}).max_calls == 500
        assert get_request_limiter({'live_max_requests_per_minute': 0}) is None

    class _SpyLimiter:
        def __init__(self):
            self.n = 0

        def acquire(self):
            self.n += 1
            return 0.0

    def t_fetch_acquires_per_request_including_retries():
        spy, fake = _SpyLimiter(), _FakeFuturesClient(10_000)
        _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 4000, rate_limiter=spy))
        assert spy.n == len(fake.calls) == 3, (spy.n, len(fake.calls))     # كل صفحة طلب
        spy2, fake2 = _SpyLimiter(), _FakeFuturesClient(10_000, fail={2: 1})
        _with_client(fake2, lambda: fetch_data('BTCUSDT', '1h', 3000, retries=3,
                                               rate_limiter=spy2))
        assert spy2.n == len(fake2.calls) == 3, (spy2.n, len(fake2.calls))  # وكل إعادة محاولة

    def t_fetch_is_actually_throttled():
        fake = _FakeFuturesClient(10_000)
        t0 = time.monotonic()
        df = _with_client(fake, lambda: fetch_data('BTCUSDT', '1h', 4500,
                                                   rate_limiter=RateLimiter(2, 0.3)))
        assert len(fake.calls) == 3 and len(df) == 4500
        assert time.monotonic() - t0 >= 0.25        # الطلب الثالث انتظر نافذة الحدّ (2 لكل 0.3ث)

    # ── الهدف كعائد مباشر (Qlib idea 1) ─────────────────────────────────────
    def _price_asset(n=60, seed=0):
        """سلسلة يومية واقعية (مشي عشوائي هندسي) — كافية لإحماء كل المؤشرات."""
        rng = np.random.default_rng(seed)
        idx = pd.date_range('2025-01-01', periods=n, freq='D', tz='UTC')
        close = 100 * np.exp(np.cumsum(rng.normal(0, 0.012, n)))
        open_ = np.r_[close[0], close[:-1]]
        high = np.maximum(open_, close) * (1 + rng.random(n) * 0.006)
        low = np.minimum(open_, close) * (1 - rng.random(n) * 0.006)
        return pd.DataFrame({'open': open_, 'high': high, 'low': low, 'close': close,
                             'volume': rng.random(n) * 1000 + 50}, index=idx)

    def _reg_cfg(**over):
        cfg = deepcopy(CONFIG)
        cfg.update(tf_order=['1D'], base_tf='1D', window_sizes={'1D': 20}, stride=1,
                   feature_order=None, targets=['close'],
                   enabled_heads={'close_class': True, 'close_reg': True},
                   forecast_horizon=1, reg_target_mode='return', reg_target_clip=None,
                   scaler_type='robust', abstention={'enabled': False},
                   market_context={'enabled': False},
                   funding_rate={'enabled': False}, open_interest={'enabled': False})
        cfg.update(over)
        return cfg

    def t_reg_target_matches_direct_return():
        cfg = _reg_cfg()
        df = _price_asset(n=70, seed=1)
        dfs = _quiet(resample_timeframes, df, ['1D'], base_tf='1D', config=cfg)
        X, y, bases, last = _quiet(prepare_single_asset, dfs, config=cfg)
        assert 'close_reg' in y and len(y['close_reg']) > 0
        close = dfs['1D']['close'].values      # بعد add_features، لم يُمسّ close نفسه
        win = cfg['window_sizes']['1D']
        expected = [close[i + win] / close[i + win - 1] - 1.0
                   for i in range(len(y['close_reg']))]
        np.testing.assert_allclose(y['close_reg'], expected, atol=1e-5)
        # يتّفق في الاتجاه مع رأس التصنيف لنفس الهدف (نفس المرجع الآن)
        agree = np.sign(y['close_reg']) == y['close_class']
        assert agree.all(), agree.mean()

    def t_reg_target_uses_own_kind_not_last_close():
        cfg = _reg_cfg(targets=['high'], window_sizes={'1D': 15},
                       enabled_heads={'high_class': True, 'high_reg': True})
        df = _price_asset(n=60, seed=2)
        # نتحكّم بالشمعتين الأخيرتين فقط: قمة الشمعة الأخيرة (نهاية النافذة)
        # تساوي قمة الشمعة المستقبلية (120=120) بينما إغلاقاهما مختلفان تماماً
        # (100 مقابل 110) — لو استُخدم last_close خطأً كمرجع لكانت النتيجة
        # (120-100)/100=0.20 لا 0؛ الصحيح مرجعه last_high=120 فالنتيجة 0.
        df.iloc[-2, df.columns.get_loc('high')] = 120.0
        df.iloc[-2, df.columns.get_loc('close')] = 100.0
        df.iloc[-1, df.columns.get_loc('high')] = 120.0
        df.iloc[-1, df.columns.get_loc('close')] = 110.0
        dfs = _quiet(resample_timeframes, df, ['1D'], base_tf='1D', config=cfg)
        X, y, bases, last = _quiet(prepare_single_asset, dfs, config=cfg)
        assert len(y['high_reg'])
        assert abs(y['high_reg'][-1]) < 1e-6, y['high_reg'][-1]

    def t_reg_target_clip_default_depends_on_mode():
        df = _price_asset(n=90, seed=4)
        df.iloc[-1, df.columns.get_loc('close')] = 100_000.0   # قفزة ضخمة في آخر شمعة
        df.iloc[-1, df.columns.get_loc('high')] = 100_000.0
        df.iloc[-1, df.columns.get_loc('low')] = 100_000.0
        cfg_ret = _reg_cfg(window_sizes={'1D': 10}, reg_target_clip=None)
        dfs_ret = _quiet(resample_timeframes, df, ['1D'], base_tf='1D', config=cfg_ret)
        _, y_ret, _, _ = _quiet(prepare_single_asset, dfs_ret, config=cfg_ret)
        assert np.all(np.abs(y_ret['close_reg']) <= 1.0 + 1e-6), y_ret['close_reg']
        cfg_win = _reg_cfg(window_sizes={'1D': 10}, reg_target_mode='window_scale',
                           reg_target_clip=None)
        dfs_win = _quiet(resample_timeframes, df, ['1D'], base_tf='1D', config=cfg_win)
        _, y_win, _, _ = _quiet(prepare_single_asset, dfs_win, config=cfg_win)
        assert np.any(np.abs(y_win['close_reg']) > 1.0), \
            "الوضع القديم يجب ألّا يتأثر بحدّ 1.0 الجديد"
        assert np.all(np.abs(y_win['close_reg']) <= 10.0 + 1e-6)

    def t_invert_reg_predictions_return_mode():
        last = np.zeros((3, len(LAST_COLUMNS)))
        last[:, LAST_COLUMNS.index('last_high')] = [100.0, 50.0, 10.0]
        last[:, LAST_COLUMNS.index('last_close')] = [90.0, 45.0, 9.0]
        preds = np.array([0.10, -0.20, 0.0])
        cfg = _reg_cfg()
        got = invert_reg_predictions(preds, 'high_reg', last_candles=last, config=cfg)
        np.testing.assert_allclose(got, [110.0, 40.0, 10.0])
        got_close = invert_reg_predictions(preds, 'close_reg', last_candles=last, config=cfg)
        np.testing.assert_allclose(got_close, [99.0, 36.0, 9.0])

    def t_invert_reg_predictions_window_scale_delegates():
        bases = np.array([[100.0, 5.0], [50.0, 2.0]])
        preds = np.array([1.0, -1.5])
        cfg = _reg_cfg(reg_target_mode='window_scale')
        got = invert_reg_predictions(preds, 'close_reg', bases=bases, config=cfg)
        np.testing.assert_allclose(got, inverse_scale(preds, bases))

    # ── التطبيع المقطعي عبر الأصول (Qlib idea 2) ────────────────────────────
    def _fake_reg_dataset(groups):
        """``groups``: قائمة ``[{asset_name: عائد}, ...]`` — كل عنصر لحظة زمنية
        واحدة تشارك فيها كل الأصول المذكورة فيه."""
        day = 86_400 * 10 ** 9
        base = pd.Timestamp('2026-01-01', tz='UTC').value
        ts, y = [], []
        for k, grp in enumerate(groups):
            for ret in grp.values():
                ts.append(base + k * day)
                y.append(ret)
        n = len(ts)
        last = np.zeros((n, len(LAST_COLUMNS)))
        last[:, TS_COL] = ts
        return {'close_class': np.sign(y).astype('float32'),
                'close_reg': np.asarray(y, dtype='float32'),
                'last_candles': last}

    def t_cs_normalize_matches_manual_zscore():
        groups = [{'A': 0.05, 'B': -0.02, 'C': 0.01, 'D': 0.10, 'E': -0.05},
                  {'A': 0.02, 'B': 0.03, 'C': -0.01, 'D': 0.00, 'E': 0.04}]
        ds = _fake_reg_dataset(groups)
        cfg = _reg_cfg()
        out = _quiet(cross_sectional_normalize, ds, heads=['close_reg'],
                    method='zscore', min_assets=3, clip=10.0, config=cfg)
        for k, grp in enumerate(groups):
            vals = np.array(list(grp.values()))
            mu, sd = vals.mean(), vals.std()
            expected = (vals - mu) / sd
            got = out['close_reg'][k * len(grp):(k + 1) * len(grp)]
            np.testing.assert_allclose(sorted(got), sorted(expected), atol=1e-5)

    def t_cs_normalize_respects_min_assets():
        groups = [{'A': 0.05, 'B': -0.02}, {'A': 0.02, 'B': 0.03, 'C': -0.01, 'D': 0.00}]
        ds = _fake_reg_dataset(groups)
        cfg = _reg_cfg()
        out = _quiet(cross_sectional_normalize, ds, heads=['close_reg'],
                    method='zscore', min_assets=3, config=cfg)
        # اليوم الأول (أصلان فقط) بقي عائداً خاماً كما هو
        np.testing.assert_allclose(out['close_reg'][:2], ds['close_reg'][:2])
        # اليوم الثاني (٤ أصول ≥ 3) تغيّر فعلاً
        assert not np.allclose(out['close_reg'][2:], ds['close_reg'][2:])

    def t_cs_normalize_does_not_mutate_input():
        groups = [{'A': 0.05, 'B': -0.02, 'C': 0.01, 'D': 0.02}]
        ds = _fake_reg_dataset(groups)
        original = ds['close_reg'].copy()
        _quiet(cross_sectional_normalize, ds, heads=['close_reg'], min_assets=2,
              config=_reg_cfg())
        np.testing.assert_allclose(ds['close_reg'], original)

    def t_cs_normalize_rejects_window_scale_mode():
        ds = _fake_reg_dataset([{'A': 0.05, 'B': -0.02, 'C': 0.01}])
        try:
            _quiet(cross_sectional_normalize, ds, heads=['close_reg'],
                  config=_reg_cfg(reg_target_mode='window_scale'))
        except ValueError:
            return
        raise AssertionError("cs_norm على 'window_scale' يجب أن يُرفض")

    def t_cs_normalize_rank_bounded():
        groups = [{'A': 0.05, 'B': -0.02, 'C': 0.01, 'D': 9.0, 'E': -9.0}]  # شواذ متعمَّدة
        ds = _fake_reg_dataset(groups)
        out = _quiet(cross_sectional_normalize, ds, heads=['close_reg'],
                    method='rank', min_assets=3, config=_reg_cfg())
        assert np.all(out['close_reg'] >= -1.0 - 1e-6) and np.all(out['close_reg'] <= 1.0 + 1e-6)

    def t_invert_cross_sectional_roundtrips_zscore():
        groups = [{'A': 0.05, 'B': -0.02, 'C': 0.01, 'D': 0.02, 'E': -0.01}]
        ds = _fake_reg_dataset(groups)
        cfg = _reg_cfg()
        out = _quiet(cross_sectional_normalize, ds, heads=['close_reg'],
                    method='zscore', min_assets=3, clip=10.0, config=cfg)
        ts = out['last_candles'][:, TS_COL]
        back = invert_cross_sectional(out['close_reg'], ts, 'close_reg', out)
        np.testing.assert_allclose(back, ds['close_reg'], atol=1e-4)

    def t_invert_cross_sectional_rejects_rank():
        ds = _fake_reg_dataset([{'A': 0.05, 'B': -0.02, 'C': 0.01}])
        out = _quiet(cross_sectional_normalize, ds, heads=['close_reg'],
                    method='rank', min_assets=3, config=_reg_cfg())
        try:
            invert_cross_sectional(out['close_reg'], out['last_candles'][:, TS_COL],
                                   'close_reg', out)
        except ValueError:
            return
        raise AssertionError("عكس 'rank' يجب أن يُرفض")

    # ── حدّ فاصل بديل: نسبة العيّنات الخام (compute_global_cutoff) ──────────
    def t_compute_global_cutoff_matches_target_fraction():
        ds = _fake_dataset([400] * 30, stride=1)      # موزَّع بانتظام، بلا تكدّس
        ts = sample_timestamps(ds)
        for pct in (0.10, 0.30, 0.50):
            cutoff = compute_global_cutoff(ds, pct)
            frac = (ts >= cutoff).mean()
            assert abs(frac - pct) < 0.02, (pct, frac)

    def t_compute_global_cutoff_rejects_bad_pct():
        ds = _fake_dataset([100] * 5, stride=1)
        for bad in (0.0, 1.0, -0.1, 1.5):
            try:
                compute_global_cutoff(ds, bad)
            except ValueError:
                continue
            raise AssertionError(f"pct={bad} يجب أن يُرفض")

    def t_extract_test_dates_handles_both_shapes():
        ds = _fake_dataset([300] * 10, stride=1)
        cfg = _cfg()
        _, _, test_flat = _quiet(split_data, ds, config=cfg,
                                 keep_asset_test_separate=False)
        _, _, test_sep = _quiet(split_data, ds, config=cfg,
                                keep_asset_test_separate=True)
        d1 = extract_test_dates(test_flat)
        d2 = extract_test_dates(test_sep)
        assert len(d1) == len(test_flat['last_candles'])
        assert len(d2) == sum(len(v['last_candles']) for v in test_sep.values())

    def t_raw_quantile_converges_to_kept_share_as_data_grows():
        # الفرق بين الطريقتين يتناسب مع (عيّنات فجوة العزل الضائعة / حجم
        # القسم المستهدف) — يصغر كلما كبر val_pct×الإجمالي عن خسارة العزل،
        # حتى على بيانات موزَّعة بانتظام تماماً (5 أصول بنفس المدى الزمني).
        ds = _fake_dataset([15000] * 5, stride=1)
        cfg = _cfg()
        tr1, va1, te1 = _quiet(split_data, ds, config={**cfg, 'split_cutoff_method': 'kept_share'})
        tr2, va2, te2 = _quiet(split_data, ds, config={**cfg, 'split_cutoff_method': 'raw_quantile'})
        for a, b, name in ((tr1, tr2, 'train'), (va1, va2, 'val'), (te1, te2, 'test')):
            n1, n2 = len(a['base_params']), len(b['base_params'])
            assert abs(n1 - n2) / max(n1, n2, 1) < 0.15, (name, n1, n2)

    def t_raw_quantile_fails_loudly_on_crowded_data():
        ds = _fake_dataset(CROWDED)                    # نفس بيانات "التكدّس" الحقيقية
        cfg = _cfg()
        _quiet(split_data, ds, config={**cfg, 'split_cutoff_method': 'kept_share'})  # ينجح
        try:
            _quiet(split_data, ds, config={**cfg, 'split_cutoff_method': 'raw_quantile'})
        except ValueError as exc:
            assert 'min_split_samples' in str(exc) or 'أصغر من الحدّ الأدنى' in str(exc)
            return
        raise AssertionError(
            "raw_quantile على بيانات مكدَّسة كان يجب أن يفشل بوضوح لا أن ينجح صامتاً بقسم شبه فارغ")

    def t_build_leak_free_split_no_cross_asset_overlap():
        ds = _fake_dataset([3000] * 4, stride=1)
        cfg = _cfg()
        train, val, test = _quiet(build_leak_free_split, ds, config=cfg)
        gap = embargo_duration(ds, cfg)
        assert len(train['base_params']) and len(val['base_params'])
        t_tr = sample_timestamps(train)
        t_te = extract_test_dates(test)
        assert len(t_te)
        assert t_te.min() - t_tr.max() >= gap         # بلا أي تداخل بين أي عملتين

    def t_build_leak_free_split_matches_split_data_raw_quantile():
        ds = _fake_dataset([1500] * 4, stride=1)
        cfg = _cfg()
        tr_a, va_a, te_a = _quiet(build_leak_free_split, ds, config=cfg)
        tr_b, va_b, te_b = _quiet(split_data, ds, config={**cfg, 'split_cutoff_method': 'raw_quantile'})
        np.testing.assert_array_equal(tr_a['base_params'], tr_b['base_params'])
        np.testing.assert_array_equal(va_a['base_params'], va_b['base_params'])
        np.testing.assert_array_equal(te_a['base_params'], te_b['base_params'])

    # ── فريم التحميل مستقلّ عن فريم النموذج ─────────────────────────────────
    def t_fetch_data_uses_requested_interval_not_hardcoded():
        fake = _FakeFuturesClient(500)
        _with_client(fake, lambda: fetch_data('BTCUSDT', '4h', 100))
        assert fake.calls[0]['interval'] == '4h', fake.calls[0]

    def t_live_pad_length_scales_with_download_interval():
        cfg1 = _cfg(tf_order=['1D'], window_sizes={'1D': 32}, forecast_horizon=1)
        p_1h = _live_pad_length(['1D'], {'1D': 32}, 1, '1h')
        p_1d = _live_pad_length(['1D'], {'1D': 32}, 1, '1D')
        assert p_1h == 24 * p_1d, (p_1h, p_1d)      # وحدة أدقّ 24× ⇒ padding أكبر 24×

    def t_auto_live_limit_scales_with_download_interval():
        lim_1h = _auto_live_limit(['1D'], {'1D': 32}, 1, 25, '1h')
        lim_1d = _auto_live_limit(['1D'], {'1D': 32}, 1, 25, '1D')
        assert lim_1h > lim_1d, (lim_1h, lim_1d)   # بوحدة أدقّ يلزم شموع أكثر (قد يُقصّ عند BINANCE_MAX_LIMIT)
        lim_1h_small = _auto_live_limit(['1D'], {'1D': 5}, 1, 5, '1h')
        lim_1d_small = _auto_live_limit(['1D'], {'1D': 5}, 1, 5, '1D')
        assert lim_1h_small > lim_1d_small * 10, (lim_1h_small, lim_1d_small)   # بلا قصّ: الفارق واضح

    def t_build_dataset_live_decoupled_intervals_end_to_end():
        class _FakeMultiTF:
            H = {'1h': 3_600_000, '1d': 86_400_000}

            def __init__(self, days=120, end='2026-09-19 20:00'):
                self.end = pd.Timestamp(end, tz='UTC')
                self.days = days
                self.calls = []

            def futures_klines(self, symbol, interval, limit, endTime=None):
                self.calls.append(interval)
                step = self.H[interval]
                n = self.days * (24 if interval == '1h' else 1)
                end_ms = int(self.end.timestamp() * 1000) // step * step
                ts = end_ms - (n - 1 - np.arange(n)) * step
                if endTime is not None:
                    ts = ts[ts <= endTime]
                ts = ts[-limit:]
                rng = np.random.default_rng(1)
                c = 100 * np.exp(np.cumsum(rng.normal(0, .01, len(ts))))
                return [[int(t), str(c[i]), str(c[i] * 1.01), str(c[i] * .99), str(c[i]),
                        "100", int(t) + step - 1, "0", 0, "0", "0", "0"]
                       for i, t in enumerate(ts)]

        cfg = _cfg(tf_order=['1D'], base_tf='1D', window_sizes={'1D': 20},
                  forecast_horizon=1, download_interval='1h')
        fake = _FakeMultiTF()
        old_default_workers = globals().get('default_workers')
        globals()['default_workers'] = lambda n=0: 2
        try:
            ds = _with_client(fake, lambda: build_dataset_live(
                ['AAAUSDT', 'BBBUSDT'], max_workers=1, config=cfg))
        finally:
            globals()['default_workers'] = old_default_workers
        assert set(fake.calls) == {'1h'}, fake.calls    # طُلبت الساعية فقط، لا اليومية إطلاقاً
        assert len(ds['base_params']) > 0
        assert ds['X_1D'].shape[1] == 20                # نافذة يومية سليمة رغم التحميل الساعي

    # ── سياق سوقي عابر للأصول (BTC) ─────────────────────────────────────────
    def t_market_context_columns_names():
        cfg = _cfg(market_context={'enabled': True, 'reference_symbol': 'BTCUSDT',
                                   'return_periods': [1, 3]})
        assert market_context_columns(cfg) == ['MKT_ret_1', 'MKT_ret_3']
        cfg_off = _cfg(market_context={'enabled': False})
        assert market_context_columns(cfg_off) == []

    def t_add_market_context_aligns_and_zeroes_reference():
        cfg = _cfg(market_context={'enabled': True, 'reference_symbol': 'BTCUSDT',
                                   'return_periods': [1]})
        idx = pd.date_range('2025-01-01', periods=10, freq='D', tz='UTC')
        btc_close = pd.Series(np.linspace(100, 109, 10), index=idx)
        market_dfs = {'1D': pd.DataFrame({'MKT_ret_1': btc_close.pct_change(1)}, index=idx)}
        alt_df = pd.DataFrame({'close': np.linspace(1, 2, 10)}, index=idx)

        out_alt = add_market_context({'1D': alt_df}, market_dfs, is_reference=False, config=cfg)
        np.testing.assert_allclose(out_alt['1D']['MKT_ret_1'].iloc[1:].values,
                                   btc_close.pct_change(1).iloc[1:].values)

        out_btc = add_market_context({'1D': alt_df}, market_dfs, is_reference=True, config=cfg)
        assert (out_btc['1D']['MKT_ret_1'] == 0.0).all()

    def t_market_context_reaches_dataset_via_build_dataset():
        cfg = _cfg(market_context={'enabled': True, 'reference_symbol': 'AAAUSDT',
                                   'return_periods': [1]})
        names = ['AAAUSDT', 'BBBUSDT']
        data = {n: _quiet(resample_timeframes, _ohlcv(200, seed=i), ['1D'], config=cfg)
               for i, n in enumerate(names)}
        ds = _quiet(build_dataset_from_preloaded, [{'name': n} for n in names], data,
                   config=cfg, max_workers=1)
        assert 'MKT_ret_1' in ds['feature_order']
        i = ds['feature_order'].index('MKT_ret_1')
        b_bbb = next(b for b in ds['asset_bounds'] if b['name'] == 'BBBUSDT')
        b_aaa = next(b for b in ds['asset_bounds'] if b['name'] == 'AAAUSDT')
        # BBBUSDT (ليست المرجع) لها قيم سياق غير صفرية في مكان ما على الأقل
        assert np.any(ds['X_1D'][b_bbb['start']:b_bbb['end'], :, i] != 0.0)
        # AAAUSDT هي المرجع نفسه: صفر دائماً
        assert np.all(ds['X_1D'][b_aaa['start']:b_aaa['end'], :, i] == 0.0)

    # ── إعادة تدريب دورية (rolling_splits) ──────────────────────────────────
    def t_rolling_schedule_expanding_basic():
        ds = _fake_dataset([3000] * 3, stride=1)
        cfg = _cfg()
        sched = rolling_split_schedule(ds, test_span='20D', val_span='10D',
                                       initial_train_span='60D', config=cfg)
        assert len(sched) >= 2
        for w in sched:
            assert w['train_end'] < w['val_end'] < w['test_end']
        # النافذة الأولى تبدأ من أقدم عيّنة، والثانية train_end أبعد من الأولى (تمدّد)
        assert sched[0]['train_start'] == sample_timestamps(ds).min()
        assert sched[1]['train_end'] > sched[0]['train_end']
        assert sched[1]['train_start'] == sched[0]['train_start']   # نفس البداية دائماً (متمدّدة)

    def t_rolling_schedule_sliding_basic():
        ds = _fake_dataset([3000] * 3, stride=1)
        cfg = _cfg()
        sched = rolling_split_schedule(ds, test_span='20D', val_span='10D',
                                       train_span='60D', config=cfg)
        assert len(sched) >= 2
        for w in sched:
            assert (w['train_end'] - w['train_start']) == pd.Timedelta('60D')
        assert sched[1]['train_start'] > sched[0]['train_start']    # ينزلق فعلاً

    def t_rolling_schedule_requires_initial_train_span():
        ds = _fake_dataset([1000] * 2, stride=1)
        try:
            rolling_split_schedule(ds, test_span='20D', val_span='10D', config=_cfg())
        except ValueError as exc:
            assert 'initial_train_span' in str(exc)
            return
        raise AssertionError("نافذة متمدّدة بلا initial_train_span يجب أن تُرفض")

    def t_rolling_splits_no_leak_between_train_and_test_per_window():
        ds = _fake_dataset([4000] * 3, stride=1)
        cfg = _cfg(min_split_samples=20)
        windows = _quiet(rolling_splits, ds, test_span='20D', val_span='10D',
                         initial_train_span='60D', config=cfg)
        assert len(windows) >= 2
        gap = embargo_duration(ds, cfg)
        for train, val, test in windows:
            assert len(train['base_params']) and len(val['base_params'])
            t_tr, t_va = sample_timestamps(train), sample_timestamps(val)
            t_te = extract_test_dates(test)
            assert len(t_te)
            assert t_va.min() - t_tr.max() >= gap
            assert t_te.min() - t_va.max() >= gap

    def t_rolling_splits_max_windows_respected():
        ds = _fake_dataset([6000] * 3, stride=1)
        cfg = _cfg(min_split_samples=10)
        windows = _quiet(rolling_splits, ds, test_span='10D', val_span='5D',
                         initial_train_span='30D', max_windows=2, config=cfg)
        assert len(windows) == 2

    # ── معدّل التمويل والفائدة المفتوحة ──────────────────────────────────────
    def t_fetch_funding_rate_paginates_and_merges():
        class _FakeFunding:
            def __init__(self, n=2500, end_ms=1_800_000_000_000):
                self.n, self.end_ms, self.calls = n, end_ms, []
                self.step = 8 * 3_600_000

            def futures_funding_rate(self, symbol, limit, endTime=None):
                self.calls.append(limit)
                assert limit <= FUNDING_MAX_LIMIT
                idx = [i for i in range(self.n)
                      if endTime is None or self._ts(i) <= endTime]
                return [{"symbol": symbol, "fundingTime": self._ts(i),
                        "fundingRate": str(0.0001 * (i % 7 - 3))} for i in idx[-limit:]]

            def _ts(self, i):
                return self.end_ms - (self.n - 1 - i) * self.step

        fake = _FakeFunding()
        df = _with_client(fake, lambda: fetch_funding_rate('BTCUSDT', 1800))
        assert len(fake.calls) == 2 and fake.calls == [1000, 800]
        assert len(df) == 1800
        assert df['timestamp'].is_monotonic_increasing and not df['timestamp'].duplicated().any()
        step = df['timestamp'].diff().dropna().unique()
        assert len(step) == 1 and step[0] == pd.Timedelta(hours=8)

    def t_fetch_open_interest_hist_capped_by_platform_lookback():
        class _FakeOI:
            """يحاكي قيد Binance: لا يُرجع شيئاً أبعد من ٣٠ يوماً مهما طُلب."""
            def __init__(self, end_ms=1_800_000_000_000):
                self.end_ms, self.calls = end_ms, []
                self.step = 3_600_000
                self.available = 30 * 24     # ٣٠ يوماً بدقة ساعة

            def futures_open_interest_hist(self, symbol, period, limit, endTime=None):
                self.calls.append(limit)
                cutoff = self.end_ms - self.available * self.step
                lo = endTime if endTime is not None else self.end_ms
                idx = [t for t in range(cutoff, lo + 1, self.step)]
                idx = idx[-limit:]
                return [{"symbol": symbol, "timestamp": t,
                        "sumOpenInterest": "1000.0"} for t in idx]

        fake = _FakeOI()
        df = _with_client(fake, lambda: fetch_open_interest_hist(
            'BTCUSDT', 5000, period='1h'))
        assert len(df) <= 30 * 24 + 1          # لم يتجاوز قيد المنصّة رغم limit=5000
        assert len(fake.calls) >= 2            # احتاج أكثر من صفحة (٥٠٠ الحدّ الأقصى للطلب)

    def t_save_and_load_funding_open_interest_accumulates():
        import tempfile
        class _FakeFR:
            def __init__(self, rows):
                self.rows = rows

            def futures_funding_rate(self, symbol, limit, endTime=None):
                return self.rows[-limit:]

            def futures_open_interest_hist(self, symbol, period, limit, endTime=None):
                return []

        cfg = _cfg(funding_rate={'enabled': True, 'drive_dir': 'funding_rate'},
                  open_interest={'enabled': False})
        tmp = Path(tempfile.mkdtemp())
        old_mount = globals()['mount_drive']
        globals()['mount_drive'] = lambda mount_point=None, config=None: tmp
        try:
            batch1 = [{"symbol": "BTCUSDT", "fundingTime": 1_700_000_000_000 + i * 28_800_000,
                      "fundingRate": "0.0001"} for i in range(5)]
            fake1 = _FakeFR(batch1)
            _with_client(fake1, lambda: save_funding_open_interest(
                ['BTCUSDT'], funding_limit=5, config=cfg, verbose=False))
            first = load_funding_open_interest('BTCUSDT', 'funding_rate', config=cfg)
            assert len(first) == 5

            batch2 = batch1 + [{"symbol": "BTCUSDT",
                                "fundingTime": 1_700_000_000_000 + 5 * 28_800_000,
                                "fundingRate": "0.0002"}]
            fake2 = _FakeFR(batch2)
            _with_client(fake2, lambda: save_funding_open_interest(
                ['BTCUSDT'], funding_limit=6, config=cfg, verbose=False))
            second = load_funding_open_interest('BTCUSDT', 'funding_rate', config=cfg)
            assert len(second) == 6            # تراكم بلا تكرار، لا استبدال بالكامل
        finally:
            globals()['mount_drive'] = old_mount

    # ── دمج معدّل التمويل/الفائدة المفتوحة كميزات ────────────────────────────
    def t_funding_oi_columns_names():
        cfg = _cfg(funding_rate={'enabled': True, 'as_feature': True, 'zscore_window': 90},
                  open_interest={'enabled': True, 'as_feature': True, 'change_period': 1})
        assert funding_oi_feature_columns(cfg) == [
            'FUND_rate', 'FUND_rate_z', 'FUND_available', 'OI_chg_1', 'OI_available']

        cfg_off = _cfg(funding_rate={'enabled': False}, open_interest={'enabled': False})
        assert funding_oi_feature_columns(cfg_off) == []

        # enabled (جلب/أرشفة) بلا as_feature (إدماج) لا يُنتج أي عمود مدخل
        cfg_fetch_only = _cfg(funding_rate={'enabled': True, 'as_feature': False},
                              open_interest={'enabled': True, 'as_feature': False})
        assert funding_oi_feature_columns(cfg_fetch_only) == []

    def t_add_funding_oi_features_aligns_and_flags_availability():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        old_mount = globals()['mount_drive']
        globals()['mount_drive'] = lambda mount_point=None, config=None: tmp
        try:
            cfg = _cfg(funding_rate={'enabled': True, 'drive_dir': 'funding_rate',
                                     'as_feature': True, 'zscore_window': 5},
                      open_interest={'enabled': True, 'drive_dir': 'open_interest',
                                    'as_feature': True, 'change_period': 1})
            fr = pd.DataFrame({
                'timestamp': pd.date_range('2025-01-01', periods=10, freq='8h', tz='UTC'),
                'funding_rate': np.linspace(0.0001, 0.001, 10)})
            fr.to_csv(_funding_oi_drive_path(tmp, 'funding_rate', 'AAAUSDT', cfg), index=False)
            oi = pd.DataFrame({
                'timestamp': pd.date_range('2025-01-03', periods=5, freq='1h', tz='UTC'),
                'open_interest': np.linspace(1000.0, 1100.0, 5)})
            oi.to_csv(_funding_oi_drive_path(tmp, 'open_interest', 'AAAUSDT', cfg), index=False)

            idx = pd.date_range('2025-01-01', periods=5, freq='D', tz='UTC')
            df = pd.DataFrame({'close': np.linspace(1, 2, 5)}, index=idx)
            out = add_funding_oi_features({'1D': df}, 'AAAUSDT', config=cfg)['1D']

            assert list(out['FUND_available']) == [1.0, 1.0, 1.0, 1.0, 1.0]
            # الفائدة المفتوحة تبدأ 2025-01-03، لكن أول نقطة فيها NaN بالبناء
            # (diff() بلا سابقة) — reindex(ffill) لا يستبدل قيمة موجودة
            # فعلاً بالفهرس (حتى لو NaN)، فتبقى 2025-01-03 نفسها غير متاحة
            assert list(out['OI_available']) == [0.0, 0.0, 0.0, 1.0, 1.0]
            assert out.loc[idx[0], 'OI_chg_1'] == 0.0    # محايد صفر لا NaN حين لا تغطية
            assert not out['FUND_rate'].isna().any() and not out['OI_chg_1'].isna().any()
        finally:
            globals()['mount_drive'] = old_mount

    def t_add_funding_oi_features_neutral_when_no_archive():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        old_mount = globals()['mount_drive']
        globals()['mount_drive'] = lambda mount_point=None, config=None: tmp
        try:
            cfg = _cfg(funding_rate={'enabled': True, 'as_feature': True},
                      open_interest={'enabled': True, 'as_feature': True})
            idx = pd.date_range('2025-01-01', periods=3, freq='D', tz='UTC')
            df = pd.DataFrame({'close': [1.0, 2.0, 3.0]}, index=idx)
            out = add_funding_oi_features({'1D': df}, 'GHOSTUSDT', config=cfg)['1D']
            assert (out['FUND_rate'] == 0.0).all() and (out['FUND_available'] == 0.0).all()
            assert (out['OI_chg_1'] == 0.0).all() and (out['OI_available'] == 0.0).all()
        finally:
            globals()['mount_drive'] = old_mount

    def t_funding_oi_disabled_returns_dfs_unchanged():
        cfg = _cfg(funding_rate={'enabled': False}, open_interest={'enabled': False})
        idx = pd.date_range('2025-01-01', periods=3, freq='D', tz='UTC')
        df = pd.DataFrame({'close': [1.0, 2.0, 3.0]}, index=idx)
        out = add_funding_oi_features({'1D': df}, 'AAAUSDT', config=cfg)
        assert out['1D'] is df               # بلا نسخ إضافي حين لا شيء مُفعَّل

    def t_funding_oi_reaches_dataset_via_build_dataset():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        old_mount = globals()['mount_drive']
        globals()['mount_drive'] = lambda mount_point=None, config=None: tmp
        try:
            cfg = _cfg(funding_rate={'enabled': True, 'drive_dir': 'funding_rate',
                                     'as_feature': True, 'zscore_window': 5},
                      open_interest={'enabled': False}, market_context={'enabled': False})
            # يغطّي كامل مدى الأصل (~200 يوماً) بقيم متغيّرة باستمرار — أرشيف
            # قصير (يغطّي جزءاً من المدى فقط) يُصبح ثابتاً بعد ffill داخل أغلب
            # النوافذ، فيُصفّره process_windows كعمود بلا تباين (بنفس منطق
            # t_process_windows_degenerate_column_zeroed)، فيفشل فحص != 0.0 هنا
            # لا لخلل في الدمج بل لأن القيمة صفر عن حقّ في تلك النوافذ تحديداً.
            n_fr = 700
            fr = pd.DataFrame({
                'timestamp': pd.date_range('2025-01-01', periods=n_fr, freq='8h', tz='UTC'),
                'funding_rate': 0.0005 * np.sin(np.linspace(0, 40 * np.pi, n_fr))})
            fr.to_csv(_funding_oi_drive_path(tmp, 'funding_rate', 'AAAUSDT', cfg), index=False)

            names = ['AAAUSDT']
            data = {n: _quiet(resample_timeframes, _ohlcv(200, seed=0), ['1D'], config=cfg)
                   for n in names}
            ds = _quiet(build_dataset_from_preloaded, [{'name': n} for n in names], data,
                       config=cfg, max_workers=1)
            assert 'FUND_rate' in ds['feature_order']
            i = ds['feature_order'].index('FUND_rate')
            assert np.any(ds['X_1D'][:, :, i] != 0.0)   # قيم حقيقية وصلت للمصفوفة النهائية
        finally:
            globals()['mount_drive'] = old_mount

    def t_checkpoint_fingerprint_changes_with_funding_oi_config():
        cfg1 = _cfg(funding_rate={'enabled': True, 'as_feature': False})
        cfg2 = _cfg(funding_rate={'enabled': True, 'as_feature': True})
        fp1 = _checkpoint_fingerprint(cfg1, ['a'], 0)
        fp2 = _checkpoint_fingerprint(cfg2, ['a'], 0)
        assert fp1 != fp2
        assert fp1 == _checkpoint_fingerprint(cfg1, ['a'], 0)


    # ── تسريع النوافذ: مسار فريم واحد بلا تكرار نسخ (align) ─────────────────
    def t_align_single_tf_fast_path_matches_manual_windows():
        idx = pd.date_range('2025-01-01', periods=10, freq='D', tz='UTC')
        df = pd.DataFrame({'a': np.arange(10.0), 'b': np.arange(10.0) * 10}, index=idx)
        cfg = _cfg(tf_order=['1D'], window_sizes={'1D': 3}, stride=1)
        windows, end_times = align_multi_timeframes_time_based({'1D': df}, config=cfg)
        assert windows['1D'].shape == (8, 3, 2)
        np.testing.assert_array_equal(windows['1D'][0], df.iloc[0:3].values.astype('float32'))
        np.testing.assert_array_equal(windows['1D'][-1], df.iloc[7:10].values.astype('float32'))
        assert list(end_times) == list(df.index[2:])

    def t_align_single_tf_fast_path_respects_stride():
        idx = pd.date_range('2025-01-01', periods=20, freq='D', tz='UTC')
        df = pd.DataFrame({'a': np.arange(20.0)}, index=idx)
        cfg = _cfg(tf_order=['1D'], window_sizes={'1D': 5}, stride=3)
        windows, end_times = align_multi_timeframes_time_based({'1D': df}, config=cfg)
        n_expected = len(range(4, 20, 3))
        assert windows['1D'].shape[0] == n_expected
        np.testing.assert_array_equal(windows['1D'][1][:, 0], df.iloc[3:8]['a'].values)

    def t_align_single_tf_fast_path_empty_when_too_short():
        idx = pd.date_range('2025-01-01', periods=3, freq='D', tz='UTC')
        df = pd.DataFrame({'a': [1.0, 2.0, 3.0]}, index=idx)
        cfg = _cfg(tf_order=['1D'], window_sizes={'1D': 5}, stride=1)
        windows, end_times = align_multi_timeframes_time_based({'1D': df}, config=cfg)
        assert windows['1D'].shape == (0, 5, 1)
        assert end_times == []

    def t_align_single_tf_peak_memory_bounded():
        import tracemalloc
        n = 2000
        idx = pd.date_range('2025-01-01', periods=n, freq='D', tz='UTC')
        rng = np.random.default_rng(0)
        df = pd.DataFrame({f'f{i}': rng.normal(0, 1, n) for i in range(20)}, index=idx)
        cfg = _cfg(tf_order=['1D'], window_sizes={'1D': 32}, stride=1)
        tracemalloc.start()
        windows, _ = align_multi_timeframes_time_based({'1D': df}, config=cfg)
        peak = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()
        assert peak < windows['1D'].nbytes * 2.5, (peak, windows['1D'].nbytes)   # قِيس فعلياً ~1.13×

    # ── تطبيع النوافذ: float32 من البداية للنهاية بلا مصفوفة out موازية ─────
    def t_process_windows_price_level_matches_manual_formula():
        cols = ['close']
        windows = np.zeros((2, 4, 1), dtype='float32')
        windows[0, :, 0] = [10, 11, 12, 13]
        windows[1, :, 0] = [20, 19, 18, 17]
        centers = np.array([12.0, 18.0], dtype='float32')
        scales = np.array([2.0, 2.0], dtype='float32')
        out = process_windows(windows.copy(), cols, centers, scales, 'robust', config=_cfg())
        expected = (windows - centers.reshape(2, 1, 1)) / scales.reshape(2, 1, 1)
        np.testing.assert_allclose(out, expected, atol=1e-5)

    def t_process_windows_degenerate_column_zeroed():
        cols = ['RSI_14']
        windows = np.full((2, 5, 1), 50.0, dtype='float32')
        out = process_windows(windows.copy(), cols, np.zeros(2, 'float32'),
                              np.ones(2, 'float32'), config=_cfg())
        assert np.all(out == 0.0)

    def t_process_windows_clip_applied():
        cols = ['RET_1']
        windows = np.zeros((1, 5, 1), dtype='float32')
        windows[0, :, 0] = [0, 0, 0, 0, 1000.0]
        out = process_windows(windows.copy(), cols, np.zeros(1, 'float32'),
                              np.ones(1, 'float32'), config=_cfg(clip_abs=5.0))
        assert np.all(np.abs(out) <= 5.0 + 1e-6)

    def t_process_windows_peak_memory_bounded():
        # مزيج واقعي من الأنواع الدلالية (كما في مجموعة ميزات حقيقية) —
        # الذروة أفضل من حالة نوع واحد يغطّي كل الأعمدة (يُنسخ كل X دفعة واحدة).
        import tracemalloc
        N, T, F = 500, 32, 30
        rng = np.random.default_rng(0)
        windows = rng.normal(0, 1, (N, T, F)).astype('float32')
        kinds = ['close', 'RSI_14', 'RET_1', 'MACDh_12_26_9', 'BBP_20_2.0', 'OBV']
        cols = [kinds[i % len(kinds)] + f'_{i}' if i % len(kinds) not in (0,) else 'close'
               for i in range(F)]
        tracemalloc.start()
        out = process_windows(windows.copy(), cols, np.full(N, 100.0, 'float32'),
                              np.full(N, 5.0, 'float32'), config=_cfg())
        peak = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()
        assert peak < out.nbytes * 6, (peak, out.nbytes)          # هامش أمان (قِيس فعلياً ~2.6-4.6×)

    # ── عدد الخيوط الافتراضي: سقف رام احترازي ────────────────────────────────
    def t_default_workers_capped_by_ram():
        old_fn = globals()['_system_ram_mb']
        globals()['_system_ram_mb'] = lambda: 4096.0
        try:
            w = default_workers(1000)
        finally:
            globals()['_system_ram_mb'] = old_fn
        assert w <= 4, w

    def t_default_workers_unaffected_when_ram_unknown():
        old_fn = globals()['_system_ram_mb']
        globals()['_system_ram_mb'] = lambda: None
        try:
            w = default_workers(1000)
        finally:
            globals()['_system_ram_mb'] = old_fn
        assert w >= 4

    # ── نقاط استئناف لكل أصل (checkpoint/resume) ────────────────────────────
    def t_checkpoint_fingerprint_changes_with_relevant_config():
        cfg1 = _cfg()
        cfg2 = _cfg(window_sizes={'1D': 64})
        fp1 = _checkpoint_fingerprint(cfg1, ['a', 'b'], 0)
        fp2 = _checkpoint_fingerprint(cfg2, ['a', 'b'], 0)
        assert fp1 != fp2
        assert fp1 == _checkpoint_fingerprint(cfg1, ['a', 'b'], 0)

    def t_save_load_checkpoint_roundtrip():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        rng = np.random.default_rng(0)
        X_tf = {'1D': rng.normal(0, 1, (5, 32, 3)).astype('float32')}
        y_t = {'close_reg': rng.normal(0, 1, 5).astype('float32')}
        bases = rng.normal(0, 1, (5, 2)).astype('float32')
        last = rng.normal(0, 1, (5, 7)).astype('float64')
        _save_asset_checkpoint(tmp, 'AAAUSDT', X_tf, y_t, bases, last, 'fp123')
        loaded = _load_asset_checkpoint(tmp, 'AAAUSDT', 'fp123')
        assert loaded is not None
        lX, ly, lb, ll = loaded
        np.testing.assert_array_equal(lX['1D'], X_tf['1D'])
        np.testing.assert_array_equal(ly['close_reg'], y_t['close_reg'])
        np.testing.assert_array_equal(lb, bases)
        np.testing.assert_array_equal(ll, last)

    def t_load_checkpoint_rejects_mismatched_fingerprint():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        X_tf = {'1D': np.zeros((2, 3, 1), 'float32')}
        y_t = {'close_reg': np.zeros(2, 'float32')}
        bases = np.zeros((2, 2), 'float32')
        last = np.zeros((2, 7), 'float64')
        _save_asset_checkpoint(tmp, 'X', X_tf, y_t, bases, last, 'fp_old')
        assert _load_asset_checkpoint(tmp, 'X', 'fp_new') is None

    def t_load_checkpoint_missing_returns_none():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        assert _load_asset_checkpoint(tmp, 'GHOST', 'anyfp') is None

    def t_clear_checkpoint_removes_files():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        X_tf = {'1D': np.zeros((1, 2, 1), 'float32')}
        y_t = {'close_reg': np.zeros(1, 'float32')}
        bases = np.zeros((1, 2), 'float32')
        last = np.zeros((1, 7), 'float64')
        _save_asset_checkpoint(tmp, 'A', X_tf, y_t, bases, last, 'fp')
        _save_asset_checkpoint(tmp, 'B', X_tf, y_t, bases, last, 'fp')
        n = clear_checkpoint(tmp, names=['A'])
        assert n == 1
        assert _load_asset_checkpoint(tmp, 'A', 'fp') is None
        assert _load_asset_checkpoint(tmp, 'B', 'fp') is not None
        n2 = clear_checkpoint(tmp)
        assert n2 == 1
        assert _load_asset_checkpoint(tmp, 'B', 'fp') is None

    def t_build_dataset_from_loader_checkpoint_resume_end_to_end():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        cfg = _cfg()
        seeds = {'AAAUSDT': 1, 'BBBUSDT': 2, 'CCCUSDT': 3}
        call_count = {'n': 0}

        def _loader(file_id, name):
            call_count['n'] += 1
            return _ohlcv(200, seed=seeds[name])

        def resample_fn(df, tf_order):
            return _quiet(resample_timeframes, df, tf_order, config=cfg)

        names = list(seeds)
        ds1 = _quiet(build_dataset_from_loader, [{'name': n} for n in names], _loader,
                    resample_fn, config=cfg, max_workers=1, checkpoint_dir=tmp)
        n_calls_first = call_count['n']
        assert n_calls_first == 3

        ds2 = _quiet(build_dataset_from_loader, [{'name': n} for n in names], _loader,
                    resample_fn, config=cfg, max_workers=1, checkpoint_dir=tmp)
        assert call_count['n'] == n_calls_first            # لا تحميل جديد إطلاقاً
        np.testing.assert_array_equal(ds1['base_params'], ds2['base_params'])
        assert [b['name'] for b in ds1['asset_bounds']] == [b['name'] for b in ds2['asset_bounds']]

    def t_build_dataset_checkpoint_invalidated_by_config_change():
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        call_count = {'n': 0}

        def _loader(file_id, name):
            call_count['n'] += 1
            return _ohlcv(200, seed=1)

        cfg1 = _cfg(window_sizes={'1D': 20})

        def resample_fn1(df, tf_order):
            return _quiet(resample_timeframes, df, tf_order, config=cfg1)

        _quiet(build_dataset_from_loader, [{'name': 'AAAUSDT'}], _loader, resample_fn1,
              config=cfg1, max_workers=1, checkpoint_dir=tmp)
        assert call_count['n'] == 1

        cfg2 = _cfg(window_sizes={'1D': 25})

        def resample_fn2(df, tf_order):
            return _quiet(resample_timeframes, df, tf_order, config=cfg2)

        _quiet(build_dataset_from_loader, [{'name': 'AAAUSDT'}], _loader, resample_fn2,
              config=cfg2, max_workers=1, checkpoint_dir=tmp)
        assert call_count['n'] == 2                        # أُعيدت المعالجة (البصمة اختلفت)

    tests = [t_exclude_matching, t_exclude_edge_cases, t_exclude_in_pipeline,
             t_hour_features_daily, t_hour_features_intraday,
             t_split_crowded_timeline, t_split_uniform_matches_requested,
             t_split_raises_when_impossible, t_split_explicit_dates_guarded,
             t_fetch_single_page_unchanged, t_fetch_paginates_and_merges,
             t_fetch_exact_multiple_of_max, t_fetch_history_shorter_than_limit,
             t_fetch_retries_transient_page_error,
             t_fetch_returns_none_on_persistent_page_failure,
             t_fetch_rejects_nonpositive_limit,
             t_exclude_features_names_and_patterns, t_exclude_features_idempotent_and_guard,
             t_exclude_features_reaches_dataset,
             t_ratelimiter_sliding_window, t_ratelimiter_threadsafe_never_exceeds,
             t_default_request_limit_is_2000_per_minute,
             t_fetch_acquires_per_request_including_retries, t_fetch_is_actually_throttled,
             t_reg_target_matches_direct_return, t_reg_target_uses_own_kind_not_last_close,
             t_reg_target_clip_default_depends_on_mode,
             t_invert_reg_predictions_return_mode, t_invert_reg_predictions_window_scale_delegates,
             t_cs_normalize_matches_manual_zscore, t_cs_normalize_respects_min_assets,
             t_cs_normalize_does_not_mutate_input, t_cs_normalize_rejects_window_scale_mode,
             t_cs_normalize_rank_bounded, t_invert_cross_sectional_roundtrips_zscore,
             t_invert_cross_sectional_rejects_rank,
             t_compute_global_cutoff_matches_target_fraction, t_compute_global_cutoff_rejects_bad_pct,
             t_extract_test_dates_handles_both_shapes,
             t_raw_quantile_converges_to_kept_share_as_data_grows, t_raw_quantile_fails_loudly_on_crowded_data,
             t_build_leak_free_split_no_cross_asset_overlap,
             t_build_leak_free_split_matches_split_data_raw_quantile,
             t_compute_global_cutoff_matches_target_fraction, t_compute_global_cutoff_rejects_bad_pct,
             t_extract_test_dates_handles_both_shapes,
             t_raw_quantile_converges_to_kept_share_as_data_grows, t_raw_quantile_fails_loudly_on_crowded_data,
             t_build_leak_free_split_no_cross_asset_overlap,
             t_build_leak_free_split_matches_split_data_raw_quantile,
             t_fetch_data_uses_requested_interval_not_hardcoded,
             t_live_pad_length_scales_with_download_interval, t_auto_live_limit_scales_with_download_interval,
             t_build_dataset_live_decoupled_intervals_end_to_end,
             t_market_context_columns_names, t_add_market_context_aligns_and_zeroes_reference,
             t_market_context_reaches_dataset_via_build_dataset,
             t_rolling_schedule_expanding_basic, t_rolling_schedule_sliding_basic,
             t_rolling_schedule_requires_initial_train_span,
             t_rolling_splits_no_leak_between_train_and_test_per_window,
             t_rolling_splits_max_windows_respected,
             t_fetch_funding_rate_paginates_and_merges,
             t_fetch_open_interest_hist_capped_by_platform_lookback,
             t_save_and_load_funding_open_interest_accumulates,
             t_funding_oi_columns_names, t_add_funding_oi_features_aligns_and_flags_availability,
             t_add_funding_oi_features_neutral_when_no_archive, t_funding_oi_disabled_returns_dfs_unchanged,
             t_funding_oi_reaches_dataset_via_build_dataset,
             t_checkpoint_fingerprint_changes_with_funding_oi_config,
             t_align_single_tf_fast_path_matches_manual_windows,
             t_align_single_tf_fast_path_respects_stride,
             t_align_single_tf_fast_path_empty_when_too_short,
             t_align_single_tf_peak_memory_bounded,
             t_process_windows_price_level_matches_manual_formula,
             t_process_windows_degenerate_column_zeroed, t_process_windows_clip_applied,
             t_process_windows_peak_memory_bounded,
             t_default_workers_capped_by_ram, t_default_workers_unaffected_when_ram_unknown,
             t_checkpoint_fingerprint_changes_with_relevant_config,
             t_save_load_checkpoint_roundtrip, t_load_checkpoint_rejects_mismatched_fingerprint,
             t_load_checkpoint_missing_returns_none, t_clear_checkpoint_removes_files,
             t_build_dataset_from_loader_checkpoint_resume_end_to_end,
             t_build_dataset_checkpoint_invalidated_by_config_change]

    failed = []
    for t in tests:
        try:
            t()
            if verbose:
                print(f"  ✅ {t.__name__}")
        except Exception as exc:                            # noqa: BLE001
            failed.append((t.__name__, f"{type(exc).__name__}: {exc}"))
            if verbose:
                print(f"  ❌ {t.__name__}: {type(exc).__name__}: {str(exc)[:160]}")

    if failed:
        raise AssertionError(f"فشل {len(failed)} من {len(tests)} اختباراً: "
                             f"{[n for n, _ in failed]}")
    if verbose:
        print(f"✅ نجحت كل الاختبارات الذاتية ({len(tests)}).")
    return True


run_pipeline_selftests()


  ✅ t_exclude_matching
  ✅ t_exclude_edge_cases
  ✅ t_exclude_in_pipeline
  ✅ t_hour_features_daily
  ✅ t_hour_features_intraday
  ✅ t_split_crowded_timeline
  ✅ t_split_uniform_matches_requested
  ✅ t_split_raises_when_impossible
  ✅ t_split_explicit_dates_guarded
  ✅ t_fetch_single_page_unchanged
  ✅ t_fetch_paginates_and_merges
  ✅ t_fetch_exact_multiple_of_max
  ✅ t_fetch_history_shorter_than_limit
  ✅ t_fetch_retries_transient_page_error
  ✅ t_fetch_returns_none_on_persistent_page_failure
  ✅ t_fetch_rejects_nonpositive_limit
  ✅ t_exclude_features_names_and_patterns
  ✅ t_exclude_features_idempotent_and_guard
  ✅ t_exclude_features_reaches_dataset
  ✅ t_ratelimiter_sliding_window
  ✅ t_ratelimiter_threadsafe_never_exceeds
  ✅ t_default_request_limit_is_2000_per_minute
  ✅ t_fetch_acquires_per_request_including_retries
  ✅ t_fetch_is_actually_throttled
  ✅ t_reg_target_matches_direct_return
  ✅ t_reg_target_uses_own_kind_not_last_close
  ✅ t_reg_target_clip_default_depends_on_mode

True

## 20) مثال استخدام شامل (تاريخي)

التسلسل الكامل من سجل الأصول حتى `split_data`، عبر Google Drive المُركَّب
حصراً. **لن يعمل فعلياً** قبل أن تُعدّل خلية CONFIG بمسارات Drive الحقيقية
لديك (`drive_raw_dir`, `asset_registry_path`) وتتأكد أن ملفات CSV الخام
موجودة هناك بأعمدة `timestamp, open, high, low, close, volume`.

> ℹ️ `flatten_coins_by_category()` بلا وسائط تأخذ الآن `DEFAULT_ENABLED_CATEGORIES`
> الحقيقية (`["Large_Caps"]`) بدل "كل السجل" كما كانت في النسخة التخمينية
> السابقة من هذا الدفتر. مرّر `categories=ALL_GLOBAL` صراحةً لأخذ السجل كاملاً.


In [ ]:
# ⚠️ أُزيلت هنا: هذه الخلية كانت تُعرِّف load_asse() (تحميل سجل الأصول عبر رابط
# Drive عام، pd.read_csv('https://drive.google.com/uc?id=...')) ثم تستدعيها
# فوراً (registry = load_asse()) — تنفيذ فعلي غير مُعلَّق يناقض تحذير الخلية
# السابقة ("لن يعمل فعلياً قبل...") ويخالف سياسة هذا الدفتر المُعلَنة في الرأس
# ("تحميل البيانات التاريخية حصراً عبر Google Drive المُركَّب — لا رابط عام")،
# فيحاول اتصالاً شبكياً في كل مرّة يُشغَّل فيها هذا الدفتر (أو يُستورَد عبر
# %run من دفتر main). الدالة أيضاً مكرَّرة تماماً — load_asset_registry()
# (القسم ١١) تفعل نفس الشيء عبر Drive المُركَّب بلا رابط عام. استخدمها بدلاً:
#
#     registry = load_asset_registry()  # يقرأ CONFIG['asset_registry_path'] من MyDrive


In [55]:

def _system_ram_mb() -> Optional[float]:
    """رام النظام الكلي بالميجابايت من ``/proc/meminfo`` (لينكس — بيئة Colab
    القياسية). ``None`` إن تعذّرت القراءة (نظام غير لينكس مثلاً) — لا يُعطَّل
    شيء عندها، فقط يُفقَد هذا السقف الاحترازي الإضافي."""
    try:
        with open('/proc/meminfo') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    return int(line.split()[1]) / 1024.0
    except Exception:                                          # noqa: BLE001
        pass
    return None


def default_workers(n_items: int = 0) -> int:
    """عدد خيوط افتراضي معقول: مقيَّد بالمهام المتاحة، وعدد الأنوية، ورام
    النظام الكلي.

    ✅ سقف رام إضافي (جيجا واحد متاح لكل خيط تقريباً، تقدير محافظ): على آلة
    بأنوية كثيرة لكن رام محدودة (شائع في Colab المجانية، ~12 جيجا) كان عدد
    الخيوط يُشتقّ من الأنوية وحدها فيُفرط في التوازي، وكل خيط يحمل ذروته
    الخاصة من `align_multi_timeframes_time_based`/`process_windows` —
    تراكمها معاً هو ما سبّب استهلاك رام أكبر من حجم البيانات النهائي بأضعاف.
    هذا السقف احترازي إضافي فوق إصلاح الذروة نفسها في الدالتين أعلاه، لا بديل
    عنه (لا يعرف حجم كل أصل مسبقاً، فهو تقدير خشن لا مضبوط بدقة).
    """
    cpu = os.cpu_count() or 4
    workers = min(100, max(4, cpu * 2))
    ram_mb = _system_ram_mb()
    if ram_mb is not None:
        ram_cap = max(2, int(ram_mb // 1024))
        workers = min(workers, ram_cap)
    return max(1, min(workers, n_items)) if n_items else workers


In [56]:
# استبعاد عملات بعينها من كل ما يلي — تُتخطّى قبل محاولة تحميلها أصلاً.
# المطابقة على الاسم الكامل بلا حساسية لحالة الأحرف. أي اسم لا وجود له في
# السجل يُحذَّر منه (غالباً خطأ إملائي). ملخّص build_dataset يطبع في نهايته
# قائمة العملات التي تعذّر تحميلها جاهزة للصق هنا.
EXCLUDED_COINS = [
    # 'TSLAUSDT', 'XAUUSDT', 'USDCUSDT',
]
update_config(excluded_coins=EXCLUDED_COINS)
# لمرة واحدة بلا لمس CONFIG:  configs = exclude_coins(configs, ['TSLAUSDT'])


{'tf_order': ['1D'],
 'base_tf': '1D',
 'window_sizes': {'1D': 32},
 'stride': 1,
 'higher_tf_offset': 2,
 'targets': ['high', 'low', 'close'],
 'forecast_horizon': 1,
 'target_mode': 'direction',
 'enabled_heads': {'high_class': True,
  'high_reg': True,
  'low_class': True,
  'low_reg': True,
  'close_class': True,
  'close_reg': True},
 'target_loss_weights': {'high': 1.0, 'low': 1.0, 'close': 1.0},
 'abstention': {'enabled': True,
  'use_uncertainty_head': False,
  'head_hidden_dim': 128,
  'head_dropout': 0.15,
  'signals': {'margin': True, 'uncertainty': True, 'head_agreement': True},
  'min_margin': 0.2,
  'max_uncertainty_ratio': 1.0,
  'require_head_agreement': True,
  'min_coverage': 0.05,
  'dynamic_by_volatility': False,
  'volatility_sensitivity': 0.5,
  'cost_per_trade_pct': 0.06,
  'slippage_pct': 0.02,
  'opportunity_cost_pct': 0.0,
  'decision_head': 'close_class'},
 'scaler_type': 'robust',
 'scaling_window': 32,
 'eps': 1e-08,
 'split_mode': 'global_time',
 'split_da

In [57]:
# # 1) تركيب Drive (يحدث تلقائياً أيضاً داخل load_asset_registry/load_asset،
# #    لكن استدعاؤه هنا صراحةً يُظهر شاشة إذن Colab مبكراً)
# mount_drive()

# # 2) سجل الأصول الكامل من Drive (CONFIG['asset_registry_path'])
# # registry = load_asset_registry(CONFIG['asset_registry_file_id'])

# # 3) اختيار العملات المطلوبة (بلا وسائط = DEFAULT_ENABLED_CATEGORIES = ["Large_Caps"]؛
# DEFAULT_ENABLED_CATEGORIES=ALL_GLOBAL
# #    مرّر categories=ALL_GLOBAL لأخذ سجل الأصول كاملاً بلا تصفية فئات)
# desired = flatten_coins_by_category()
# configs = filter_desired_coins(registry, desired)

# # 4) دالة resample مربوطة بـ CONFIG الحالي (تستدعي add_features لكل فريم)
# resample_fn = make_resample_fn(CONFIG)

# # 5) بناء مجموعة البيانات: تحميل من Drive + مؤشرات + محاذاة + تطبيع،
# #    متوازياً بخيوط. load_asset هي دالة التحميل الوحيدة (Drive فقط).
# dataset = build_dataset(
#     configs,
#     load_asset_fn=load_asset,
#     resample_fn=resample_fn,
#     config=CONFIG,
# )

# # 6) فحوصات قبل التدريب
# summarize_dataset(dataset)
# # audit_normalization(dataset[f"X_{CONFIG['base_tf']}"], dataset['feature_order'])

# # 7) التقسيم الزمني train/val/test (بلا تسرّب بين العملات)
# # train, val, test = split_data(dataset, config=CONFIG)


In [ ]:
# save_data_to_drive(dataset)  # يحفظ نسخة مؤرّخة + preprocessing_output_latest.pkl.gz

# data = load_data_from_drive()  # يقرأ preprocessing_output_latest.pkl.gz من نفس مسار الحفظ


In [59]:
# train, val, test = split_data(dataset, config=CONFIG)

In [60]:
# import gdown
# import os,keras
# import pandas as pd

# def download_notebook_from_drive(
#     file_id: str,
#     notebook_name: str = 'dataprocess.ipynb',
#     download_dir: str = '.',
#     quiet: bool = False
# ) -> str:
#     """
#     تحميل notebook من Google Drive

#     Args:
#         file_id: معرّف الملف (يُستخرج من رابط المشاركة)
#         notebook_name: اسم الملف المحلي
#         download_dir: مجلد الحفظ
#         quiet: إخفاء تفاصيل التحميل

#     Returns:
#         المسار الكامل للملف المحمّل
#     """
#     # ✅ تنظيف file_id من أي معاملات إضافية
#     file_id = file_id.split('?')[0].strip()

#     output_path = os.path.join(download_dir, notebook_name)
#     url = f'https://drive.google.com/uc?id={file_id}'

#     print(f"📥 جاري تحميل {notebook_name}...")

#     try:
#         gdown.download(url, output_path, quiet=quiet, fuzzy=True)
#     except Exception as e:
#         print(f"❌ خطأ في التحميل: {e}")
#         print(f"💡 تأكد من:")
#         print(f"   1. الملف مشارك بإعداد 'Anyone with the link'")
#         print(f"   2. file_id صحيح: {file_id}")
#         print(f"   3. الرابط الكامل: {url}")
#         raise

#     if not os.path.exists(output_path):
#         raise FileNotFoundError(f"❌ فشل التحميل: {output_path}")

#     file_size = os.path.getsize(output_path) / 1024
#     print(f"✅ تم التحميل: {output_path} ({file_size:.1f} KB)")

#     return output_path


# file_id = '1BhNFaxq10SZRmdP3T4Gs3E8XxVulGYDy'  # بدون ?usp=sharing
# path = download_notebook_from_drive(file_id)
# # لتنفيذ كود من notebook آخر داخل notebook الحالي
# %run dataprocess.ipynb



## 21) مثال استخدام شامل (حيّ)

بنفس `CONFIG` المستخدَم للتاريخي — التجهيز مطابق تماماً، والاختلاف الوحيد
مصدر البيانات. تشغيل هذه الخلية يستهلك حصص Binance API الفعلية.


In [61]:
# audit_normalization(data[f"X_{CONFIG['base_tf']}"], data['feature_order'])

In [ ]:
# 1) رموز العقود الآجلة النشطة (أو اكتفِ بقائمة يدوية إن كنت تعرف ما تريد)
# symbols = get_normalized_futures_symbols()[:20]
# symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]

# 2) بناء مجموعة بيانات حيّة: نفس resample_fn/prepare_single_asset المستخدمة
#    تماماً في المسار التاريخي — فقط المصدر Binance بدل Drive.
# live_dataset = build_dataset_live(symbols, config=CONFIG)

# 3) استخلاص آخر عيّنة لكل عملة = نقطة live الحقيقية (ميزات حقيقية 100%،
#    هدفها صوري ويجب تجاهله كلياً):
# live_only = extract_last_batch(live_dataset, n=1)

# 4) البقية (كل شيء غير آخر عيّنة لكل عملة) أهدافها حقيقية 100% — اختبار
#    فوري لأداء نموذج مُدرَّب على أحدث بيانات لم يرها التدريب إطلاقاً:
# test_live = drop_tail_per_asset(live_dataset, n=1)

# 5) preds = model.predict({f"x_{tf}": test_live[f"X_{tf}"] for tf in CONFIG["tf_order"]})
#    قارن بـ test_live["y_<head>"] لتقييم الأداء على الأيام الأخيرة.
# 6) live_preds = model.predict({f"x_{tf}": live_only[f"X_{tf}"] for tf in CONFIG["tf_order"]})
#    هذا هو التنبؤ الفعلي الحالي — last_candles يحدّد زمن/سعر كل عيّنة فيه.
# print("live-only samples (1 per symbol):", len(live_only["base_params"]))
# print("test-live samples (real targets):", len(test_live["base_params"]))


📡 بناء بيانات حيّة لـ 3 رمز (limit=4512 شمعة حقيقية + 48 صورية، stride=1)...
🧬 عدد الميزات المُشتقة: 42
⚠️ [BTCUSDT/1h] خطأ غير متوقع (محاولة 1): 'NoneType' object has no attribute 'futures_klines'
⚠️ [BTCUSDT/1h] خطأ غير متوقع (محاولة 2): 'NoneType' object has no attribute 'futures_klines'
⚠️ [BTCUSDT/1h] خطأ غير متوقع (محاولة 3): 'NoneType' object has no attribute 'futures_klines'
❌ [BTCUSDT/1h] فشلت كل محاولات الجلب
   ⚠️ تعذّر بناء السياق السوقي من 'BTCUSDT': تعذّر جلب BTCUSDT من Binance — سيُترَك بلا سياق سوقي (أعمدة MKT_ = صفر للجميع).
⚙️ معالجة 3 عملة (بـ 3 خيطاً متوازياً)...
⚠️ [BTCUSDT/1h] خطأ غير متوقع (محاولة 1): 'NoneType' object has no attribute 'futures_klines'
⚠️ [ETHUSDT/1h] خطأ غير متوقع (محاولة 1): 'NoneType' object has no attribute 'futures_klines'
⚠️ [SOLUSDT/1h] خطأ غير متوقع (محاولة 1): 'NoneType' object has no attribute 'futures_klines'
⚠️ [BTCUSDT/1h] خطأ غير متوقع (محاولة 2): 'NoneType' object has no attribute 'futures_klines'
⚠️ [SOLUSDT/1h] خطأ غير متوقع (محا

ValueError: ❌ لم تُجمَّع أي عملة بنجاح — راجع 'أسباب التخطي' أعلاه.
   الاحتمال الأكثر شيوعاً: أسماء configs لا تطابق مفاتيح data (استخدم diagnose_data_vs_configs للمقارنة).

In [ ]:
# import os

# folder_path = f"/content/drive/MyDrive/{CONFIG['drive_raw_dir']}"
# os.chdir(folder_path)

# !ls